In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

print(os.path.exists(""))
print(os.path.exists(""))

In [ ]:
import os

BASE_PATH = "/content/drive/MyDrive/SANC/datasets/kvasir"

ZIP_PATH = os.path.join(
    BASE_PATH,
    "Kvasir_SEG_processed.zip"
)

CSV_PATH = os.path.join(
    BASE_PATH,
    "kvasir_metadata.csv"
)

print("ZIP exists:", os.path.exists(ZIP_PATH))
print("CSV exists:", os.path.exists(CSV_PATH))

In [ ]:
df["original"] = df["filename"].apply(
    lambda x: os.path.join(
        PROCESSED_PATH,
        "original",
        x
    )
)

df["low_light"] = df["filename"].apply(
    lambda x: os.path.join(
        PROCESSED_PATH,
        "low_light",
        x
    )
)

df["mask"] = df["filename"].apply(
    lambda x: os.path.join(
        PROCESSED_PATH,
        "masks",
        x
    )
)

df["dataset"] = "Kvasir-SEG"

In [ ]:
print(df.iloc[0])

In [ ]:
print(
    "Original:",
    df["original"].apply(os.path.isfile).sum(),
    "/",
    len(df)
)

print(
    "Low-light:",
    df["low_light"].apply(os.path.isfile).sum(),
    "/",
    len(df)
)

print(
    "Masks:",
    df["mask"].apply(os.path.isfile).sum(),
    "/",
    len(df)
)

In [ ]:
class KvasirDataset(Dataset):

    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):

        row = self.df.iloc[index]

        # -------------------------
        # Load original image
        # -------------------------
        original = Image.open(
            row["original"]
        ).convert("RGB")

        # -------------------------
        # Load low-light image
        # -------------------------
        low_light = Image.open(
            row["low_light"]
        ).convert("RGB")

        # -------------------------
        # Load segmentation mask
        # -------------------------
        mask = Image.open(
            row["mask"]
        ).convert("L")

        # -------------------------
        # Convert images to NumPy
        # -------------------------
        original = np.array(
            original
        ).astype(np.float32) / 255.0

        low_light = np.array(
            low_light
        ).astype(np.float32) / 255.0

        # -------------------------
        # Convert mask to binary
        # -------------------------
        mask = np.array(
            mask
        ).astype(np.float32)

        mask = (mask > 0).astype(np.float32)

        # -------------------------
        # H,W,C → C,H,W
        # -------------------------
        original = torch.from_numpy(
            original
        ).permute(2, 0, 1)

        low_light = torch.from_numpy(
            low_light
        ).permute(2, 0, 1)

        # -------------------------
        # H,W → 1,H,W
        # -------------------------
        mask = torch.from_numpy(
            mask
        ).unsqueeze(0)

        return {
            "original": original,
            "low_light": low_light,
            "mask": mask,
            "dataset": row["dataset"],
            "filename": row["filename"]
        }

In [ ]:
kvasir_dataset = KvasirDataset(df)

print("Dataset size:", len(kvasir_dataset))

In [ ]:
sample = kvasir_dataset[0]

print("Original:", sample["original"].shape)
print("Low-light:", sample["low_light"].shape)
print("Mask:", sample["mask"].shape)
print("Dataset:", sample["dataset"])
print("Filename:", sample["filename"])

In [ ]:
print(
    "Original range:",
    sample["original"].min().item(),
    "to",
    sample["original"].max().item()
)

print(
    "Low-light range:",
    sample["low_light"].min().item(),
    "to",
    sample["low_light"].max().item()
)

print(
    "Mask values:",
    torch.unique(sample["mask"])
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(
    sample["original"].permute(1, 2, 0)
)
axes[0].set_title("Original")
axes[0].axis("off")

axes[1].imshow(
    sample["low_light"].permute(1, 2, 0)
)
axes[1].set_title("Low-light")
axes[1].axis("off")

axes[2].imshow(
    sample["mask"].squeeze(0),
    cmap="gray"
)
axes[2].set_title("Mask")
axes[2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    shuffle=True
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

In [ ]:
train_dataset = KvasirDataset(train_df)
val_dataset = KvasirDataset(val_df)
test_dataset = KvasirDataset(test_df)

print(
    len(train_dataset),
    len(val_dataset),
    len(test_dataset)
)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0
)

In [ ]:
batch = next(iter(train_loader))

print("Original:", batch["original"].shape)
print("Low-light:", batch["low_light"].shape)
print("Mask:", batch["mask"].shape)
print("Dataset:", batch["dataset"])

In [ ]:
os.makedirs(
    "/content/drive/MyDrive/SANC/datasets/metadata",
    exist_ok=True
)

In [ ]:
train_df.to_csv(
    "/content/drive/MyDrive/SANC/datasets/metadata/kvasir_train.csv",
    index=False
)

val_df.to_csv(
    "/content/drive/MyDrive/SANC/datasets/metadata/kvasir_val.csv",
    index=False
)

test_df.to_csv(
    "/content/drive/MyDrive/SANC/datasets/metadata/kvasir_test.csv",
    index=False
)

print("Split metadata saved.")

In [ ]:
class KvasirDataset(Dataset):

    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):

        row = self.df.iloc[index]

        # Load images
        original = Image.open(row["original"]).convert("RGB")
        low_light = Image.open(row["low_light"]).convert("RGB")

        # Load mask
        mask = Image.open(row["mask"]).convert("L")

        # Convert images to NumPy and scale to [0, 1]
        original = np.array(original).astype(np.float32) / 255.0
        low_light = np.array(low_light).astype(np.float32) / 255.0

        # Convert mask to binary 0/1
        mask = np.array(mask).astype(np.float32)
        mask = (mask > 0).astype(np.float32)

        # H,W,C → C,H,W
        original = torch.from_numpy(original).permute(2, 0, 1)
        low_light = torch.from_numpy(low_light).permute(2, 0, 1)

        # H,W → 1,H,W
        mask = torch.from_numpy(mask).unsqueeze(0)

        return {
            "original": original,
            "low_light": low_light,
            "mask": mask,
            "dataset": row["dataset"],
            "filename": row["filename"]
        }

In [ ]:
kvasir_train = KvasirDataset(train_df)

print("Number of training samples:", len(kvasir_train))

sample = kvasir_train[0]

print("Original:", sample["original"].shape)
print("Low-light:", sample["low_light"].shape)
print("Mask:", sample["mask"].shape)
print("Dataset:", sample["dataset"])
print("Filename:", sample["filename"])

In [ ]:
print("Original range:",
      sample["original"].min().item(),
      sample["original"].max().item())

print("Low-light range:",
      sample["low_light"].min().item(),
      sample["low_light"].max().item())

print("Mask values:",
      torch.unique(sample["mask"]))

In [ ]:
import os
import pandas as pd

ORIGA_PATH = "/content/drive/MyDrive/SANC/datasets/ORIGA_processed"
ORIGA_INTEGRATION = os.path.join(ORIGA_PATH, "integration")

ORIGA_CSV = os.path.join(
    ORIGA_INTEGRATION,
    "ORIGA_metadata.csv"
)

origa_df = pd.read_csv(ORIGA_CSV)

print("Number of rows:", len(origa_df))
print("Columns:", origa_df.columns.tolist())

print("\nFirst 5 rows:")
display(origa_df.head())

In [ ]:
print("Images:",
      len(os.listdir(
          os.path.join(ORIGA_INTEGRATION, "images")
      )))

print("Masks:",
      len(os.listdir(
          os.path.join(ORIGA_INTEGRATION, "masks")
      )))

print("Low-light:",
      len(os.listdir(
          os.path.join(ORIGA_INTEGRATION, "low_light")
      )))

In [ ]:
from PIL import Image

image_folder = os.path.join(
    ORIGA_INTEGRATION,
    "images"
)

mask_folder = os.path.join(
    ORIGA_INTEGRATION,
    "masks"
)

image_files = os.listdir(image_folder)
mask_files = os.listdir(mask_folder)

print("First image:", image_files[0])
print("First mask:", mask_files[0])

img = Image.open(
    os.path.join(image_folder, image_files[0])
)

mask = Image.open(
    os.path.join(mask_folder, mask_files[0])
)

print("Image size:", img.size)
print("Image mode:", img.mode)

print("Mask size:", mask.size)
print("Mask mode:", mask.mode)

In [ ]:
print("First 10 images:")
print(sorted(image_files)[:10])

print("\nFirst 10 masks:")
print(sorted(mask_files)[:10])

In [ ]:
ORIGA_CSV = "/content/drive/MyDrive/SANC/datasets/ORIGA_processed/integration/ORIGA_metadata.csv"

origa_df = pd.read_csv(ORIGA_CSV)

print("Rows:", len(origa_df))
print("Columns:", origa_df.columns.tolist())

display(origa_df.head())

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image

ORIGA_INTEGRATION = "/content/drive/MyDrive/SANC/datasets/ORIGA_processed/integration"

ORIGA_IMAGES = os.path.join(
    ORIGA_INTEGRATION,
    "images"
)

ORIGA_MASKS = os.path.join(
    ORIGA_INTEGRATION,
    "masks"
)

ORIGA_LOW_LIGHT = os.path.join(
    ORIGA_INTEGRATION,
    "low_light"
)

print("Images:", ORIGA_IMAGES)
print("Masks:", ORIGA_MASKS)
print("Low-light:", ORIGA_LOW_LIGHT)

In [ ]:
os.makedirs(ORIGA_LOW_LIGHT, exist_ok=True)

image_files = sorted([
    f for f in os.listdir(ORIGA_IMAGES)
    if f.lower().endswith((".png", ".jpg", ".jpeg"))
])

print("Number of ORIGA images:", len(image_files))

In [ ]:
def simulate_low_light(
    image,
    brightness_factor=0.40,
    gamma=1.5,
    noise_std=0.01
):
    # Convert [0,255] → [0,1]
    image = image.astype(np.float32) / 255.0

    # Reduce illumination
    image = image * brightness_factor

    # Gamma transformation
    image = np.power(
        np.clip(image, 0.0, 1.0),
        gamma
    )

    # Add small noise
    noise = np.random.normal(
        0,
        noise_std,
        image.shape
    ).astype(np.float32)

    image = image + noise

    # Clip to valid range
    image = np.clip(
        image,
        0.0,
        1.0
    )

    # Convert back to uint8
    image = (
        image * 255
    ).astype(np.uint8)

    return image

In [ ]:
np.random.seed(42)

for filename in image_files:

    input_path = os.path.join(
        ORIGA_IMAGES,
        filename
    )

    output_path = os.path.join(
        ORIGA_LOW_LIGHT,
        filename
    )

    image = np.array(
        Image.open(input_path).convert("RGB")
    )

    low_light = simulate_low_light(image)

    Image.fromarray(low_light).save(
        output_path
    )

print("ORIGA low-light generation complete.")

In [ ]:
low_light_files = sorted([
    f for f in os.listdir(ORIGA_LOW_LIGHT)
    if f.lower().endswith((".png", ".jpg", ".jpeg"))
])

print("Original images:", len(image_files))
print("Low-light images:", len(low_light_files))

In [ ]:
print("Original files:", len(os.listdir(ORIGA_IMAGES)))
print("Low-light files:", len(os.listdir(ORIGA_LOW_LIGHT)))

print("\nOriginal first 20:")
print(sorted(os.listdir(ORIGA_IMAGES))[:20])

print("\nLow-light first 20:")
print(sorted(os.listdir(ORIGA_LOW_LIGHT))[:20])

In [ ]:
def get_image_files(folder):
    return sorted([
        f for f in os.listdir(folder)
        if f.lower().endswith((".png", ".jpg", ".jpeg"))
    ])

original_files = get_image_files(ORIGA_IMAGES)
low_light_files = get_image_files(ORIGA_LOW_LIGHT)

print("Original image files:", len(original_files))
print("Low-light image files:", len(low_light_files))

In [ ]:
missing = sorted(
    set(original_files) - set(low_light_files)
)

print("Missing low-light files:", len(missing))
print("First 30 missing:")
print(missing[:30])

In [ ]:
import os
import numpy as np
from PIL import Image

# Make sure the output folder exists
os.makedirs(ORIGA_LOW_LIGHT, exist_ok=True)

# Reproducibility
np.random.seed(42)

generated = 0

for filename in original_files:

    input_path = os.path.join(
        ORIGA_IMAGES,
        filename
    )

    output_path = os.path.join(
        ORIGA_LOW_LIGHT,
        filename
    )

    image = np.array(
        Image.open(input_path).convert("RGB")
    )

    low_light = simulate_low_light(
        image,
        brightness_factor=0.40,
        gamma=1.5,
        noise_std=0.01
    )

    Image.fromarray(low_light).save(
        output_path
    )

    generated += 1

print("Generated:", generated)

In [ ]:
low_light_files = get_image_files(ORIGA_LOW_LIGHT)

print("Original image files:", len(original_files))
print("Low-light image files:", len(low_light_files))

In [ ]:
print("Original list:", len(original_files))
print("Unique original names:", len(set(original_files)))

print("Low-light list:", len(low_light_files))
print("Unique low-light names:", len(set(low_light_files)))

print("\nDuplicate original names:",
      len(original_files) - len(set(original_files)))

print("Duplicate low-light names:",
      len(low_light_files) - len(set(low_light_files)))

In [ ]:
missing = sorted(
    set(original_files) - set(low_light_files)
)

print("Missing unique low-light files:", len(missing))
print("First 30 missing:")
print(missing[:30])

In [ ]:
print("Low-light folder path:")
print(ORIGA_LOW_LIGHT)

print("\nActual folder contents:")
print(len(os.listdir(ORIGA_LOW_LIGHT)))

print("\nFirst 20:")
print(os.listdir(ORIGA_LOW_LIGHT)[:20])

In [ ]:
test_filename = original_files[0]

test_input = os.path.join(
    ORIGA_IMAGES,
    test_filename
)

test_output = os.path.join(
    ORIGA_LOW_LIGHT,
    "TEST_WRITE.png"
)

image = np.array(
    Image.open(test_input).convert("RGB")
)

low_light = simulate_low_light(image)

Image.fromarray(low_light).save(test_output)

print("Saved:", test_output)
print("Exists immediately after saving:", os.path.exists(test_output))
print("Folder contents:", os.listdir(ORIGA_LOW_LIGHT)[:10])

In [ ]:
test_output = os.path.join(
    ORIGA_LOW_LIGHT,
    "TEST_WRITE.png"
)

if os.path.exists(test_output):
    os.remove(test_output)

print("Test file removed.")
print("Files currently:", len(os.listdir(ORIGA_LOW_LIGHT)))

In [ ]:
import time

batch_size = 50

for start in range(0, len(original_files), batch_size):

    batch = original_files[start:start + batch_size]

    for filename in batch:

        input_path = os.path.join(
            ORIGA_IMAGES,
            filename
        )

        output_path = os.path.join(
            ORIGA_LOW_LIGHT,
            filename
        )

        image = np.array(
            Image.open(input_path).convert("RGB")
        )

        low_light = simulate_low_light(image)

        Image.fromarray(low_light).save(output_path)

    # Verify after every batch
    current_files = get_image_files(ORIGA_LOW_LIGHT)

    print(
        f"Processed {min(start + batch_size, len(original_files))}"
        f"/{len(original_files)} | "
        f"Files in Drive: {len(current_files)}"
    )

    # Give Drive a moment to sync
    time.sleep(1)

In [ ]:
import os
import shutil

LOCAL_LOW_LIGHT = "/content/ORIGA_low_light"

if os.path.exists(LOCAL_LOW_LIGHT):
    shutil.rmtree(LOCAL_LOW_LIGHT)

os.makedirs(LOCAL_LOW_LIGHT)

print("Local folder ready:", LOCAL_LOW_LIGHT)

In [ ]:
np.random.seed(42)

for filename in original_files:

    input_path = os.path.join(
        ORIGA_IMAGES,
        filename
    )

    output_path = os.path.join(
        LOCAL_LOW_LIGHT,
        filename
    )

    image = np.array(
        Image.open(input_path).convert("RGB")
    )

    low_light = simulate_low_light(
        image,
        brightness_factor=0.40,
        gamma=1.5,
        noise_std=0.01
    )

    Image.fromarray(low_light).save(output_path)

print("Generation finished.")

In [ ]:
local_files = get_image_files(LOCAL_LOW_LIGHT)

print("Original:", len(original_files))
print("Generated locally:", len(local_files))

In [ ]:
missing_local = sorted(
    set(original_files) - set(local_files)
)

print("Missing locally:", len(missing_local))

In [ ]:
import shutil

LOCAL_ZIP = "/content/ORIGA_low_light"

zip_file = shutil.make_archive(
    LOCAL_ZIP,
    "zip",
    LOCAL_LOW_LIGHT
)

print("ZIP created:")
print(zip_file)

In [ ]:
DRIVE_ZIP = "/content/drive/MyDrive/SANC/datasets/ORIGA_processed/integration/ORIGA_low_light.zip"

shutil.copy2(
    zip_file,
    DRIVE_ZIP
)

print("Saved to Drive:")
print(DRIVE_ZIP)

In [ ]:
ORIGA_COMMON_CSV = os.path.join(
    ORIGA_INTEGRATION,
    "ORIGA_common_metadata.csv"
)

origa_common_df = pd.DataFrame({
    "filename": origa_df["image_path"].apply(
        os.path.basename
    ),

    "original": origa_df["image_path"].apply(
        lambda x: os.path.join(
            ORIGA_INTEGRATION,
            x
        )
    ),

    "low_light": origa_df["image_path"].apply(
        lambda x: os.path.join(
            ORIGA_LOW_LIGHT,
            os.path.basename(x)
        )
    ),

    "mask": origa_df["mask_path"].apply(
        lambda x: os.path.join(
            ORIGA_INTEGRATION,
            x
        )
    ),

    "dataset": origa_df["dataset_name"],

    "split": origa_df["split"]
})

display(origa_common_df.head())
print("Rows:", len(origa_common_df))

In [ ]:
import zipfile
import os

ORIGA_LOW_LIGHT_ZIP = "/content/drive/MyDrive/SANC/datasets/ORIGA_processed/integration/ORIGA_low_light.zip"

LOCAL_ORIGA_LOW_LIGHT = "/content/ORIGA_low_light"

os.makedirs(LOCAL_ORIGA_LOW_LIGHT, exist_ok=True)

with zipfile.ZipFile(ORIGA_LOW_LIGHT_ZIP, "r") as zip_ref:
    zip_ref.extractall(LOCAL_ORIGA_LOW_LIGHT)

print("Extraction complete.")
print("Files:", len(os.listdir(LOCAL_ORIGA_LOW_LIGHT)))

In [ ]:
test_path = os.path.join(
    LOCAL_ORIGA_LOW_LIGHT,
    "334.png"
)

print("334.png exists:", os.path.isfile(test_path))

In [ ]:
origa_common_df["low_light"] = origa_common_df["filename"].apply(
    lambda x: os.path.join(
        LOCAL_ORIGA_LOW_LIGHT,
        x
    )
)

In [ ]:
print(
    "Original:",
    origa_common_df["original"].apply(os.path.isfile).sum(),
    "/",
    len(origa_common_df)
)

print(
    "Low-light:",
    origa_common_df["low_light"].apply(os.path.isfile).sum(),
    "/",
    len(origa_common_df)
)

print(
    "Masks:",
    origa_common_df["mask"].apply(os.path.isfile).sum(),
    "/",
    len(origa_common_df)
)

In [ ]:
ORIGA_COMMON_CSV = "/content/drive/MyDrive/SANC/datasets/ORIGA_processed/integration/ORIGA_common_metadata.csv"

origa_common_df.to_csv(
    ORIGA_COMMON_CSV,
    index=False
)

print("Saved:", ORIGA_COMMON_CSV)

In [ ]:
import os

KVASIR_BASE = "/content/drive/MyDrive/SANC/datasets/kvasir"

print("Kvasir folder contents:")
print(os.listdir(KVASIR_BASE))

In [ ]:
METADATA_PATH = "/content/drive/MyDrive/SANC/datasets/metadata"

print("Metadata folder contents:")
print(os.listdir(METADATA_PATH))

In [ ]:
KVASIR_CSV = "/content/drive/MyDrive/SANC/datasets/kvasir/kvasir_metadata.csv"

kvasir_df = pd.read_csv(KVASIR_CSV)

print("Kvasir rows:", len(kvasir_df))
print("Kvasir columns:", kvasir_df.columns.tolist())

In [ ]:
KVASIR_PATH = "/content/drive/MyDrive/SANC/datasets/kvasir"

kvasir_df["original"] = kvasir_df["filename"].apply(
    lambda x: os.path.join(
        KVASIR_PATH,
        "original",
        x
    )
)

kvasir_df["low_light"] = kvasir_df["filename"].apply(
    lambda x: os.path.join(
        KVASIR_PATH,
        "low_light",
        x
    )
)

kvasir_df["mask"] = kvasir_df["filename"].apply(
    lambda x: os.path.join(
        KVASIR_PATH,
        "masks",
        x
    )
)

kvasir_df["dataset"] = "Kvasir-SEG"

In [ ]:
print(
    "Original:",
    kvasir_df["original"].apply(os.path.isfile).sum(),
    "/",
    len(kvasir_df)
)

print(
    "Low-light:",
    kvasir_df["low_light"].apply(os.path.isfile).sum(),
    "/",
    len(kvasir_df)
)

print(
    "Masks:",
    kvasir_df["mask"].apply(os.path.isfile).sum(),
    "/",
    len(kvasir_df)
)

In [ ]:
print("Kvasir columns:")
print(kvasir_df.columns.tolist())

print("\nORIGA columns:")
print(origa_df.columns.tolist())

In [ ]:
display(kvasir_df.head(2))
display(origa_df.head(2))

In [ ]:
METADATA_PATH = "/content/drive/MyDrive/SANC/datasets/metadata"

train_df = pd.read_csv(
    os.path.join(METADATA_PATH, "kvasir_train.csv")
)

val_df = pd.read_csv(
    os.path.join(METADATA_PATH, "kvasir_val.csv")
)

test_df = pd.read_csv(
    os.path.join(METADATA_PATH, "kvasir_test.csv")
)

print("Kvasir train:", len(train_df))
print("Kvasir validation:", len(val_df))
print("Kvasir test:", len(test_df))

In [ ]:
print("\nTrain columns:")
print(train_df.columns.tolist())

print("\nValidation columns:")
print(val_df.columns.tolist())

print("\nTest columns:")
print(test_df.columns.tolist())

In [ ]:
train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

print("Train:")
display(train_df.head(2))

print("Validation:")
display(val_df.head(2))

print("Test:")
display(test_df.head(2))

In [ ]:
kvasir_common_df = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)

print("Total Kvasir samples:", len(kvasir_common_df))
print("\nSplit distribution:")
print(kvasir_common_df["split"].value_counts())

In [ ]:
print(kvasir_common_df.columns.tolist())

In [ ]:
display(kvasir_common_df.head())

In [ ]:
# Make sure both datasets have exactly the same columns
print("Kvasir columns:")
print(kvasir_common_df.columns.tolist())

print("\nORIGA columns:")
print(origa_common_df.columns.tolist())

In [ ]:
combined_df = pd.concat(
    [kvasir_common_df, origa_common_df],
    ignore_index=True
)

print("Total samples:", len(combined_df))

print("\nDataset distribution:")
print(combined_df["dataset"].value_counts())

print("\nSplit distribution:")
print(combined_df["split"].value_counts())

In [ ]:
display(
    combined_df[
        ["filename", "dataset", "split"]
    ].head(10)
)

In [ ]:
print(
    combined_df.groupby(
        ["dataset", "split"]
    ).size()
)

In [ ]:
print("Checking combined dataset paths...\n")

print(
    "Original:",
    combined_df["original"].apply(os.path.isfile).sum(),
    "/",
    len(combined_df)
)

print(
    "Low-light:",
    combined_df["low_light"].apply(os.path.isfile).sum(),
    "/",
    len(combined_df)
)

print(
    "Masks:",
    combined_df["mask"].apply(os.path.isfile).sum(),
    "/",
    len(combined_df)
)

In [ ]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import numpy as np


class SANCMedicalDataset(Dataset):

    def __init__(self, dataframe):

        self.df = dataframe.reset_index(drop=True)

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        # Load original image
        original = Image.open(
            row["original"]
        ).convert("RGB")

        # Load low-light image
        low_light = Image.open(
            row["low_light"]
        ).convert("RGB")

        # Load binary mask
        mask = Image.open(
            row["mask"]
        ).convert("L")

        # Convert images to numpy arrays
        original = np.array(original)
        low_light = np.array(low_light)
        mask = np.array(mask)

        # Convert to PyTorch tensors
        original = torch.from_numpy(
            original
        ).permute(2, 0, 1).float() / 255.0

        low_light = torch.from_numpy(
            low_light
        ).permute(2, 0, 1).float() / 255.0

        mask = torch.from_numpy(
            mask
        ).float()

        # Convert mask to binary 0/1
        mask = (mask > 0).float()

        # Add channel dimension to mask
        mask = mask.unsqueeze(0)

        return {
            "original": original,
            "low_light": low_light,
            "mask": mask,
            "dataset": row["dataset"],
            "filename": row["filename"],
            "split": row["split"]
        }

In [ ]:
train_data = combined_df[
    combined_df["split"] == "train"
].copy()

val_data = combined_df[
    combined_df["split"] == "val"
].copy()

test_data = combined_df[
    combined_df["split"] == "test"
].copy()

print("Train:", len(train_data))
print("Validation:", len(val_data))
print("Test:", len(test_data))

In [ ]:
train_dataset = SANCMedicalDataset(
    train_data
)

val_dataset = SANCMedicalDataset(
    val_data
)

test_dataset = SANCMedicalDataset(
    test_data
)

print("Training dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

In [ ]:
sample = train_dataset[0]

print("Dataset:", sample["dataset"])
print("Filename:", sample["filename"])
print("Split:", sample["split"])

print("\nOriginal shape:", sample["original"].shape)
print("Low-light shape:", sample["low_light"].shape)
print("Mask shape:", sample["mask"].shape)

print("\nOriginal range:",
      sample["original"].min().item(),
      "to",
      sample["original"].max().item())

print("Low-light range:",
      sample["low_light"].min().item(),
      "to",
      sample["low_light"].max().item())

print("Mask values:",
      torch.unique(sample["mask"]))

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:
batch = next(iter(train_loader))

print("Original batch:", batch["original"].shape)
print("Low-light batch:", batch["low_light"].shape)
print("Mask batch:", batch["mask"].shape)

print("Datasets:", batch["dataset"][:5])
print("Filenames:", batch["filename"][:5])

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(
    3, 3,
    figsize=(10, 10)
)

for i in range(3):

    # Low-light image
    image = batch["low_light"][i]
    image = image.permute(1, 2, 0).numpy()

    # Mask
    mask = batch["mask"][i][0].numpy()

    axes[i, 0].imshow(image)
    axes[i, 0].set_title(
        f"Low-light\n{batch['dataset'][i]}"
    )
    axes[i, 0].axis("off")

    axes[i, 1].imshow(mask, cmap="gray")
    axes[i, 1].set_title("Ground-truth Mask")
    axes[i, 1].axis("off")

    # Original
    original = batch["original"][i]
    original = original.permute(1, 2, 0).numpy()

    axes[i, 2].imshow(original)
    axes[i, 2].set_title("Original")
    axes[i, 2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import os
import pandas as pd

HAM_BASE = "/content/drive/MyDrive/SANC/datasets/ham/HAM preprocessing"

HAM_CSV = os.path.join(
    HAM_BASE,
    "metadata.csv"
)

ham_df = pd.read_csv(HAM_CSV)

print("HAM10000 rows:", len(ham_df))
print("HAM10000 columns:")
print(ham_df.columns.tolist())

print("\nFirst 5 rows:")
display(ham_df.head())

In [ ]:
import os
import zipfile

HAM_BASE = "/content/drive/MyDrive/SANC/datasets/ham/HAM preprocessing"

HAM_EXTRACT = "/content/HAM10000_processed"

os.makedirs(HAM_EXTRACT, exist_ok=True)

zip_files = {
    "images": os.path.join(HAM_BASE, "images.zip"),
    "low_light": os.path.join(HAM_BASE, "low_light.zip"),
    "masks": os.path.join(HAM_BASE, "masks.zip")
}

for name, zip_path in zip_files.items():

    extract_path = os.path.join(
        HAM_EXTRACT,
        name
    )

    os.makedirs(extract_path, exist_ok=True)

    print(f"Extracting {name}...")

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_path)

    print("Done:", extract_path)

print("\nExtraction complete.")

In [ ]:
print("HAM extracted contents:")

for root, dirs, files in os.walk(HAM_EXTRACT):

    level = root.replace(HAM_EXTRACT, "").count(os.sep)

    if level <= 2:
        print(
            "  " * level + os.path.basename(root) + "/"
        )

        if level <= 1:
            for f in files[:5]:
                print(
                    "  " * (level + 1) + f
                )

In [ ]:
HAM_IMAGES = "/content/HAM10000_processed/images/images"
HAM_LOW_LIGHT = "/content/HAM10000_processed/low_light/low_light"
HAM_MASKS = "/content/HAM10000_processed/masks/masks"

print("Images folder exists:", os.path.exists(HAM_IMAGES))
print("Low-light folder exists:", os.path.exists(HAM_LOW_LIGHT))
print("Masks folder exists:", os.path.exists(HAM_MASKS))

In [ ]:
def get_image_files(folder):
    return sorted([
        f for f in os.listdir(folder)
        if f.lower().endswith((".png", ".jpg", ".jpeg"))
    ])


ham_image_files = get_image_files(HAM_IMAGES)
ham_low_light_files = get_image_files(HAM_LOW_LIGHT)
ham_mask_files = get_image_files(HAM_MASKS)

print("Original images:", len(ham_image_files))
print("Low-light images:", len(ham_low_light_files))
print("Masks:", len(ham_mask_files))

In [ ]:
print("Original unique:", len(set(ham_image_files)))
print("Low-light unique:", len(set(ham_low_light_files)))
print("Mask unique:", len(set(ham_mask_files)))

print("\nFirst 10 original:")
print(ham_image_files[:10])

print("\nFirst 10 low-light:")
print(ham_low_light_files[:10])

print("\nFirst 10 masks:")
print(ham_mask_files[:10])

In [ ]:
print(
    "Original vs Low-light:",
    set(ham_image_files) == set(ham_low_light_files)
)

print(
    "Original vs Masks:",
    set(ham_image_files) == set(ham_mask_files)
)

In [ ]:
from PIL import Image

# Pick one HAM sample
test_filename = ham_image_files[0]

img = Image.open(
    os.path.join(HAM_IMAGES, test_filename)
)

low = Image.open(
    os.path.join(HAM_LOW_LIGHT, test_filename)
)

mask = Image.open(
    os.path.join(HAM_MASKS, test_filename)
)

print("Filename:", test_filename)

print("\nOriginal:")
print("Size:", img.size)
print("Mode:", img.mode)

print("\nLow-light:")
print("Size:", low.size)
print("Mode:", low.mode)

print("\nMask:")
print("Size:", mask.size)
print("Mode:", mask.mode)

print("\nMask unique values:")
print(np.unique(np.array(mask)))

In [ ]:
HAM_COMMON = pd.DataFrame({
    "filename": ham_df["image_id"] + ".png",

    "original": ham_df["image_path"].apply(
        lambda x: os.path.join(
            HAM_IMAGES,
            os.path.basename(x)
        )
    ),

    "low_light": ham_df["low_light_path"].apply(
        lambda x: os.path.join(
            HAM_LOW_LIGHT,
            os.path.basename(x)
        )
    ),

    "mask": ham_df["mask_path"].apply(
        lambda x: os.path.join(
            HAM_MASKS,
            os.path.basename(x)
        )
    ),

    "dataset": ham_df["dataset"],

    "split": ham_df["split"]
})

display(HAM_COMMON.head())

print("HAM common metadata rows:", len(HAM_COMMON))
print("Columns:", HAM_COMMON.columns.tolist())

In [ ]:
print(
    "Original:",
    HAM_COMMON["original"].apply(os.path.isfile).sum(),
    "/",
    len(HAM_COMMON)
)

print(
    "Low-light:",
    HAM_COMMON["low_light"].apply(os.path.isfile).sum(),
    "/",
    len(HAM_COMMON)
)

print(
    "Masks:",
    HAM_COMMON["mask"].apply(os.path.isfile).sum(),
    "/",
    len(HAM_COMMON)
)

In [ ]:
print("\nHAM split distribution:")
print(HAM_COMMON["split"].value_counts())

print("\nDataset:")
print(HAM_COMMON["dataset"].value_counts())

In [ ]:
combined_all_df = pd.concat(
    [
        kvasir_common_df,
        origa_common_df,
        HAM_COMMON
    ],
    ignore_index=True
)

print("Total samples:", len(combined_all_df))

print("\nDataset distribution:")
print(
    combined_all_df["dataset"].value_counts()
)

print("\nSplit distribution:")
print(
    combined_all_df["split"].value_counts()
)

print("\nDataset + split distribution:")
print(
    combined_all_df.groupby(
        ["dataset", "split"]
    ).size()
)

In [ ]:
print("Checking all 2300 samples...\n")

print(
    "Original:",
    combined_all_df["original"].apply(os.path.isfile).sum(),
    "/",
    len(combined_all_df)
)

print(
    "Low-light:",
    combined_all_df["low_light"].apply(os.path.isfile).sum(),
    "/",
    len(combined_all_df)
)

print(
    "Masks:",
    combined_all_df["mask"].apply(os.path.isfile).sum(),
    "/",
    len(combined_all_df)
)

In [ ]:
train_data = combined_all_df[
    combined_all_df["split"] == "train"
].copy()

val_data = combined_all_df[
    combined_all_df["split"] == "val"
].copy()

test_data = combined_all_df[
    combined_all_df["split"] == "test"
].copy()

print("Train:", len(train_data))
print("Validation:", len(val_data))
print("Test:", len(test_data))

In [ ]:
train_dataset = SANCMedicalDataset(train_data)
val_dataset = SANCMedicalDataset(val_data)
test_dataset = SANCMedicalDataset(test_data)

print("Training dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:
batch = next(iter(train_loader))

print("Original batch:", batch["original"].shape)
print("Low-light batch:", batch["low_light"].shape)
print("Mask batch:", batch["mask"].shape)

print("\nDatasets in this batch:")
print(batch["dataset"])

print("\nUnique datasets:")
print(set(batch["dataset"]))

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

    gpu = torch.cuda.get_device_properties(0)

    print(
        "VRAM:",
        round(gpu.total_memory / 1024**3, 2),
        "GB"
    )

In [ ]:
print("Variables currently available:")

print("combined_all_df:", "combined_all_df" in globals())
print("train_dataset:", "train_dataset" in globals())
print("val_dataset:", "val_dataset" in globals())
print("test_dataset:", "test_dataset" in globals())

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

print("Pandas:", pd.__version__)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
print("kvasir_df:", "kvasir_df" in globals())
print("origa_df:", "origa_df" in globals())
print("ham_df:", "ham_df" in globals())

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from google.colab import drive

drive.mount("/content/drive")

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
BASE = "/content/drive/MyDrive/SANC/datasets"

# Kvasir
KVASIR_BASE = os.path.join(
    BASE,
    "kvasir"
)

KVASIR_PROCESSED = os.path.join(
    KVASIR_BASE,
    "Kvasir_SEG_processed"
)

KVASIR_METADATA = os.path.join(
    KVASIR_BASE,
    "kvasir_metadata.csv"
)

# ORIGA
ORIGA_BASE = os.path.join(
    BASE,
    "ORIGA_processed"
)

ORIGA_INTEGRATION = os.path.join(
    ORIGA_BASE,
    "integration"
)

ORIGA_IMAGES = os.path.join(
    ORIGA_INTEGRATION,
    "images"
)

ORIGA_MASKS = os.path.join(
    ORIGA_INTEGRATION,
    "masks"
)

ORIGA_LOW_LIGHT = os.path.join(
    ORIGA_INTEGRATION,
    "low_light"
)

ORIGA_CSV = os.path.join(
    ORIGA_INTEGRATION,
    "ORIGA_common_metadata.csv"
)

# HAM10000
HAM_BASE = os.path.join(
    BASE,
    "ham",
    "HAM preprocessing"
)

HAM_EXTRACT = os.path.join(
    HAM_BASE,
    "HAM10000_processed"
)

HAM_IMAGES = os.path.join(
    HAM_EXTRACT,
    "images",
    "images"
)

HAM_MASKS = os.path.join(
    HAM_EXTRACT,
    "masks",
    "masks"
)

HAM_LOW_LIGHT = os.path.join(
    HAM_EXTRACT,
    "low_light",
    "low_light"
)

HAM_CSV = os.path.join(
    HAM_BASE,
    "metadata.csv"
)

print("Paths defined.")

In [ ]:
paths_to_check = {
    "Kvasir metadata": KVASIR_METADATA,
    "Kvasir processed": KVASIR_PROCESSED,

    "ORIGA images": ORIGA_IMAGES,
    "ORIGA masks": ORIGA_MASKS,
    "ORIGA low-light": ORIGA_LOW_LIGHT,
    "ORIGA metadata": ORIGA_CSV,

    "HAM images": HAM_IMAGES,
    "HAM masks": HAM_MASKS,
    "HAM low-light": HAM_LOW_LIGHT,
    "HAM metadata": HAM_CSV
}

for name, path in paths_to_check.items():
    print(
        f"{name}:",
        "OK" if os.path.exists(path) else "MISSING",
        "\n   ",
        path
    )

In [ ]:
import os

DATASETS = "/content/drive/MyDrive/SANC/datasets"

print("=== DATASETS FOLDER ===")
print(os.listdir(DATASETS))

print("\n=== KVASIR ===")
KVASIR_DIR = os.path.join(DATASETS, "kvasir")
print(os.listdir(KVASIR_DIR))

print("\n=== HAM ===")
HAM_DIR = os.path.join(DATASETS, "ham")
print(os.listdir(HAM_DIR))

In [ ]:
print("combined_all_df:", "combined_all_df" in globals())
print("train_dataset:", "train_dataset" in globals())
print("val_dataset:", "val_dataset" in globals())
print("test_dataset:", "test_dataset" in globals())
print("train_loader:", "train_loader" in globals())

In [ ]:
import os
import pandas as pd

BASE = "/content/drive/MyDrive/SANC/datasets"

# Metadata paths
KVASIR_CSV = os.path.join(
    BASE, "kvasir", "kvasir_metadata.csv"
)

ORIGA_CSV = os.path.join(
    BASE, "ORIGA_processed",
    "integration",
    "ORIGA_common_metadata.csv"
)

HAM_CSV = os.path.join(
    BASE, "ham",
    "HAM preprocessing",
    "metadata.csv"
)

# Load metadata
kvasir_df = pd.read_csv(KVASIR_CSV)
origa_df = pd.read_csv(ORIGA_CSV)
ham_df = pd.read_csv(HAM_CSV)

print("Kvasir:", len(kvasir_df))
print("ORIGA:", len(origa_df))
print("HAM10000:", len(ham_df))

In [ ]:
print("\nKvasir columns:")
print(kvasir_df.columns.tolist())

print("\nORIGA columns:")
print(origa_df.columns.tolist())

print("\nHAM columns:")
print(ham_df.columns.tolist())

In [ ]:
# ============================================
# CREATE COMMON DATAFRAME FORMAT
# ============================================

# ---------- KVASIR ----------
kvasir_common_df = kvasir_df.copy()

# Kvasir already has the desired columns
kvasir_common_df = kvasir_common_df[
    ["filename", "original", "low_light", "mask", "dataset"]
].copy()

# Kvasir metadata doesn't currently have split,
# so we'll add it from the saved split CSVs later.
# For now, leave split empty.
kvasir_common_df["split"] = None


# ---------- ORIGA ----------
origa_common_df = origa_df.copy()

# ORIGA already has the desired common columns
origa_common_df = origa_common_df[
    ["filename", "original", "low_light", "mask", "dataset", "split"]
].copy()


# ---------- HAM10000 ----------
ham_common_df = pd.DataFrame({
    "filename": ham_df["image_id"].astype(str) + ".png",
    "original": ham_df["image_path"],
    "low_light": ham_df["low_light_path"],
    "mask": ham_df["mask_path"],
    "dataset": ham_df["dataset"],
    "split": ham_df["split"]
})


print("Kvasir:", len(kvasir_common_df))
print("ORIGA:", len(origa_common_df))
print("HAM10000:", len(ham_common_df))

print("\nColumns:")
print(kvasir_common_df.columns.tolist())
print(origa_common_df.columns.tolist())
print(ham_common_df.columns.tolist())

In [ ]:
# Find the saved Kvasir split metadata files

import os

METADATA_PATH = "/content/drive/MyDrive/SANC/datasets/metadata"

print("Metadata folder exists:", os.path.exists(METADATA_PATH))

if os.path.exists(METADATA_PATH):
    print("\nFiles:")
    for f in os.listdir(METADATA_PATH):
        print(f)

In [ ]:
# ============================================
# RESTORE KVASIR TRAIN / VAL / TEST SPLITS
# ============================================

train_split = pd.read_csv(
    os.path.join(METADATA_PATH, "kvasir_train.csv")
)

val_split = pd.read_csv(
    os.path.join(METADATA_PATH, "kvasir_val.csv")
)

test_split = pd.read_csv(
    os.path.join(METADATA_PATH, "kvasir_test.csv")
)

print("Kvasir train:", len(train_split))
print("Kvasir val:", len(val_split))
print("Kvasir test:", len(test_split))

print("\nTrain columns:")
print(train_split.columns.tolist())

In [ ]:
# ============================================
# ADD SPLIT INFORMATION TO KVASIR
# ============================================

kvasir_common_df["split"] = "train"

kvasir_common_df.loc[
    kvasir_common_df["filename"].isin(val_split["filename"]),
    "split"
] = "val"

kvasir_common_df.loc[
    kvasir_common_df["filename"].isin(test_split["filename"]),
    "split"
] = "test"


print("Kvasir split distribution:")
print(kvasir_common_df["split"].value_counts())

print("\nMissing split values:")
print(kvasir_common_df["split"].isna().sum())

In [ ]:
print("Kvasir:", len(kvasir_common_df))
print("ORIGA:", len(origa_common_df))
print("HAM10000:", len(ham_common_df))

print("\nKvasir:")
print(kvasir_common_df["split"].value_counts())

print("\nORIGA:")
print(origa_common_df["split"].value_counts())

print("\nHAM10000:")
print(ham_common_df["split"].value_counts())

In [ ]:
# ============================================
# COMBINE ALL 3 DATASETS
# ============================================

combined_all_df = pd.concat(
    [
        kvasir_common_df,
        origa_common_df,
        ham_common_df
    ],
    ignore_index=True
)

print("Total samples:", len(combined_all_df))

print("\nDataset distribution:")
print(combined_all_df["dataset"].value_counts())

print("\nSplit distribution:")
print(combined_all_df["split"].value_counts())

print("\nDataset + split:")
print(
    combined_all_df
    .groupby(["dataset", "split"])
    .size()
)

In [ ]:
# ============================================
# SANC MEDICAL DATASET
# ============================================

class SANCMedicalDataset(Dataset):

    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        # -------------------------
        # Original image
        # -------------------------
        original = Image.open(
            row["original"]
        ).convert("RGB")

        # -------------------------
        # Low-light image
        # -------------------------
        low_light = Image.open(
            row["low_light"]
        ).convert("RGB")

        # -------------------------
        # Binary mask
        # -------------------------
        mask = Image.open(
            row["mask"]
        ).convert("L")

        # -------------------------
        # Convert to NumPy
        # -------------------------
        original = np.array(original)
        low_light = np.array(low_light)
        mask = np.array(mask)

        # -------------------------
        # Convert to tensors
        # -------------------------
        original = torch.from_numpy(
            original
        ).permute(2, 0, 1).float() / 255.0

        low_light = torch.from_numpy(
            low_light
        ).permute(2, 0, 1).float() / 255.0

        # Binary mask: 0 or 1
        mask = torch.from_numpy(
            (mask > 0).astype(np.float32)
        ).unsqueeze(0)

        return {
            "original": original,
            "low_light": low_light,
            "mask": mask,
            "filename": row["filename"],
            "dataset": row["dataset"],
            "split": row["split"]
        }


print("SANCMedicalDataset defined.")

In [ ]:
# ============================================
# TRAIN / VALIDATION / TEST DATASETS
# ============================================

train_data = combined_all_df[
    combined_all_df["split"] == "train"
].reset_index(drop=True)

val_data = combined_all_df[
    combined_all_df["split"] == "val"
].reset_index(drop=True)

test_data = combined_all_df[
    combined_all_df["split"] == "test"
].reset_index(drop=True)


train_dataset = SANCMedicalDataset(train_data)
val_dataset = SANCMedicalDataset(val_data)
test_dataset = SANCMedicalDataset(test_data)


print("Training dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

In [ ]:
print("Original folder exists:",
      os.path.exists(KVASIR_ORIGINAL))

print("Low-light folder exists:",
      os.path.exists(KVASIR_LOW_LIGHT))

print("Masks folder exists:",
      os.path.exists(KVASIR_MASKS))

print("\nOriginal files:",
      len(os.listdir(KVASIR_ORIGINAL)))

print("Low-light files:",
      len(os.listdir(KVASIR_LOW_LIGHT)))

print("Mask files:",
      len(os.listdir(KVASIR_MASKS)))

In [ ]:
print("\nFirst Kvasir sample:")
print("Filename:", kvasir_common_df.iloc[0]["filename"])
print("Original:", kvasir_common_df.iloc[0]["original"])
print("Low-light:", kvasir_common_df.iloc[0]["low_light"])
print("Mask:", kvasir_common_df.iloc[0]["mask"])

print("\nPath checks:")

print(
    "Original:",
    os.path.isfile(kvasir_common_df.iloc[0]["original"])
)

print(
    "Low-light:",
    os.path.isfile(kvasir_common_df.iloc[0]["low_light"])
)

print(
    "Mask:",
    os.path.isfile(kvasir_common_df.iloc[0]["mask"])
)

In [ ]:
combined_all_df = pd.concat(
    [
        kvasir_common_df,
        origa_common_df,
        ham_common_df
    ],
    ignore_index=True
)

print("Total:", len(combined_all_df))
print(combined_all_df["dataset"].value_counts())
print(combined_all_df["split"].value_counts())

In [ ]:
# ============================================
# VERIFY ALL 2300 IMAGE/MASK PATHS
# ============================================

for dataset_name, df in [
    ("Kvasir-SEG", kvasir_common_df),
    ("ORIGA", origa_common_df),
    ("HAM10000", ham_common_df)
]:

    print(f"\n===== {dataset_name} =====")

    print(
        "Original:",
        df["original"].apply(os.path.isfile).sum(),
        "/",
        len(df)
    )

    print(
        "Low-light:",
        df["low_light"].apply(os.path.isfile).sum(),
        "/",
        len(df)
    )

    print(
        "Masks:",
        df["mask"].apply(os.path.isfile).sum(),
        "/",
        len(df)
    )

In [ ]:
print("ORIGA low-light folder:")
print(ORIGA_LOW_LIGHT)

print("\nExists:", os.path.exists(ORIGA_LOW_LIGHT))

if os.path.exists(ORIGA_LOW_LIGHT):
    files = os.listdir(ORIGA_LOW_LIGHT)
    print("Number of files:", len(files))
    print("First 20 files:")
    print(files[:20])

In [ ]:
HAM_PREPROCESSING = (
    "/content/drive/MyDrive/SANC/datasets/ham/"
    "HAM preprocessing"
)

print("HAM preprocessing:")
print(HAM_PREPROCESSING)

print("\nContents:")
print(os.listdir(HAM_PREPROCESSING))

In [ ]:
# ============================================
# FINAL PATH CHECK
# ============================================

for dataset_name, df in [
    ("Kvasir-SEG", kvasir_common_df),
    ("ORIGA", origa_common_df),
    ("HAM10000", ham_common_df)
]:

    print(f"\n===== {dataset_name} =====")

    print(
        "Original:",
        df["original"].apply(os.path.isfile).sum(),
        "/",
        len(df)
    )

    print(
        "Low-light:",
        df["low_light"].apply(os.path.isfile).sum(),
        "/",
        len(df)
    )

    print(
        "Masks:",
        df["mask"].apply(os.path.isfile).sum(),
        "/",
        len(df)
    )

In [ ]:
# ============================================
# REBUILD FINAL COMBINED DATAFRAME
# ============================================

combined_all_df = pd.concat(
    [
        kvasir_common_df,
        origa_common_df,
        ham_common_df
    ],
    ignore_index=True
)

print("Total samples:", len(combined_all_df))

print("\nDataset distribution:")
print(combined_all_df["dataset"].value_counts())

print("\nSplit distribution:")
print(combined_all_df["split"].value_counts())

In [ ]:
# ============================================
# RECREATE PYTORCH DATASETS
# ============================================

train_data = combined_all_df[
    combined_all_df["split"] == "train"
].reset_index(drop=True)

val_data = combined_all_df[
    combined_all_df["split"] == "val"
].reset_index(drop=True)

test_data = combined_all_df[
    combined_all_df["split"] == "test"
].reset_index(drop=True)

train_dataset = SANCMedicalDataset(train_data)
val_dataset = SANCMedicalDataset(val_data)
test_dataset = SANCMedicalDataset(test_data)

print("Training dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

In [ ]:
# ============================================
# U-NET FOR SALIENCY ESTIMATION
# ============================================

import torch
import torch.nn as nn


class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):

    def __init__(self, in_channels=3, out_channels=1):
        super().__init__()

        # -------------------------
        # Encoder
        # -------------------------
        self.enc1 = DoubleConv(3, 64)
        self.enc2 = DoubleConv(64, 128)
        self.enc3 = DoubleConv(128, 256)
        self.enc4 = DoubleConv(256, 512)

        self.pool = nn.MaxPool2d(
            kernel_size=2,
            stride=2
        )

        # -------------------------
        # Bottleneck
        # -------------------------
        self.bottleneck = DoubleConv(
            512,
            1024
        )

        # -------------------------
        # Decoder
        # -------------------------
        self.up4 = nn.ConvTranspose2d(
            1024,
            512,
            kernel_size=2,
            stride=2
        )

        self.dec4 = DoubleConv(
            1024,
            512
        )

        self.up3 = nn.ConvTranspose2d(
            512,
            256,
            kernel_size=2,
            stride=2
        )

        self.dec3 = DoubleConv(
            512,
            256
        )

        self.up2 = nn.ConvTranspose2d(
            256,
            128,
            kernel_size=2,
            stride=2
        )

        self.dec2 = DoubleConv(
            256,
            128
        )

        self.up1 = nn.ConvTranspose2d(
            128,
            64,
            kernel_size=2,
            stride=2
        )

        self.dec1 = DoubleConv(
            128,
            64
        )

        # -------------------------
        # Output
        # -------------------------
        self.output = nn.Conv2d(
            64,
            out_channels,
            kernel_size=1
        )

    def forward(self, x):

        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        # Bottleneck
        b = self.bottleneck(
            self.pool(e4)
        )

        # Decoder
        d4 = self.up4(b)
        d4 = torch.cat([d4, e4], dim=1)
        d4 = self.dec4(d4)

        d3 = self.up3(d4)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.output(d1)


print("U-Net defined successfully.")

In [ ]:
# ============================================
# CREATE U-NET ON GPU
# ============================================

model = UNet(
    in_channels=3,
    out_channels=1
).to(device)

print("Model device:", next(model.parameters()).device)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)

In [ ]:
# ============================================
# TEST U-NET FORWARD PASS
# ============================================

batch = next(iter(train_loader))

low_light = batch["low_light"].to(device)
mask = batch["mask"].to(device)

with torch.no_grad():
    prediction = model(low_light)

print("Input:", low_light.shape)
print("Prediction:", prediction.shape)
print("Ground truth:", mask.shape)

In [ ]:
# ============================================
# BCE + DICE LOSS
# ============================================

class DiceLoss(nn.Module):

    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):

        probs = torch.sigmoid(logits)

        probs = probs.view(-1)
        targets = targets.view(-1)

        intersection = (probs * targets).sum()

        dice = (
            2.0 * intersection + self.smooth
        ) / (
            probs.sum() + targets.sum() + self.smooth
        )

        return 1.0 - dice


class BCEDiceLoss(nn.Module):

    def __init__(self):
        super().__init__()

        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()

    def forward(self, logits, targets):

        bce_loss = self.bce(logits, targets)
        dice_loss = self.dice(logits, targets)

        return bce_loss + dice_loss


criterion = BCEDiceLoss()

print("Loss function created.")

In [ ]:
# ============================================
# TEST LOSS
# ============================================

loss = criterion(prediction, mask)

print("Prediction shape:", prediction.shape)
print("Mask shape:", mask.shape)
print("Initial loss:", loss.item())

In [ ]:
# ============================================
# OPTIMIZER
# ============================================

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

print("Optimizer:", optimizer)
print("Learning rate:", 1e-4)

In [ ]:
# ============================================
# METRICS
# ============================================

def dice_score(logits, targets, threshold=0.5, smooth=1e-6):

    probs = torch.sigmoid(logits)
    preds = (probs >= threshold).float()

    preds = preds.view(preds.size(0), -1)
    targets = targets.view(targets.size(0), -1)

    intersection = (preds * targets).sum(dim=1)

    dice = (
        (2 * intersection + smooth) /
        (preds.sum(dim=1) + targets.sum(dim=1) + smooth)
    )

    return dice.mean().item()


def iou_score(logits, targets, threshold=0.5, smooth=1e-6):

    probs = torch.sigmoid(logits)
    preds = (probs >= threshold).float()

    preds = preds.view(preds.size(0), -1)
    targets = targets.view(targets.size(0), -1)

    intersection = (preds * targets).sum(dim=1)

    union = (
        preds.sum(dim=1)
        + targets.sum(dim=1)
        - intersection
    )

    iou = (intersection + smooth) / (union + smooth)

    return iou.mean().item()


print("Metrics defined.")

In [ ]:
# ============================================
# U-NET TRAINING
# ============================================

NUM_EPOCHS = 20

best_val_dice = 0.0

best_model_path = "/content/drive/MyDrive/SANC/models/best_unet_saliency.pth"

# Create model directory
os.makedirs(
    os.path.dirname(best_model_path),
    exist_ok=True
)

history = {
    "train_loss": [],
    "val_loss": [],
    "val_dice": [],
    "val_iou": []
}


for epoch in range(NUM_EPOCHS):

    # ========================================
    # TRAINING
    # ========================================

    model.train()

    running_train_loss = 0.0

    for batch in train_loader:

        low_light = batch["low_light"].to(
            device,
            non_blocking=True
        )

        mask = batch["mask"].to(
            device,
            non_blocking=True
        )

        # Clear gradients
        optimizer.zero_grad(set_to_none=True)

        # Forward pass
        prediction = model(low_light)

        # Loss
        loss = criterion(
            prediction,
            mask
        )

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        running_train_loss += loss.item()

    train_loss = (
        running_train_loss /
        len(train_loader)
    )


    # ========================================
    # VALIDATION
    # ========================================

    model.eval()

    running_val_loss = 0.0
    running_val_dice = 0.0
    running_val_iou = 0.0

    with torch.no_grad():

        for batch in val_loader:

            low_light = batch["low_light"].to(
                device,
                non_blocking=True
            )

            mask = batch["mask"].to(
                device,
                non_blocking=True
            )

            prediction = model(
                low_light
            )

            loss = criterion(
                prediction,
                mask
            )

            running_val_loss += loss.item()

            running_val_dice += dice_score(
                prediction,
                mask
            )

            running_val_iou += iou_score(
                prediction,
                mask
            )


    val_loss = (
        running_val_loss /
        len(val_loader)
    )

    val_dice = (
        running_val_dice /
        len(val_loader)
    )

    val_iou = (
        running_val_iou /
        len(val_loader)
    )


    # ========================================
    # SAVE HISTORY
    # ========================================

    history["train_loss"].append(
        train_loss
    )

    history["val_loss"].append(
        val_loss
    )

    history["val_dice"].append(
        val_dice
    )

    history["val_iou"].append(
        val_iou
    )


    # ========================================
    # SAVE BEST MODEL
    # ========================================

    if val_dice > best_val_dice:

        best_val_dice = val_dice

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_dice": val_dice,
                "val_iou": val_iou,
                "val_loss": val_loss
            },
            best_model_path
        )

        best_marker = " ← BEST"

    else:

        best_marker = ""


    # ========================================
    # PRINT PROGRESS
    # ========================================

    print(
        f"Epoch [{epoch + 1:02d}/{NUM_EPOCHS}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Dice: {val_dice:.4f} | "
        f"Val IoU: {val_iou:.4f}"
        f"{best_marker}"
    )


print("\nTraining complete.")
print("Best validation Dice:", best_val_dice)
print("Best model saved to:")
print(best_model_path)

In [ ]:
# ============================================================
# U-NET — FINAL TEST EVALUATION
# ============================================================

model.eval()

test_loss = 0.0
test_dice = 0.0
test_iou = 0.0

with torch.no_grad():

    for batch in test_eval_loader:

        low_light = batch["low_light"].to(
            device,
            non_blocking=True
        )

        mask = batch["mask"].to(
            device,
            non_blocking=True
        )

        prediction = model(low_light)

        loss = criterion(
            prediction,
            mask
        )

        test_loss += loss.item()

        test_dice += dice_score(
            prediction,
            mask
        )

        test_iou += iou_score(
            prediction,
            mask
        )


test_loss /= len(test_eval_loader)
test_dice /= len(test_eval_loader)
test_iou /= len(test_eval_loader)


print("========================================")
print("FINAL TEST RESULTS")
print("========================================")

print("Official test samples:", len(test_data))
print("Valid evaluated samples:", len(test_eval_dataset))
print("Excluded corrupted samples:",
      len(test_data) - len(test_eval_dataset))

print()
print("Test Loss:", round(test_loss, 4))
print("Test Dice:", round(test_dice, 4))
print("Test IoU:", round(test_iou, 4))

In [ ]:
# ============================================================
# VISUALIZE U-NET PREDICTIONS
# ============================================================

import matplotlib.pyplot as plt

model.eval()

# Number of examples to display
NUM_EXAMPLES = 6

shown = 0

plt.figure(figsize=(16, 12))

with torch.no_grad():

    for batch in test_eval_loader:

        low_light = batch["low_light"].to(device)

        prediction = model(low_light)

        prediction_prob = torch.sigmoid(
            prediction
        )

        prediction_mask = (
            prediction_prob >= 0.5
        ).float()

        for i in range(
            low_light.size(0)
        ):

            if shown >= NUM_EXAMPLES:
                break

            # Convert tensors to numpy
            original_img = (
                batch["original"][i]
                .permute(1, 2, 0)
                .cpu()
                .numpy()
            )

            low_img = (
                batch["low_light"][i]
                .permute(1, 2, 0)
                .cpu()
                .numpy()
            )

            gt_mask = (
                batch["mask"][i]
                .squeeze()
                .cpu()
                .numpy()
            )

            pred_prob = (
                prediction_prob[i]
                .squeeze()
                .cpu()
                .numpy()
            )

            pred_mask = (
                prediction_mask[i]
                .squeeze()
                .cpu()
                .numpy()
            )

            dataset_name = batch["dataset"][i]
            filename = batch["filename"][i]

            row = shown

            # Original
            plt.subplot(
                NUM_EXAMPLES,
                4,
                row * 4 + 1
            )

            plt.imshow(original_img)
            plt.axis("off")
            plt.title(
                f"{dataset_name}\nOriginal"
            )

            # Low-light
            plt.subplot(
                NUM_EXAMPLES,
                4,
                row * 4 + 2
            )

            plt.imshow(low_img)
            plt.axis("off")
            plt.title("Low-light")

            # Ground truth
            plt.subplot(
                NUM_EXAMPLES,
                4,
                row * 4 + 3
            )

            plt.imshow(
                gt_mask,
                cmap="gray"
            )

            plt.axis("off")
            plt.title("Ground truth")

            # Prediction
            plt.subplot(
                NUM_EXAMPLES,
                4,
                row * 4 + 4
            )

            plt.imshow(
                pred_prob,
                cmap="hot"
            )

            plt.axis("off")
            plt.title(
                "U-Net saliency"
            )

            print(
                f"{shown + 1}. "
                f"{dataset_name} — "
                f"{filename}"
            )

            shown += 1

        if shown >= NUM_EXAMPLES:
            break


plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# DATASET-WISE TEST PERFORMANCE
# ============================================================

model.eval()

results = {}

for dataset_name in [
    "Kvasir-SEG",
    "ORIGA",
    "HAM10000"
]:

    df = test_eval_data[
        test_eval_data["dataset"] == dataset_name
    ].reset_index(drop=True)

    dataset = SANCMedicalDataset(df)

    loader = DataLoader(
        dataset,
        batch_size=8,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    total_loss = 0.0
    total_dice = 0.0
    total_iou = 0.0

    with torch.no_grad():

        for batch in loader:

            low_light = batch["low_light"].to(
                device,
                non_blocking=True
            )

            mask = batch["mask"].to(
                device,
                non_blocking=True
            )

            prediction = model(low_light)

            loss = criterion(
                prediction,
                mask
            )

            total_loss += loss.item()

            total_dice += dice_score(
                prediction,
                mask
            )

            total_iou += iou_score(
                prediction,
                mask
            )

    total_loss /= len(loader)
    total_dice /= len(loader)
    total_iou /= len(loader)

    results[dataset_name] = {
        "samples": len(dataset),
        "loss": total_loss,
        "dice": total_dice,
        "iou": total_iou
    }


print("========================================")
print("DATASET-WISE TEST RESULTS")
print("========================================")

for name, r in results.items():

    print(f"\n{name}")
    print("Samples:", r["samples"])
    print("Loss:", round(r["loss"], 4))
    print("Dice:", round(r["dice"], 4))
    print("IoU:", round(r["iou"], 4))

In [ ]:
# ============================================================
# CROSS-DATASET INPUT / MASK / PREDICTION DIAGNOSTIC
# ============================================================

model.eval()

for dataset_name in [
    "Kvasir-SEG",
    "ORIGA",
    "HAM10000"
]:

    df = test_eval_data[
        test_eval_data["dataset"] == dataset_name
    ].reset_index(drop=True)

    dataset = SANCMedicalDataset(df)

    loader = DataLoader(
        dataset,
        batch_size=8,
        shuffle=False,
        num_workers=0
    )

    print("\n========================================")
    print(dataset_name)
    print("========================================")

    with torch.no_grad():

        batch = next(iter(loader))

        low_light = batch["low_light"].to(device)
        mask = batch["mask"].to(device)

        prediction = model(low_light)
        probability = torch.sigmoid(prediction)

        print("Low-light shape:", low_light.shape)

        print(
            "Low-light range:",
            low_light.min().item(),
            "to",
            low_light.max().item()
        )

        print(
            "Low-light mean:",
            low_light.mean().item()
        )

        print(
            "Mask unique values:",
            torch.unique(mask)
        )

        print(
            "Mask foreground ratio:",
            mask.mean().item()
        )

        print(
            "Prediction logits:",
            prediction.min().item(),
            "to",
            prediction.max().item()
        )

        print(
            "Prediction probability:",
            probability.min().item(),
            "to",
            probability.max().item()
        )

        print(
            "Predicted foreground ratio:",
            (probability >= 0.5).float().mean().item()
        )

In [ ]:
# ============================================================
# CORRECTED SANC MEDICAL DATASET
# ============================================================

import torch
from torch.utils.data import Dataset
from PIL import Image
import numpy as np


class SANCMedicalDataset(Dataset):

    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        # ----------------------------------------------------
        # Load original image
        # ----------------------------------------------------

        original = Image.open(
            row["original"]
        ).convert("RGB")

        # ----------------------------------------------------
        # Load low-light image
        # ----------------------------------------------------

        low_light = Image.open(
            row["low_light"]
        ).convert("RGB")

        # ----------------------------------------------------
        # Load mask
        # ----------------------------------------------------

        mask = Image.open(
            row["mask"]
        ).convert("L")

        # ----------------------------------------------------
        # Convert to numpy
        # ----------------------------------------------------

        original = np.array(
            original,
            dtype=np.float32
        )

        low_light = np.array(
            low_light,
            dtype=np.float32
        )

        mask = np.array(
            mask,
            dtype=np.float32
        )

        # ----------------------------------------------------
        # Normalize images
        # ----------------------------------------------------

        original = original / 255.0
        low_light = low_light / 255.0

        # ----------------------------------------------------
        # ROBUST MASK NORMALIZATION
        #
        # Kvasir:
        #   mask values approximately 0-255
        #
        # ORIGA/HAM:
        #   mask values 0-1
        # ----------------------------------------------------

        if mask.max() > 1.0:

            # Kvasir-style mask
            mask = mask / 255.0

        # Binary mask
        mask = (mask > 0.5).astype(
            np.float32
        )

        # ----------------------------------------------------
        # Convert images to tensors
        # ----------------------------------------------------

        original = torch.from_numpy(
            original
        ).permute(2, 0, 1).float()

        low_light = torch.from_numpy(
            low_light
        ).permute(2, 0, 1).float()

        # ----------------------------------------------------
        # Mask tensor
        # ----------------------------------------------------

        mask = torch.from_numpy(
            mask
        ).float().unsqueeze(0)

        return {
            "original": original,
            "low_light": low_light,
            "mask": mask,
            "dataset": row["dataset"],
            "filename": row["filename"],
            "split": row["split"]
        }


print("Corrected SANCMedicalDataset defined.")

In [ ]:
# ============================================================
# REBUILD DATASET OBJECTS
# ============================================================

train_dataset = SANCMedicalDataset(
    train_data
)

val_dataset = SANCMedicalDataset(
    val_data
)

test_eval_dataset = SANCMedicalDataset(
    test_eval_data
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test evaluation:", len(test_eval_dataset))

In [ ]:
# ============================================================
# RESTORE BEST U-NET MODEL
# ============================================================

import torch

# Make sure device exists
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

BEST_MODEL_PATH = (
    "/content/drive/MyDrive/SANC/models/"
    "best_unet_saliency.pth"
)

# Recreate U-Net
model = UNet(
    in_channels=3,
    out_channels=1
).to(device)

# Load best checkpoint
checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("Best U-Net restored successfully.")
print("Device:", device)
print("Saved epoch:", checkpoint["epoch"])
print("Validation Dice:", checkpoint["val_dice"])
print("Validation IoU:", checkpoint["val_iou"])
print("Validation Loss:", checkpoint["val_loss"])

In [ ]:
# ============================================================
# FIND SAVED METADATA FILES
# ============================================================

import os

METADATA_PATH = "/content/drive/MyDrive/SANC/datasets/metadata"

print("Metadata folder exists:", os.path.exists(METADATA_PATH))

if os.path.exists(METADATA_PATH):
    files = os.listdir(METADATA_PATH)

    print("\nAvailable metadata files:")
    for f in files:
        print(f)

In [ ]:
# ============================================================
# FINAL U-NET SALIENCY VISUALIZATION
# ============================================================

import matplotlib.pyplot as plt
import torch
import numpy as np

model.eval()

DATASETS = [
    "Kvasir-SEG",
    "ORIGA",
    "HAM10000"
]

SAMPLES_PER_DATASET = 3

fig, axes = plt.subplots(
    9,
    5,
    figsize=(15, 27)
)

plot_row = 0

for dataset_name in DATASETS:

    df = test_eval_data[
        test_eval_data["dataset"] == dataset_name
    ].reset_index(drop=True)

    dataset = SANCMedicalDataset(df)

    for idx in range(
        min(SAMPLES_PER_DATASET, len(dataset))
    ):

        sample = dataset[idx]

        low_light = sample["low_light"].to(device)
        mask = sample["mask"].to(device)

        # Model prediction
        with torch.no_grad():

            logits = model(
                low_light.unsqueeze(0)
            )

            probability = torch.sigmoid(logits)

        prediction = (
            probability >= 0.5
        ).float()

        # Convert images for visualization

        original_img = (
            sample["original"]
            .cpu()
            .permute(1, 2, 0)
            .numpy()
        )

        low_light_img = (
            sample["low_light"]
            .cpu()
            .permute(1, 2, 0)
            .numpy()
        )

        ground_truth = (
            mask[0]
            .cpu()
            .numpy()
        )

        predicted_mask = (
            prediction[0, 0]
            .cpu()
            .numpy()
        )

        # Model probability / activation map
        saliency = (
            probability[0, 0]
            .cpu()
            .numpy()
        )

        # Plot

        axes[plot_row, 0].imshow(
            original_img
        )

        axes[plot_row, 1].imshow(
            low_light_img
        )

        axes[plot_row, 2].imshow(
            ground_truth,
            cmap="gray"
        )

        axes[plot_row, 3].imshow(
            predicted_mask,
            cmap="gray"
        )

        axes[plot_row, 4].imshow(
            saliency,
            cmap="hot"
        )

        axes[plot_row, 0].set_title(
            f"{dataset_name}\nOriginal"
        )

        axes[plot_row, 1].set_title(
            "Low-light"
        )

        axes[plot_row, 2].set_title(
            "Ground truth"
        )

        axes[plot_row, 3].set_title(
            "Prediction"
        )

        axes[plot_row, 4].set_title(
            "U-Net saliency"
        )

        for col in range(5):
            axes[plot_row, col].axis("off")

        plot_row += 1


plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# TRUE GRADIENT-BASED SALIENCY MAP
# ============================================================

import torch
import numpy as np
import matplotlib.pyplot as plt

model.eval()

DATASETS = [
    "Kvasir-SEG",
    "ORIGA",
    "HAM10000"
]

SAMPLES_PER_DATASET = 3


def generate_gradient_saliency(model, image, device):
    """
    Generate gradient-based saliency for a segmentation model.

    image:
        Tensor [3, H, W]

    Returns:
        probability [H, W]
        prediction [H, W]
        saliency [H, W]
    """

    # Create independent input with gradients enabled
    x = image.unsqueeze(0).to(device)
    x = x.clone().detach().requires_grad_(True)

    # Forward pass
    logits = model(x)
    probability = torch.sigmoid(logits)

    # --------------------------------------------------------
    # Select the predicted foreground response
    # --------------------------------------------------------

    pred_mask = (probability > 0.5).float()

    # If prediction is empty, use the strongest responses
    if pred_mask.sum() == 0:

        flat_logits = logits.view(-1)

        k = max(
            1,
            int(flat_logits.numel() * 0.01)
        )

        top_values = torch.topk(
            flat_logits,
            k=k
        ).values

        target = top_values.mean()

    else:

        target = (
            logits * pred_mask
        ).sum() / (
            pred_mask.sum() + 1e-8
        )

    # --------------------------------------------------------
    # Backpropagate
    # --------------------------------------------------------

    model.zero_grad(set_to_none=True)

    target.backward()

    gradients = x.grad.detach()

    # --------------------------------------------------------
    # Convert 3-channel gradients into one saliency map
    # --------------------------------------------------------

    saliency = gradients.abs().max(dim=1)[0]

    # Normalize 0-1
    saliency_min = saliency.min()
    saliency_max = saliency.max()

    saliency = (
        saliency - saliency_min
    ) / (
        saliency_max - saliency_min + 1e-8
    )

    probability = probability.detach()[0, 0]
    prediction = pred_mask.detach()[0, 0]

    return (
        probability.cpu().numpy(),
        prediction.cpu().numpy(),
        saliency[0].cpu().numpy()
    )


# ============================================================
# CREATE VISUALIZATION
# ============================================================

fig, axes = plt.subplots(
    9,
    6,
    figsize=(18, 27)
)

plot_row = 0

for dataset_name in DATASETS:

    df = test_eval_data[
        test_eval_data["dataset"] == dataset_name
    ].reset_index(drop=True)

    dataset = SANCMedicalDataset(df)

    for idx in range(
        min(SAMPLES_PER_DATASET, len(dataset))
    ):

        sample = dataset[idx]

        # ----------------------------------------------------
        # Generate true gradient saliency
        # ----------------------------------------------------

        (
            probability,
            prediction,
            saliency
        ) = generate_gradient_saliency(
            model,
            sample["low_light"],
            device
        )

        # ----------------------------------------------------
        # Prepare images
        # ----------------------------------------------------

        original = (
            sample["original"]
            .cpu()
            .permute(1, 2, 0)
            .numpy()
        )

        low_light = (
            sample["low_light"]
            .cpu()
            .permute(1, 2, 0)
            .numpy()
        )

        ground_truth = (
            sample["mask"][0]
            .cpu()
            .numpy()
        )

        # ----------------------------------------------------
        # Saliency overlay
        # ----------------------------------------------------

        # Resize not necessary because everything is 128x128
        saliency_rgb = plt.cm.hot(saliency)[..., :3]

        overlay = (
            0.55 * low_light
            + 0.45 * saliency_rgb
        )

        overlay = np.clip(
            overlay,
            0,
            1
        )

        # ----------------------------------------------------
        # Plot
        # ----------------------------------------------------

        axes[plot_row, 0].imshow(original)

        axes[plot_row, 1].imshow(low_light)

        axes[plot_row, 2].imshow(
            ground_truth,
            cmap="gray"
        )

        axes[plot_row, 3].imshow(
            prediction,
            cmap="gray"
        )

        axes[plot_row, 4].imshow(
            saliency,
            cmap="hot"
        )

        axes[plot_row, 5].imshow(overlay)

        axes[plot_row, 0].set_title(
            f"{dataset_name}\nOriginal"
        )

        axes[plot_row, 1].set_title(
            "Low-light"
        )

        axes[plot_row, 2].set_title(
            "Ground truth"
        )

        axes[plot_row, 3].set_title(
            "Prediction"
        )

        axes[plot_row, 4].set_title(
            "Gradient saliency"
        )

        axes[plot_row, 5].set_title(
            "Saliency overlay"
        )

        for col in range(6):
            axes[plot_row, col].axis("off")

        plot_row += 1


plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CHECK THE EXACT MASKS USED IN SALIENCY VISUALIZATION
# ============================================================

for dataset_name in ["Kvasir-SEG", "ORIGA", "HAM10000"]:

    print("\n" + "=" * 55)
    print(dataset_name)
    print("=" * 55)

    df = test_eval_data[
        test_eval_data["dataset"] == dataset_name
    ].reset_index(drop=True)

    dataset = SANCMedicalDataset(df)

    for idx in range(min(3, len(dataset))):

        sample = dataset[idx]

        mask = sample["mask"]

        print("\nFilename:", sample["filename"])
        print("Mask shape:", mask.shape)
        print("Unique:", torch.unique(mask))
        print("Foreground pixels:", int(mask.sum().item()))
        print("Foreground ratio:", mask.mean().item())

In [ ]:
# ============================================================
# VERIFY ACTUAL MASK FILES VS DATASET LOADER
# ============================================================

from PIL import Image
import numpy as np
import os

checks = [
    ("ORIGA", "638.png"),
    ("ORIGA", "221.png"),
    ("ORIGA", "429.png"),
    ("HAM10000", "ISIC_0029784"),
    ("HAM10000", "ISIC_0033923"),
    ("HAM10000", "ISIC_0033735"),
]

for dataset_name, filename in checks:

    print("\n" + "=" * 60)
    print(dataset_name, filename)
    print("=" * 60)

    # Find row in evaluation dataframe
    rows = test_eval_data[
        (test_eval_data["dataset"] == dataset_name) &
        (test_eval_data["filename"].astype(str) == filename)
    ]

    if len(rows) == 0:
        print("NOT FOUND in test_eval_data")
        continue

    row = rows.iloc[0]

    mask_path = row["mask"]

    print("Dataframe mask path:")
    print(mask_path)

    print("Exists:", os.path.isfile(mask_path))

    # --------------------------------------------------------
    # Read mask DIRECTLY
    # --------------------------------------------------------

    with Image.open(mask_path) as img:

        print("\nDirect PIL:")
        print("Format:", img.format)
        print("Mode:", img.mode)
        print("Size:", img.size)

        mask_np = np.array(img)

        print("Min:", mask_np.min())
        print("Max:", mask_np.max())
        print(
            "Non-zero pixels:",
            np.count_nonzero(mask_np)
        )

    # --------------------------------------------------------
    # Read through SANCMedicalDataset
    # --------------------------------------------------------

    one_dataset = SANCMedicalDataset(
        rows.reset_index(drop=True)
    )

    sample = one_dataset[0]

    mask_tensor = sample["mask"]

    print("\nThrough SANCMedicalDataset:")
    print(
        "Unique:",
        torch.unique(mask_tensor)
    )

    print(
        "Foreground pixels:",
        int(mask_tensor.sum().item())
    )

In [ ]:
# ============================================================
# FRESH DATASET CLASS — MASK HANDLING FIX
# ============================================================

import torch
import numpy as np
from torch.utils.data import Dataset
from PIL import Image


class SANCMedicalDatasetV2(Dataset):

    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        # ----------------------------------------------------
        # Load images
        # ----------------------------------------------------

        original = Image.open(
            row["original"]
        ).convert("RGB")

        low_light = Image.open(
            row["low_light"]
        ).convert("RGB")

        # IMPORTANT:
        # Do NOT alter the mask before converting to numpy
        mask = Image.open(
            row["mask"]
        ).convert("L")

        # ----------------------------------------------------
        # Convert to numpy
        # ----------------------------------------------------

        original = np.asarray(
            original,
            dtype=np.uint8
        )

        low_light = np.asarray(
            low_light,
            dtype=np.uint8
        )

        mask = np.asarray(
            mask,
            dtype=np.uint8
        )

        # ----------------------------------------------------
        # Convert images to tensors
        # ----------------------------------------------------

        original = torch.from_numpy(
            original
        ).permute(2, 0, 1).float() / 255.0

        low_light = torch.from_numpy(
            low_light
        ).permute(2, 0, 1).float() / 255.0

        # ----------------------------------------------------
        # Convert mask to binary tensor
        # ----------------------------------------------------

        mask = torch.from_numpy(
            mask
        ).float()

        mask = (mask > 0).float()

        mask = mask.unsqueeze(0)

        return {
            "original": original,
            "low_light": low_light,
            "mask": mask,
            "dataset": row["dataset"],
            "filename": row["filename"],
            "split": row["split"]
        }


print("SANCMedicalDatasetV2 defined.")

In [ ]:
# ============================================================
# EXPERIMENT B — BCE + DICE LOSS
# ============================================================

import torch
import torch.nn as nn


class BCEDiceLoss(nn.Module):

    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super().__init__()

        self.bce_weight = bce_weight
        self.dice_weight = dice_weight

        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):

        # BCE component
        bce_loss = self.bce(
            logits,
            targets
        )

        # Convert logits to probabilities
        probs = torch.sigmoid(logits)

        # Flatten each image
        probs = probs.view(
            probs.size(0),
            -1
        )

        targets = targets.view(
            targets.size(0),
            -1
        )

        # Dice
        intersection = (
            probs * targets
        ).sum(dim=1)

        dice = (
            2 * intersection + 1e-6
        ) / (
            probs.sum(dim=1)
            + targets.sum(dim=1)
            + 1e-6
        )

        dice_loss = 1 - dice.mean()

        # Combined loss
        total_loss = (
            self.bce_weight * bce_loss
            + self.dice_weight * dice_loss
        )

        return total_loss


criterion_bce_dice = BCEDiceLoss(
    bce_weight=0.5,
    dice_weight=0.5
).to(device)

print("BCE + Dice loss created.")

In [ ]:
# ============================================================
# EXPERIMENT B — TRAIN U-NET WITH BCE + DICE
# ============================================================

import os
import torch

# ------------------------------------------------------------
# New model
# ------------------------------------------------------------

model_b = UNet(
    in_channels=3,
    out_channels=1
).to(device)

# ------------------------------------------------------------
# Loss
# ------------------------------------------------------------

criterion_b = BCEDiceLoss(
    bce_weight=0.5,
    dice_weight=0.5
).to(device)

# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

optimizer_b = torch.optim.Adam(
    model_b.parameters(),
    lr=1e-4
)

# ------------------------------------------------------------
# Training settings
# ------------------------------------------------------------

NUM_EPOCHS_B = 20

best_val_dice_b = -1

SAVE_PATH_B = (
    "/content/drive/MyDrive/SANC/models/"
    "best_unet_bce_dice.pth"
)

print("Experiment B started.")
print("Epochs:", NUM_EPOCHS_B)
print("Learning rate:", 1e-4)
print("Loss: BCE + Dice")

In [ ]:
# ============================================================
# TRAINING LOOP — EXPERIMENT B
# ============================================================

for epoch in range(NUM_EPOCHS_B):

    # ========================================================
    # TRAIN
    # ========================================================

    model_b.train()

    train_loss_total = 0.0

    for batch in train_loader:

        low_light = batch["low_light"].to(device)
        mask = batch["mask"].to(device)

        optimizer_b.zero_grad()

        prediction = model_b(
            low_light
        )

        loss = criterion_b(
            prediction,
            mask
        )

        loss.backward()

        optimizer_b.step()

        train_loss_total += (
            loss.item()
        )

    train_loss = (
        train_loss_total
        / len(train_loader)
    )

    # ========================================================
    # VALIDATION
    # ========================================================

    model_b.eval()

    val_loss_total = 0.0
    val_dice_total = 0.0
    val_iou_total = 0.0

    with torch.no_grad():

        for batch in val_loader:

            low_light = batch["low_light"].to(device)
            mask = batch["mask"].to(device)

            prediction = model_b(
                low_light
            )

            loss = criterion_b(
                prediction,
                mask
            )

            val_loss_total += loss.item()

            val_dice_total += dice_score(
                prediction,
                mask
            )

            val_iou_total += iou_score(
                prediction,
                mask
            )

    val_loss = (
        val_loss_total
        / len(val_loader)
    )

    val_dice = (
        val_dice_total
        / len(val_loader)
    )

    val_iou = (
        val_iou_total
        / len(val_loader)
    )

    # ========================================================
    # SAVE BEST MODEL
    # ========================================================

    is_best = val_dice > best_val_dice_b

    if is_best:

        best_val_dice_b = val_dice

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict":
                    model_b.state_dict(),
                "optimizer_state_dict":
                    optimizer_b.state_dict(),
                "val_loss": val_loss,
                "val_dice": val_dice,
                "val_iou": val_iou
            },
            SAVE_PATH_B
        )

    print(
        f"Epoch [{epoch+1:02d}/{NUM_EPOCHS_B}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Dice: {val_dice:.4f} | "
        f"Val IoU: {val_iou:.4f}"
        + (" | BEST" if is_best else "")
    )


print("\nExperiment B training complete.")
print(
    "Best validation Dice:",
    best_val_dice_b
)
print(
    "Saved to:",
    SAVE_PATH_B
)

In [ ]:
# ============================================================
# TRAINING LOOP — EXPERIMENT B
# ============================================================

for epoch in range(NUM_EPOCHS_B):

    # ========================================================
    # TRAIN
    # ========================================================

    model_b.train()

    train_loss_total = 0.0

    for batch in train_loader:

        low_light = batch["low_light"].to(device)
        mask = batch["mask"].to(device)

        optimizer_b.zero_grad()

        prediction = model_b(
            low_light
        )

        loss = criterion_b(
            prediction,
            mask
        )

        loss.backward()

        optimizer_b.step()

        train_loss_total += (
            loss.item()
        )

    train_loss = (
        train_loss_total
        / len(train_loader)
    )

    # ========================================================
    # VALIDATION
    # ========================================================

    model_b.eval()

    val_loss_total = 0.0
    val_dice_total = 0.0
    val_iou_total = 0.0

    with torch.no_grad():

        for batch in val_loader:

            low_light = batch["low_light"].to(device)
            mask = batch["mask"].to(device)

            prediction = model_b(
                low_light
            )

            loss = criterion_b(
                prediction,
                mask
            )

            val_loss_total += loss.item()

            val_dice_total += dice_score(
                prediction,
                mask
            )

            val_iou_total += iou_score(
                prediction,
                mask
            )

    val_loss = (
        val_loss_total
        / len(val_loader)
    )

    val_dice = (
        val_dice_total
        / len(val_loader)
    )

    val_iou = (
        val_iou_total
        / len(val_loader)
    )

    # ========================================================
    # SAVE BEST MODEL
    # ========================================================

    is_best = val_dice > best_val_dice_b

    if is_best:

        best_val_dice_b = val_dice

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict":
                    model_b.state_dict(),
                "optimizer_state_dict":
                    optimizer_b.state_dict(),
                "val_loss": val_loss,
                "val_dice": val_dice,
                "val_iou": val_iou
            },
            SAVE_PATH_B
        )

    print(
        f"Epoch [{epoch+1:02d}/{NUM_EPOCHS_B}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Dice: {val_dice:.4f} | "
        f"Val IoU: {val_iou:.4f}"
        + (" | BEST" if is_best else "")
    )


print("\nExperiment B training complete.")
print(
    "Best validation Dice:",
    best_val_dice_b
)
print(
    "Saved to:",
    SAVE_PATH_B
)

In [ ]:
# ============================================================
# EXPERIMENT C — TVERSKY LOSS
# ============================================================

import torch
import torch.nn as nn


class TverskyLoss(nn.Module):

    def __init__(self, alpha=0.4, beta=0.6, smooth=1e-6):
        super().__init__()

        self.alpha = alpha
        self.beta = beta
        self.smooth = smooth

    def forward(self, logits, targets):

        probs = torch.sigmoid(logits)

        probs = probs.view(
            probs.size(0), -1
        )

        targets = targets.view(
            targets.size(0), -1
        )

        tp = (probs * targets).sum(dim=1)

        fp = (
            probs * (1 - targets)
        ).sum(dim=1)

        fn = (
            (1 - probs) * targets
        ).sum(dim=1)

        tversky = (
            tp + self.smooth
        ) / (
            tp
            + self.alpha * fp
            + self.beta * fn
            + self.smooth
        )

        return 1 - tversky.mean()


criterion_c = TverskyLoss(
    alpha=0.4,
    beta=0.6
).to(device)

print("Tversky loss created.")
print("Alpha:", 0.4)
print("Beta:", 0.6)

In [ ]:
# ============================================================
# EXPERIMENT C — DATA AUGMENTATION
# ============================================================

import random
import torchvision.transforms.functional as TF


class AugmentedSANCMedicalDataset(Dataset):

    def __init__(self, dataframe, training=False):
        self.df = dataframe.reset_index(drop=True)
        self.training = training

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        original = Image.open(
            row["original"]
        ).convert("RGB")

        low_light = Image.open(
            row["low_light"]
        ).convert("RGB")

        mask = Image.open(
            row["mask"]
        ).convert("L")

        # ----------------------------------------------------
        # Spatial augmentation
        # ----------------------------------------------------

        if self.training:

            # Horizontal flip
            if random.random() < 0.5:
                original = TF.hflip(original)
                low_light = TF.hflip(low_light)
                mask = TF.hflip(mask)

            # Vertical flip
            if random.random() < 0.3:
                original = TF.vflip(original)
                low_light = TF.vflip(low_light)
                mask = TF.vflip(mask)

            # Small rotation
            if random.random() < 0.4:

                angle = random.uniform(
                    -15,
                    15
                )

                original = TF.rotate(
                    original,
                    angle,
                    interpolation=TF.InterpolationMode.BILINEAR
                )

                low_light = TF.rotate(
                    low_light,
                    angle,
                    interpolation=TF.InterpolationMode.BILINEAR
                )

                mask = TF.rotate(
                    mask,
                    angle,
                    interpolation=TF.InterpolationMode.NEAREST
                )

            # ------------------------------------------------
            # Photometric augmentation
            # Apply ONLY to images, NOT mask
            # ------------------------------------------------

            if random.random() < 0.5:

                brightness = random.uniform(
                    0.8,
                    1.2
                )

                contrast = random.uniform(
                    0.8,
                    1.2
                )

                low_light = TF.adjust_brightness(
                    low_light,
                    brightness
                )

                low_light = TF.adjust_contrast(
                    low_light,
                    contrast
                )

        # ----------------------------------------------------
        # Convert to numpy
        # ----------------------------------------------------

        original = np.asarray(
            original,
            dtype=np.uint8
        )

        low_light = np.asarray(
            low_light,
            dtype=np.uint8
        )

        mask = np.asarray(
            mask,
            dtype=np.uint8
        )

        # ----------------------------------------------------
        # Convert images
        # ----------------------------------------------------

        original = torch.from_numpy(
            original
        ).permute(2, 0, 1).float() / 255.0

        low_light = torch.from_numpy(
            low_light
        ).permute(2, 0, 1).float() / 255.0

        # ----------------------------------------------------
        # Binary mask
        # ----------------------------------------------------

        mask = torch.from_numpy(
            mask
        ).float()

        mask = (
            mask > 0
        ).float()

        mask = mask.unsqueeze(0)

        return {
            "original": original,
            "low_light": low_light,
            "mask": mask,
            "dataset": row["dataset"],
            "filename": row["filename"],
            "split": row["split"]
        }


print("Augmented dataset class created.")

In [ ]:
# ============================================================
# EXPERIMENT C — NEW U-NET
# ============================================================

model_c = UNet(
    in_channels=3,
    out_channels=1
).to(device)

criterion_c = TverskyLoss(
    alpha=0.4,
    beta=0.6
).to(device)

optimizer_c = torch.optim.Adam(
    model_c.parameters(),
    lr=1e-4
)

NUM_EPOCHS_C = 30

best_val_dice_c = -1

patience = 7
epochs_without_improvement = 0

SAVE_PATH_C = (
    "/content/drive/MyDrive/SANC/models/"
    "best_unet_tversky_augmented.pth"
)

print("Experiment C ready.")
print("Epochs:", NUM_EPOCHS_C)
print("Loss: Tversky")
print("Alpha:", 0.4)
print("Beta:", 0.6)
print("Early stopping patience:", patience)

In [ ]:
# ============================================================
# EXPERIMENT C — TRAINING
# ============================================================

for epoch in range(NUM_EPOCHS_C):

    # ========================================================
    # TRAIN
    # ========================================================

    model_c.train()

    train_loss_total = 0.0

    for batch in train_loader_c:

        low_light = batch["low_light"].to(device)
        mask = batch["mask"].to(device)

        optimizer_c.zero_grad()

        logits = model_c(
            low_light
        )

        loss = criterion_c(
            logits,
            mask
        )

        loss.backward()

        optimizer_c.step()

        train_loss_total += loss.item()

    train_loss = (
        train_loss_total
        / len(train_loader_c)
    )

    # ========================================================
    # VALIDATION
    # ========================================================

    model_c.eval()

    val_loss_total = 0.0
    val_dice_total = 0.0
    val_iou_total = 0.0

    with torch.no_grad():

        for batch in val_loader_c:

            low_light = batch["low_light"].to(device)
            mask = batch["mask"].to(device)

            logits = model_c(
                low_light
            )

            loss = criterion_c(
                logits,
                mask
            )

            val_loss_total += loss.item()

            val_dice_total += dice_score(
                logits,
                mask
            )

            val_iou_total += iou_score(
                logits,
                mask
            )

    val_loss = (
        val_loss_total
        / len(val_loader_c)
    )

    val_dice = (
        val_dice_total
        / len(val_loader_c)
    )

    val_iou = (
        val_iou_total
        / len(val_loader_c)
    )

    # ========================================================
    # BEST MODEL
    # ========================================================

    if val_dice > best_val_dice_c:

        best_val_dice_c = val_dice

        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict":
                    model_c.state_dict(),
                "optimizer_state_dict":
                    optimizer_c.state_dict(),
                "val_loss": val_loss,
                "val_dice": val_dice,
                "val_iou": val_iou
            },
            SAVE_PATH_C
        )

        best_text = " | BEST"

    else:

        epochs_without_improvement += 1

        best_text = ""

    print(
        f"Epoch [{epoch+1:02d}/{NUM_EPOCHS_C}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Dice: {val_dice:.4f} | "
        f"Val IoU: {val_iou:.4f}"
        f"{best_text}"
    )

    # ========================================================
    # EARLY STOPPING
    # ========================================================

    if epochs_without_improvement >= patience:

        print(
            "\nEarly stopping triggered."
        )

        break


print("\n" + "=" * 60)
print("EXPERIMENT C COMPLETE")
print("=" * 60)

print(
    "Best validation Dice:",
    best_val_dice_c
)

print(
    "Saved to:",
    SAVE_PATH_C
)

In [ ]:
# ============================================================
# EXPERIMENT D — ATTENTION U-NET
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F


# ------------------------------------------------------------
# Double convolution block
# ------------------------------------------------------------

class DoubleConv(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


# ------------------------------------------------------------
# Attention Gate
# ------------------------------------------------------------

class AttentionGate(nn.Module):

    def __init__(
        self,
        gating_channels,
        skip_channels,
        inter_channels
    ):
        super().__init__()

        self.W_g = nn.Sequential(
            nn.Conv2d(
                gating_channels,
                inter_channels,
                kernel_size=1,
                stride=1,
                padding=0,
                bias=True
            ),
            nn.BatchNorm2d(inter_channels)
        )

        self.W_x = nn.Sequential(
            nn.Conv2d(
                skip_channels,
                inter_channels,
                kernel_size=1,
                stride=1,
                padding=0,
                bias=True
            ),
            nn.BatchNorm2d(inter_channels)
        )

        self.psi = nn.Sequential(
            nn.Conv2d(
                inter_channels,
                1,
                kernel_size=1,
                stride=1,
                padding=0,
                bias=True
            ),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )

        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):

        # g = decoder/gating signal
        # x = encoder skip connection

        g1 = self.W_g(g)
        x1 = self.W_x(x)

        # Make spatial dimensions match
        if g1.shape[-2:] != x1.shape[-2:]:

            g1 = F.interpolate(
                g1,
                size=x1.shape[-2:],
                mode="bilinear",
                align_corners=False
            )

        psi = self.relu(
            g1 + x1
        )

        psi = self.psi(psi)

        return x * psi


# ------------------------------------------------------------
# Attention U-Net
# ------------------------------------------------------------

class AttentionUNet(nn.Module):

    def __init__(
        self,
        in_channels=3,
        out_channels=1
    ):
        super().__init__()

        # Encoder
        self.enc1 = DoubleConv(
            in_channels,
            64
        )

        self.enc2 = DoubleConv(
            64,
            128
        )

        self.enc3 = DoubleConv(
            128,
            256
        )

        self.enc4 = DoubleConv(
            256,
            512
        )

        # Bottleneck
        self.bottleneck = DoubleConv(
            512,
            1024
        )

        self.pool = nn.MaxPool2d(
            kernel_size=2,
            stride=2
        )

        # Decoder
        self.up4 = nn.ConvTranspose2d(
            1024,
            512,
            kernel_size=2,
            stride=2
        )

        self.att4 = AttentionGate(
            gating_channels=512,
            skip_channels=512,
            inter_channels=256
        )

        self.dec4 = DoubleConv(
            1024,
            512
        )

        self.up3 = nn.ConvTranspose2d(
            512,
            256,
            kernel_size=2,
            stride=2
        )

        self.att3 = AttentionGate(
            gating_channels=256,
            skip_channels=256,
            inter_channels=128
        )

        self.dec3 = DoubleConv(
            512,
            256
        )

        self.up2 = nn.ConvTranspose2d(
            256,
            128,
            kernel_size=2,
            stride=2
        )

        self.att2 = AttentionGate(
            gating_channels=128,
            skip_channels=128,
            inter_channels=64
        )

        self.dec2 = DoubleConv(
            256,
            128
        )

        self.up1 = nn.ConvTranspose2d(
            128,
            64,
            kernel_size=2,
            stride=2
        )

        self.att1 = AttentionGate(
            gating_channels=64,
            skip_channels=64,
            inter_channels=32
        )

        self.dec1 = DoubleConv(
            128,
            64
        )

        self.final = nn.Conv2d(
            64,
            out_channels,
            kernel_size=1
        )

    def forward(self, x):

        # Encoder
        e1 = self.enc1(x)

        e2 = self.enc2(
            self.pool(e1)
        )

        e3 = self.enc3(
            self.pool(e2)
        )

        e4 = self.enc4(
            self.pool(e3)
        )

        # Bottleneck
        b = self.bottleneck(
            self.pool(e4)
        )

        # Decoder 4
        d4 = self.up4(b)

        a4 = self.att4(
            d4,
            e4
        )

        d4 = torch.cat(
            [d4, a4],
            dim=1
        )

        d4 = self.dec4(d4)

        # Decoder 3
        d3 = self.up3(d4)

        a3 = self.att3(
            d3,
            e3
        )

        d3 = torch.cat(
            [d3, a3],
            dim=1
        )

        d3 = self.dec3(d3)

        # Decoder 2
        d2 = self.up2(d3)

        a2 = self.att2(
            d2,
            e2
        )

        d2 = torch.cat(
            [d2, a2],
            dim=1
        )

        d2 = self.dec2(d2)

        # Decoder 1
        d1 = self.up1(d2)

        a1 = self.att1(
            d1,
            e1
        )

        d1 = torch.cat(
            [d1, a1],
            dim=1
        )

        d1 = self.dec1(d1)

        return self.final(d1)


print("Attention U-Net defined.")

In [ ]:
# ============================================================
# TEST ATTENTION U-NET
# ============================================================

attention_model = AttentionUNet(
    in_channels=3,
    out_channels=1
).to(device)

test_batch = next(iter(train_loader))

test_input = test_batch["low_light"].to(device)

with torch.no_grad():

    test_output = attention_model(
        test_input
    )

print("Input shape :", test_input.shape)
print("Output shape:", test_output.shape)

print(
    "Expected output:",
    test_batch["mask"].shape
)

In [ ]:
# ============================================================
# EXPERIMENT D — TRAIN ATTENTION U-NET
# ============================================================

import torch

attention_model = AttentionUNet(
    in_channels=3,
    out_channels=1
).to(device)

# Use the ORIGINAL loss for a fair comparison
criterion_d = nn.BCEWithLogitsLoss()

optimizer_d = torch.optim.Adam(
    attention_model.parameters(),
    lr=1e-4
)

NUM_EPOCHS_D = 20

best_val_dice_d = -1

SAVE_PATH_D = (
    "/content/drive/MyDrive/SANC/models/"
    "best_attention_unet.pth"
)

print("Experiment D ready.")
print("Architecture: Attention U-Net")
print("Loss: BCEWithLogitsLoss")
print("Learning rate:", 1e-4)
print("Epochs:", NUM_EPOCHS_D)

In [ ]:
# ============================================================
# EXPERIMENT D — TRAINING LOOP
# ============================================================

for epoch in range(NUM_EPOCHS_D):

    # ========================================================
    # TRAIN
    # ========================================================

    attention_model.train()

    train_loss_total = 0.0

    for batch in train_loader:

        low_light = batch["low_light"].to(device)
        mask = batch["mask"].to(device)

        optimizer_d.zero_grad()

        logits = attention_model(
            low_light
        )

        loss = criterion_d(
            logits,
            mask
        )

        loss.backward()

        optimizer_d.step()

        train_loss_total += loss.item()

    train_loss = (
        train_loss_total
        / len(train_loader)
    )

    # ========================================================
    # VALIDATION
    # ========================================================

    attention_model.eval()

    val_loss_total = 0.0
    val_dice_total = 0.0
    val_iou_total = 0.0

    with torch.no_grad():

        for batch in val_loader:

            low_light = batch["low_light"].to(device)
            mask = batch["mask"].to(device)

            logits = attention_model(
                low_light
            )

            loss = criterion_d(
                logits,
                mask
            )

            val_loss_total += loss.item()

            val_dice_total += dice_score(
                logits,
                mask
            )

            val_iou_total += iou_score(
                logits,
                mask
            )

    val_loss = (
        val_loss_total
        / len(val_loader)
    )

    val_dice = (
        val_dice_total
        / len(val_loader)
    )

    val_iou = (
        val_iou_total
        / len(val_loader)
    )

    # ========================================================
    # SAVE BEST
    # ========================================================

    if val_dice > best_val_dice_d:

        best_val_dice_d = val_dice

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict":
                    attention_model.state_dict(),
                "optimizer_state_dict":
                    optimizer_d.state_dict(),
                "val_loss": val_loss,
                "val_dice": val_dice,
                "val_iou": val_iou
            },
            SAVE_PATH_D
        )

        best_text = " | BEST"

    else:

        best_text = ""

    print(
        f"Epoch [{epoch+1:02d}/{NUM_EPOCHS_D}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Dice: {val_dice:.4f} | "
        f"Val IoU: {val_iou:.4f}"
        f"{best_text}"
    )


print("\n" + "=" * 60)
print("EXPERIMENT D COMPLETE")
print("=" * 60)

print(
    "Best validation Dice:",
    best_val_dice_d
)

print(
    "Saved to:",
    SAVE_PATH_D
)

In [ ]:
# ============================================================
# EXPERIMENT E — UNET++
# ============================================================

import torch
import torch.nn as nn


class ConvBlock(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class UNetPlusPlus(nn.Module):

    def __init__(
        self,
        in_channels=3,
        out_channels=1
    ):
        super().__init__()

        # ====================================================
        # Encoder
        # ====================================================

        self.conv00 = ConvBlock(
            in_channels, 64
        )

        self.conv10 = ConvBlock(
            64, 128
        )

        self.conv20 = ConvBlock(
            128, 256
        )

        self.conv30 = ConvBlock(
            256, 512
        )

        self.conv40 = ConvBlock(
            512, 1024
        )

        self.pool = nn.MaxPool2d(
            2, 2
        )

        # ====================================================
        # Nested decoder nodes
        # ====================================================

        self.conv01 = ConvBlock(
            64 + 128,
            64
        )

        self.conv11 = ConvBlock(
            128 + 256,
            128
        )

        self.conv21 = ConvBlock(
            256 + 512,
            256
        )

        self.conv31 = ConvBlock(
            512 + 1024,
            512
        )

        self.conv02 = ConvBlock(
            64 * 2 + 128,
            64
        )

        self.conv12 = ConvBlock(
            128 * 2 + 256,
            128
        )

        self.conv22 = ConvBlock(
            256 * 2 + 512,
            256
        )

        self.conv03 = ConvBlock(
            64 * 3 + 128,
            64
        )

        self.conv13 = ConvBlock(
            128 * 3 + 256,
            128
        )

        self.conv04 = ConvBlock(
            64 * 4 + 128,
            64
        )

        # ====================================================
        # Final segmentation layer
        # ====================================================

        self.final = nn.Conv2d(
            64,
            out_channels,
            kernel_size=1
        )

    def upsample(self, x):

        return nn.functional.interpolate(
            x,
            scale_factor=2,
            mode="bilinear",
            align_corners=False
        )

    def forward(self, x):

        # ====================================================
        # Encoder
        # ====================================================

        x00 = self.conv00(x)

        x10 = self.conv10(
            self.pool(x00)
        )

        x20 = self.conv20(
            self.pool(x10)
        )

        x30 = self.conv30(
            self.pool(x20)
        )

        x40 = self.conv40(
            self.pool(x30)
        )

        # ====================================================
        # Nested pathway 1
        # ====================================================

        x01 = self.conv01(
            torch.cat(
                [
                    x00,
                    self.upsample(x10)
                ],
                dim=1
            )
        )

        x11 = self.conv11(
            torch.cat(
                [
                    x10,
                    self.upsample(x20)
                ],
                dim=1
            )
        )

        x21 = self.conv21(
            torch.cat(
                [
                    x20,
                    self.upsample(x30)
                ],
                dim=1
            )
        )

        x31 = self.conv31(
            torch.cat(
                [
                    x30,
                    self.upsample(x40)
                ],
                dim=1
            )
        )

        # ====================================================
        # Nested pathway 2
        # ====================================================

        x02 = self.conv02(
            torch.cat(
                [
                    x00,
                    x01,
                    self.upsample(x11)
                ],
                dim=1
            )
        )

        x12 = self.conv12(
            torch.cat(
                [
                    x10,
                    x11,
                    self.upsample(x21)
                ],
                dim=1
            )
        )

        x22 = self.conv22(
            torch.cat(
                [
                    x20,
                    x21,
                    self.upsample(x31)
                ],
                dim=1
            )
        )

        # ====================================================
        # Nested pathway 3
        # ====================================================

        x03 = self.conv03(
            torch.cat(
                [
                    x00,
                    x01,
                    x02,
                    self.upsample(x12)
                ],
                dim=1
            )
        )

        x13 = self.conv13(
            torch.cat(
                [
                    x10,
                    x11,
                    x12,
                    self.upsample(x22)
                ],
                dim=1
            )
        )

        # ====================================================
        # Nested pathway 4
        # ====================================================

        x04 = self.conv04(
            torch.cat(
                [
                    x00,
                    x01,
                    x02,
                    x03,
                    self.upsample(x13)
                ],
                dim=1
            )
        )

        # ====================================================
        # Output
        # ====================================================

        return self.final(x04)


print("UNet++ defined.")

In [ ]:
# ============================================================
# TEST UNET++
# ============================================================

unetpp_model = UNetPlusPlus(
    in_channels=3,
    out_channels=1
).to(device)

batch = next(iter(train_loader))

x = batch["low_light"].to(device)

with torch.no_grad():

    output = unetpp_model(x)

print("Input shape :", x.shape)
print("Output shape:", output.shape)
print("Expected    :", batch["mask"].shape)

In [ ]:
# ============================================================
# EXPERIMENT E — UNET++ TRAINING SETUP
# ============================================================

unetpp_model = UNetPlusPlus(
    in_channels=3,
    out_channels=1
).to(device)

criterion_e = nn.BCEWithLogitsLoss()

optimizer_e = torch.optim.Adam(
    unetpp_model.parameters(),
    lr=1e-4
)

NUM_EPOCHS_E = 20

best_val_dice_e = -1

SAVE_PATH_E = (
    "/content/drive/MyDrive/SANC/models/"
    "best_unetpp_saliency.pth"
)

print("Experiment E ready.")
print("Architecture: UNet++")
print("Loss: BCEWithLogitsLoss")
print("Learning rate:", 1e-4)
print("Epochs:", NUM_EPOCHS_E)

In [ ]:
# ============================================================
# EXPERIMENT E — UNET++ TRAINING
# ============================================================

for epoch in range(NUM_EPOCHS_E):

    # ========================================================
    # TRAIN
    # ========================================================

    unetpp_model.train()

    train_loss_total = 0.0

    for batch in train_loader:

        low_light = batch["low_light"].to(device)
        mask = batch["mask"].to(device)

        optimizer_e.zero_grad()

        logits = unetpp_model(
            low_light
        )

        loss = criterion_e(
            logits,
            mask
        )

        loss.backward()

        optimizer_e.step()

        train_loss_total += loss.item()

    train_loss = (
        train_loss_total
        / len(train_loader)
    )

    # ========================================================
    # VALIDATION
    # ========================================================

    unetpp_model.eval()

    val_loss_total = 0.0
    val_dice_total = 0.0
    val_iou_total = 0.0

    with torch.no_grad():

        for batch in val_loader:

            low_light = batch["low_light"].to(device)
            mask = batch["mask"].to(device)

            logits = unetpp_model(
                low_light
            )

            loss = criterion_e(
                logits,
                mask
            )

            val_loss_total += loss.item()

            val_dice_total += dice_score(
                logits,
                mask
            )

            val_iou_total += iou_score(
                logits,
                mask
            )

    val_loss = (
        val_loss_total
        / len(val_loader)
    )

    val_dice = (
        val_dice_total
        / len(val_loader)
    )

    val_iou = (
        val_iou_total
        / len(val_loader)
    )

    # ========================================================
    # SAVE BEST MODEL
    # ========================================================

    if val_dice > best_val_dice_e:

        best_val_dice_e = val_dice

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict":
                    unetpp_model.state_dict(),
                "optimizer_state_dict":
                    optimizer_e.state_dict(),
                "val_loss": val_loss,
                "val_dice": val_dice,
                "val_iou": val_iou
            },
            SAVE_PATH_E
        )

        best_text = " | BEST"

    else:

        best_text = ""

    print(
        f"Epoch [{epoch+1:02d}/{NUM_EPOCHS_E}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Dice: {val_dice:.4f} | "
        f"Val IoU: {val_iou:.4f}"
        f"{best_text}"
    )


print("\n" + "=" * 60)
print("EXPERIMENT E COMPLETE")
print("=" * 60)

print(
    "Best validation Dice:",
    best_val_dice_e
)

print(
    "Saved to:",
    SAVE_PATH_E
)

In [ ]:
# ============================================================
# EXPERIMENT F — FINE-TUNE ORIGINAL U-NET
# ============================================================

import os
import torch

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

ORIGINAL_CHECKPOINT = (
    "/content/drive/MyDrive/SANC/models/"
    "best_unet_saliency.pth"
)

FINETUNED_CHECKPOINT = (
    "/content/drive/MyDrive/SANC/models/"
    "best_unet_finetuned.pth"
)

# ------------------------------------------------------------
# Recreate original U-Net
# ------------------------------------------------------------

model_ft = UNet(
    in_channels=3,
    out_channels=1
).to(device)

# ------------------------------------------------------------
# Load original best checkpoint
# ------------------------------------------------------------

checkpoint = torch.load(
    ORIGINAL_CHECKPOINT,
    map_location=device
)

model_ft.load_state_dict(
    checkpoint["model_state_dict"]
)

print("Original checkpoint loaded.")
print(
    "Checkpoint epoch:",
    checkpoint.get("epoch", "unknown")
)
print(
    "Checkpoint Val Dice:",
    checkpoint.get("val_dice", "unknown")
)
print(
    "Checkpoint Val IoU:",
    checkpoint.get("val_iou", "unknown")
)

# ------------------------------------------------------------
# Original loss
# ------------------------------------------------------------

criterion_ft = nn.BCEWithLogitsLoss()

# ------------------------------------------------------------
# Lower learning rate for fine-tuning
# ------------------------------------------------------------

optimizer_ft = torch.optim.Adam(
    model_ft.parameters(),
    lr=1e-5
)

# ------------------------------------------------------------
# Fine-tuning settings
# ------------------------------------------------------------

NUM_EPOCHS_FT = 15

best_val_dice_ft = float(
    checkpoint.get("val_dice", 0.0)
)

epochs_without_improvement = 0

PATIENCE_FT = 5

print("\nFine-tuning configuration:")
print("Learning rate:", 1e-5)
print("Additional epochs:", NUM_EPOCHS_FT)
print("Early stopping patience:", PATIENCE_FT)
print("Previous best Dice:", best_val_dice_ft)
print("Save path:", FINETUNED_CHECKPOINT)

In [ ]:
# ============================================================
# LOAD ORIGINAL U-NET FOR FINE-TUNING
# ============================================================

model_ft = OriginalUNet(
    in_channels=3,
    out_channels=1
).to(device)

checkpoint = torch.load(
    "/content/drive/MyDrive/SANC/models/best_unet_saliency.pth",
    map_location=device
)

model_ft.load_state_dict(
    checkpoint["model_state_dict"]
)

print("Original U-Net checkpoint loaded successfully.")

print(
    "Starting from epoch:",
    checkpoint["epoch"]
)

print(
    "Starting Val Dice:",
    checkpoint["val_dice"]
)

print(
    "Starting Val IoU:",
    checkpoint["val_iou"]
)

In [ ]:
# ============================================================
# LOAD ORIGINAL BEST MODEL
# ============================================================

ORIGINAL_CHECKPOINT = (
    "/content/drive/MyDrive/SANC/models/"
    "best_unet_saliency.pth"
)

checkpoint = torch.load(
    ORIGINAL_CHECKPOINT,
    map_location=device
)

model_ft = OriginalUNet(
    in_channels=3,
    out_channels=1
).to(device)

model_ft.load_state_dict(
    checkpoint["model_state_dict"]
)

print("=" * 60)
print("CHECKPOINT LOADED SUCCESSFULLY")
print("=" * 60)

print(
    "Checkpoint epoch:",
    checkpoint["epoch"]
)

print(
    "Validation Dice:",
    checkpoint["val_dice"]
)

print(
    "Validation IoU:",
    checkpoint["val_iou"]
)

In [ ]:
# ============================================================
# EXPERIMENT F — FINE-TUNING SETUP
# ============================================================

criterion_ft = nn.BCEWithLogitsLoss()

optimizer_ft = torch.optim.Adam(
    model_ft.parameters(),
    lr=1e-5
)

NUM_EPOCHS_FT = 15

best_val_dice_ft = checkpoint["val_dice"]

epochs_without_improvement = 0
PATIENCE_FT = 5

FINETUNED_CHECKPOINT = (
    "/content/drive/MyDrive/SANC/models/"
    "best_unet_finetuned.pth"
)

print("=" * 60)
print("EXPERIMENT F — ORIGINAL U-NET FINE-TUNING")
print("=" * 60)

print("Starting epoch:", checkpoint["epoch"])
print("Starting Val Dice:", best_val_dice_ft)
print("Starting Val IoU:", checkpoint["val_iou"])
print("Learning rate:", 1e-5)
print("Additional epochs:", NUM_EPOCHS_FT)
print("Early stopping patience:", PATIENCE_FT)
print("Save path:", FINETUNED_CHECKPOINT)

In [ ]:
# ============================================================
# EXPERIMENT F — FINE-TUNING LOOP
# ============================================================

for epoch in range(NUM_EPOCHS_FT):

    # ========================================================
    # TRAIN
    # ========================================================

    model_ft.train()

    train_loss_total = 0.0

    for batch in train_loader:

        low_light = batch["low_light"].to(device)
        mask = batch["mask"].to(device)

        optimizer_ft.zero_grad()

        logits = model_ft(
            low_light
        )

        loss = criterion_ft(
            logits,
            mask
        )

        loss.backward()

        optimizer_ft.step()

        train_loss_total += loss.item()

    train_loss = (
        train_loss_total
        / len(train_loader)
    )

    # ========================================================
    # VALIDATION
    # ========================================================

    model_ft.eval()

    val_loss_total = 0.0
    val_dice_total = 0.0
    val_iou_total = 0.0

    with torch.no_grad():

        for batch in val_loader:

            low_light = batch["low_light"].to(device)
            mask = batch["mask"].to(device)

            logits = model_ft(
                low_light
            )

            loss = criterion_ft(
                logits,
                mask
            )

            val_loss_total += loss.item()

            val_dice_total += dice_score(
                logits,
                mask
            )

            val_iou_total += iou_score(
                logits,
                mask
            )

    val_loss = (
        val_loss_total
        / len(val_loader)
    )

    val_dice = (
        val_dice_total
        / len(val_loader)
    )

    val_iou = (
        val_iou_total
        / len(val_loader)
    )

    # ========================================================
    # SAVE ONLY IF BETTER
    # ========================================================

    if val_dice > best_val_dice_ft:

        best_val_dice_ft = val_dice

        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": checkpoint["epoch"] + epoch + 1,
                "model_state_dict":
                    model_ft.state_dict(),
                "optimizer_state_dict":
                    optimizer_ft.state_dict(),
                "val_loss": val_loss,
                "val_dice": val_dice,
                "val_iou": val_iou
            },
            FINETUNED_CHECKPOINT
        )

        best_text = " | BEST"

    else:

        epochs_without_improvement += 1

        best_text = ""

    print(
        f"Fine-tune [{epoch+1:02d}/{NUM_EPOCHS_FT}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Dice: {val_dice:.4f} | "
        f"Val IoU: {val_iou:.4f}"
        f"{best_text}"
    )

    # ========================================================
    # EARLY STOPPING
    # ========================================================

    if epochs_without_improvement >= PATIENCE_FT:

        print(
            "\nEarly stopping triggered."
        )

        break


print("\n" + "=" * 60)
print("EXPERIMENT F COMPLETE")
print("=" * 60)

print(
    "Original baseline Dice:",
    checkpoint["val_dice"]
)

print(
    "Fine-tuned best Dice:",
    best_val_dice_ft
)

print(
    "Saved to:",
    FINETUNED_CHECKPOINT
)

In [ ]:
BEST_UNET_PATH = (
    "/content/drive/MyDrive/SANC/models/"
    "best_unet_saliency.pth"
)

checkpoint = torch.load(
    BEST_UNET_PATH,
    map_location=device
)


saliency_model = OriginalUNet(
    in_channels=3,
    out_channels=1
).to(device)

saliency_model.load_state_dict(
    checkpoint["model_state_dict"]
)

saliency_model.eval()

print("Best U-Net loaded.")
print("Val Dice:", checkpoint["val_dice"])
print("Val IoU:", checkpoint["val_iou"])

In [ ]:
import matplotlib.pyplot as plt
import torch

batch = next(iter(val_loader))

low_light = batch["low_light"].to(device)
ground_truth = batch["mask"].to(device)

with torch.no_grad():
    logits = saliency_model(low_light)
    saliency = torch.sigmoid(logits)

# First sample
image = low_light[0].cpu().permute(1, 2, 0).numpy()
gt = ground_truth[0, 0].cpu().numpy()
sal = saliency[0, 0].cpu().numpy()
pred = (sal > 0.5).astype(float)

plt.figure(figsize=(12, 3))

plt.subplot(1, 4, 1)
plt.imshow(image)
plt.title("Low-light")
plt.axis("off")

plt.subplot(1, 4, 2)
plt.imshow(gt, cmap="gray")
plt.title("Ground Truth")
plt.axis("off")

plt.subplot(1, 4, 3)
plt.imshow(sal, cmap="gray", vmin=0, vmax=1)
plt.title("Continuous Saliency")
plt.axis("off")

plt.subplot(1, 4, 4)
plt.imshow(pred, cmap="gray")
plt.title("Binary Prediction")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CONTINUOUS SALIENCY ANALYSIS
# ============================================================

import torch
import numpy as np

saliency_model.eval()

thresholds = [
    0.20,
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.80
]

results = {
    t: {
        "dice": [],
        "iou": []
    }
    for t in thresholds
}

soft_dice_values = []

with torch.no_grad():

    for batch in val_loader:

        low_light = batch["low_light"].to(device)
        mask = batch["mask"].to(device)

        logits = saliency_model(
            low_light
        )

        saliency = torch.sigmoid(
            logits
        )

        # ----------------------------------------------------
        # Soft Dice
        # ----------------------------------------------------

        smooth = 1e-6

        intersection = (
            saliency * mask
        ).sum(dim=(1, 2, 3))

        soft_dice = (
            2 * intersection + smooth
        ) / (
            saliency.sum(dim=(1, 2, 3))
            + mask.sum(dim=(1, 2, 3))
            + smooth
        )

        soft_dice_values.extend(
            soft_dice.cpu().numpy()
        )

        # ----------------------------------------------------
        # Threshold sweep
        # ----------------------------------------------------

        for threshold in thresholds:

            pred = (
                saliency >= threshold
            ).float()

            intersection = (
                pred * mask
            ).sum(dim=(1, 2, 3))

            pred_sum = pred.sum(
                dim=(1, 2, 3)
            )

            mask_sum = mask.sum(
                dim=(1, 2, 3)
            )

            dice = (
                2 * intersection + smooth
            ) / (
                pred_sum
                + mask_sum
                + smooth
            )

            union = (
                pred_sum
                + mask_sum
                - intersection
            )

            iou = (
                intersection + smooth
            ) / (
                union + smooth
            )

            results[threshold]["dice"].extend(
                dice.cpu().numpy()
            )

            results[threshold]["iou"].extend(
                iou.cpu().numpy()
            )


# ============================================================
# PRINT RESULTS
# ============================================================

print("=" * 60)
print("CONTINUOUS SALIENCY ANALYSIS")
print("=" * 60)

print(
    "Soft Dice:",
    round(float(np.mean(soft_dice_values)), 4)
)

print()

for threshold in thresholds:

    mean_dice = np.mean(
        results[threshold]["dice"]
    )

    mean_iou = np.mean(
        results[threshold]["iou"]
    )

    print(
        f"Threshold {threshold:.2f} | "
        f"Dice: {mean_dice:.4f} | "
        f"IoU: {mean_iou:.4f}"
    )

In [ ]:
# ============================================================
# KVASIR-SEG — OVERSEGMENTATION / UNDERSEGMENTATION ANALYSIS
# ============================================================

import torch
import numpy as np

saliency_model.eval()

results_kvasir = []

with torch.no_grad():

    for batch in test_eval_loader:

        low_light = batch["low_light"].to(device)
        mask = batch["mask"].to(device)

        logits = saliency_model(low_light)
        probability = torch.sigmoid(logits)

        # Use our best threshold
        prediction = (
            probability >= 0.5
        ).float()

        for i in range(
            low_light.shape[0]
        ):

            gt = mask[i, 0]
            pred = prediction[i, 0]

            # ----------------------------------------------
            # Pixel-level errors
            # ----------------------------------------------

            tp = (
                (pred == 1) &
                (gt == 1)
            ).sum().item()

            fp = (
                (pred == 1) &
                (gt == 0)
            ).sum().item()

            fn = (
                (pred == 0) &
                (gt == 1)
            ).sum().item()

            gt_pixels = (
                gt == 1
            ).sum().item()

            pred_pixels = (
                pred == 1
            ).sum().item()

            # ----------------------------------------------
            # Dice / IoU
            # ----------------------------------------------

            dice = (
                2 * tp
                / (2 * tp + fp + fn + 1e-8)
            )

            iou = (
                tp
                / (tp + fp + fn + 1e-8)
            )

            # ----------------------------------------------
            # Error ratios
            # ----------------------------------------------

            fp_ratio = (
                fp
                / (gt_pixels + 1e-8)
            )

            fn_ratio = (
                fn
                / (gt_pixels + 1e-8)
            )

            # ----------------------------------------------
            # Classify error type
            # ----------------------------------------------

            if dice >= 0.70:

                error_type = "GOOD"

            elif fp > fn * 1.25:

                error_type = "OVERSEGMENTATION"

            elif fn > fp * 1.25:

                error_type = "UNDERSEGMENTATION"

            else:

                error_type = "MIXED"

            results_kvasir.append({

                "filename":
                    batch["filename"][i],

                "dice":
                    dice,

                "iou":
                    iou,

                "gt_pixels":
                    gt_pixels,

                "pred_pixels":
                    pred_pixels,

                "fp":
                    fp,

                "fn":
                    fn,

                "fp_ratio":
                    fp_ratio,

                "fn_ratio":
                    fn_ratio,

                "error_type":
                    error_type
            })


# ============================================================
# SUMMARY
# ============================================================

print("=" * 60)
print("KVASIR-SEG ERROR ANALYSIS")
print("=" * 60)

print(
    "Total samples:",
    len(results_kvasir)
)

error_types = [
    "GOOD",
    "OVERSEGMENTATION",
    "UNDERSEGMENTATION",
    "MIXED"
]

for error_type in error_types:

    subset = [
        r for r in results_kvasir
        if r["error_type"] == error_type
    ]

    print(
        f"{error_type:20s}: {len(subset)}"
    )


# ============================================================
# AVERAGES
# ============================================================

mean_dice = np.mean([
    r["dice"]
    for r in results_kvasir
])

mean_iou = np.mean([
    r["iou"]
    for r in results_kvasir
])

mean_fp = np.mean([
    r["fp"]
    for r in results_kvasir
])

mean_fn = np.mean([
    r["fn"]
    for r in results_kvasir
])

mean_fp_ratio = np.mean([
    r["fp_ratio"]
    for r in results_kvasir
])

mean_fn_ratio = np.mean([
    r["fn_ratio"]
    for r in results_kvasir
])

print()
print("Average Dice:", round(mean_dice, 4))
print("Average IoU :", round(mean_iou, 4))
print("Average FP  :", round(mean_fp, 2))
print("Average FN  :", round(mean_fn, 2))
print(
    "Average FP ratio:",
    round(mean_fp_ratio, 4)
)
print(
    "Average FN ratio:",
    round(mean_fn_ratio, 4)
)


# ============================================================
# WORST OVERSEGMENTATION
# ============================================================

print()
print("=" * 60)
print("WORST OVERSEGMENTATION CASES")
print("=" * 60)

over_cases = sorted(
    results_kvasir,
    key=lambda x: x["fp"],
    reverse=True
)

for r in over_cases[:10]:

    print(
        f'{r["filename"]} | '
        f'Dice: {r["dice"]:.4f} | '
        f'GT: {r["gt_pixels"]} | '
        f'Pred: {r["pred_pixels"]} | '
        f'FP: {r["fp"]} | '
        f'FN: {r["fn"]}'
    )


# ============================================================
# WORST UNDERSEGMENTATION
# ============================================================

print()
print("=" * 60)
print("WORST UNDERSEGMENTATION CASES")
print("=" * 60)

under_cases = sorted(
    results_kvasir,
    key=lambda x: x["fn"],
    reverse=True
)

for r in under_cases[:10]:

    print(
        f'{r["filename"]} | '
        f'Dice: {r["dice"]:.4f} | '
        f'GT: {r["gt_pixels"]} | '
        f'Pred: {r["pred_pixels"]} | '
        f'FP: {r["fp"]} | '
        f'FN: {r["fn"]}'
    )

In [ ]:
# Check available loader
print("test_loader exists:", "test_loader" in globals())

if "test_loader" in globals():
    print("Number of test batches:", len(test_loader))
    print("Test samples:", len(test_loader.dataset))

In [ ]:
# ============================================================
# EXPERIMENT G — SALIENCY REFINEMENT
# No retraining
# ============================================================

import torch
import numpy as np
import matplotlib.pyplot as plt
import cv2

saliency_model.eval()

# ------------------------------------------------------------
# Refinement function
# ------------------------------------------------------------

def refine_saliency(
    probability,
    threshold=0.5,
    min_component_size=20,
    blur_kernel=3
):
    """
    Refines a U-Net probability map.

    Steps:
    1. Threshold probability map
    2. Remove very small isolated components
    3. Use the cleaned mask to retain the original
       continuous probabilities
    """

    prob = probability.copy()

    # Binary candidate mask
    binary = (
        prob >= threshold
    ).astype(np.uint8)

    # Connected components
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        binary,
        connectivity=8
    )

    cleaned = np.zeros_like(
        binary
    )

    for label in range(
        1,
        num_labels
    ):

        area = stats[
            label,
            cv2.CC_STAT_AREA
        ]

        if area >= min_component_size:

            cleaned[
                labels == label
            ] = 1

    # --------------------------------------------------------
    # Preserve continuous probability values
    # --------------------------------------------------------

    refined = (
        prob * cleaned
    )

    # --------------------------------------------------------
    # Mild smoothing
    # --------------------------------------------------------

    if blur_kernel > 1:

        refined = cv2.GaussianBlur(
            refined,
            (
                blur_kernel,
                blur_kernel
            ),
            0
        )

    return refined


# ============================================================
# Find representative examples
# ============================================================

# Use previously calculated results
sorted_results = sorted(
    results_kvasir,
    key=lambda x: x["dice"]
)

worst_cases = sorted_results[:3]

print("=" * 60)
print("SELECTED WORST CASES")
print("=" * 60)

for r in worst_cases:

    print(
        r["filename"],
        "| Dice:",
        round(r["dice"], 4),
        "| Type:",
        r["error_type"]
    )


# ============================================================
# Visualize worst cases
# ============================================================

for result in worst_cases:

    filename = result["filename"]

    # Find sample in dataset
    sample_idx = None

    for idx in range(
        len(kvasir_test_dataset)
    ):

        if (
            kvasir_test_data.iloc[idx]["filename"]
            == filename
        ):

            sample_idx = idx
            break

    if sample_idx is None:
        continue

    sample = kvasir_test_dataset[
        sample_idx
    ]

    image = (
        sample["low_light"]
        .unsqueeze(0)
        .to(device)
    )

    gt = sample["mask"][0].numpy()

    # --------------------------------------------------------
    # Prediction
    # --------------------------------------------------------

    with torch.no_grad():

        logits = saliency_model(
            image
        )

        probability = torch.sigmoid(
            logits
        )[0, 0].cpu().numpy()

    binary_prediction = (
        probability >= 0.5
    ).astype(np.float32)

    # --------------------------------------------------------
    # Refine
    # --------------------------------------------------------

    refined = refine_saliency(
        probability,
        threshold=0.5,
        min_component_size=20,
        blur_kernel=3
    )

    refined_binary = (
        refined >= 0.5
    ).astype(np.float32)

    # --------------------------------------------------------
    # Image
    # --------------------------------------------------------

    display_image = (
        sample["low_light"]
        .permute(1, 2, 0)
        .numpy()
    )

    # ========================================================
    # DISPLAY
    # ========================================================

    plt.figure(
        figsize=(15, 3)
    )

    plt.subplot(1, 5, 1)

    plt.imshow(
        display_image
    )

    plt.title(
        "Low-light"
    )

    plt.axis("off")

    plt.subplot(1, 5, 2)

    plt.imshow(
        gt,
        cmap="gray"
    )

    plt.title(
        "Ground Truth"
    )

    plt.axis("off")

    plt.subplot(1, 5, 3)

    plt.imshow(
        probability,
        cmap="gray",
        vmin=0,
        vmax=1
    )

    plt.title(
        "Original Probability"
    )

    plt.axis("off")

    plt.subplot(1, 5, 4)

    plt.imshow(
        binary_prediction,
        cmap="gray"
    )

    plt.title(
        "Original Binary"
    )

    plt.axis("off")

    plt.subplot(1, 5, 5)

    plt.imshow(
        refined,
        cmap="gray",
        vmin=0,
        vmax=1
    )

    plt.title(
        "Refined Saliency"
    )

    plt.axis("off")

    plt.suptitle(
        f"{filename} | "
        f"Original Dice: {result['dice']:.4f}"
    )

    plt.tight_layout()

    plt.show()

In [ ]:
# ============================================================
# EXPERIMENT G — REFINEMENT QUANTITATIVE COMPARISON
# ============================================================

import numpy as np
import torch

saliency_model.eval()

original_dice = []
original_iou = []

refined_dice = []
refined_iou = []

valid_count = 0

with torch.no_grad():

    for idx in range(
        len(kvasir_test_dataset)
    ):

        try:

            sample = kvasir_test_dataset[idx]

            low_light = (
                sample["low_light"]
                .unsqueeze(0)
                .to(device)
            )

            gt = sample["mask"][0].numpy()

            logits = saliency_model(
                low_light
            )

            probability = torch.sigmoid(
                logits
            )[0, 0].cpu().numpy()

        except Exception:

            continue

        # ----------------------------------------------------
        # Original prediction
        # ----------------------------------------------------

        original = (
            probability >= 0.5
        ).astype(np.uint8)

        # ----------------------------------------------------
        # Refined prediction
        # ----------------------------------------------------

        refined_probability = refine_saliency(
            probability,
            threshold=0.5,
            min_component_size=20,
            blur_kernel=3
        )

        refined = (
            refined_probability >= 0.5
        ).astype(np.uint8)

        gt_binary = (
            gt >= 0.5
        ).astype(np.uint8)

        # ----------------------------------------------------
        # Original metrics
        # ----------------------------------------------------

        tp = np.sum(
            (original == 1) &
            (gt_binary == 1)
        )

        fp = np.sum(
            (original == 1) &
            (gt_binary == 0)
        )

        fn = np.sum(
            (original == 0) &
            (gt_binary == 1)
        )

        dice = (
            2 * tp /
            (2 * tp + fp + fn + 1e-8)
        )

        iou = (
            tp /
            (tp + fp + fn + 1e-8)
        )

        original_dice.append(dice)
        original_iou.append(iou)

        # ----------------------------------------------------
        # Refined metrics
        # ----------------------------------------------------

        tp = np.sum(
            (refined == 1) &
            (gt_binary == 1)
        )

        fp = np.sum(
            (refined == 1) &
            (gt_binary == 0)
        )

        fn = np.sum(
            (refined == 0) &
            (gt_binary == 1)
        )

        dice = (
            2 * tp /
            (2 * tp + fp + fn + 1e-8)
        )

        iou = (
            tp /
            (tp + fp + fn + 1e-8)
        )

        refined_dice.append(dice)
        refined_iou.append(iou)

        valid_count += 1


# ============================================================
# RESULTS
# ============================================================

original_mean_dice = np.mean(
    original_dice
)

original_mean_iou = np.mean(
    original_iou
)

refined_mean_dice = np.mean(
    refined_dice
)

refined_mean_iou = np.mean(
    refined_iou
)

print("=" * 60)
print("REFINEMENT COMPARISON")
print("=" * 60)

print(
    "Valid samples:",
    valid_count
)

print()

print(
    "Original Dice:",
    round(original_mean_dice, 4)
)

print(
    "Refined Dice:",
    round(refined_mean_dice, 4)
)

print()

print(
    "Original IoU:",
    round(original_mean_iou, 4)
)

print(
    "Refined IoU:",
    round(refined_mean_iou, 4)
)

print()

print(
    "Dice change:",
    round(
        refined_mean_dice -
        original_mean_dice,
        4
    )
)

print(
    "IoU change:",
    round(
        refined_mean_iou -
        original_mean_iou,
        4
    )
)

In [ ]:
# ============================================================
# STEP 1 — LOAD FINAL U-NET SALIENCY MODEL
# ============================================================

BEST_UNET_PATH = (
    "/content/drive/MyDrive/SANC/models/"
    "best_unet_saliency.pth"
)

checkpoint = torch.load(
    BEST_UNET_PATH,
    map_location=device
)

saliency_model = OriginalUNet(
    in_channels=3,
    out_channels=1
).to(device)

saliency_model.load_state_dict(
    checkpoint["model_state_dict"]
)

saliency_model.eval()

print("=" * 60)
print("FINAL SALIENCY MODEL")
print("=" * 60)

print("Checkpoint epoch:",
      checkpoint["epoch"])

print("Validation Dice:",
      checkpoint["val_dice"])

print("Validation IoU:",
      checkpoint["val_iou"])

In [ ]:
# ============================================================
# STEP 2 — CONTINUOUS SALIENCY
# ============================================================

with torch.no_grad():

    batch = next(iter(kvasir_test_loader))

    low_light = batch["low_light"].to(device)

    logits = saliency_model(
        low_light
    )

    saliency = torch.sigmoid(
        logits
    )

print("=" * 60)
print("CONTINUOUS SALIENCY")
print("=" * 60)

print("Input shape:",
      low_light.shape)

print("Saliency shape:",
      saliency.shape)

print("Minimum:",
      saliency.min().item())

print("Maximum:",
      saliency.max().item())

print("Mean:",
      saliency.mean().item())

In [ ]:
# ============================================================
# STEP 3 — SALIENCY → NOISE STRENGTH
# ============================================================

noise_strength = 1.0 - saliency

print("=" * 60)
print("SANC NOISE STRENGTH")
print("=" * 60)

print("Saliency min:",
      saliency.min().item())

print("Saliency max:",
      saliency.max().item())

print("Noise min:",
      noise_strength.min().item())

print("Noise max:",
      noise_strength.max().item())

In [ ]:
# ============================================================
# STEP 4 — CONTROLLED SPATIAL NOISE
# ============================================================

BASE_NOISE = 0.20

spatial_noise_strength = (
    BASE_NOISE *
    (1.0 - saliency)
)

print("=" * 60)
print("CONTROLLED SANC NOISE")
print("=" * 60)

print("Base noise:",
      BASE_NOISE)

print(
    "Noise strength range:",
    spatial_noise_strength.min().item(),
    "to",
    spatial_noise_strength.max().item()
)

In [ ]:
# ============================================================
# STEP 5 — VISUALIZE SANC MAPS
# ============================================================

import matplotlib.pyplot as plt

idx = 0

image = (
    low_light[idx]
    .detach()
    .cpu()
    .permute(1, 2, 0)
    .numpy()
)

sal = (
    saliency[idx, 0]
    .detach()
    .cpu()
    .numpy()
)

noise = (
    spatial_noise_strength[idx, 0]
    .detach()
    .cpu()
    .numpy()
)

plt.figure(
    figsize=(12, 4)
)

plt.subplot(1, 3, 1)

plt.imshow(image)

plt.title(
    "Low-light"
)

plt.axis("off")


plt.subplot(1, 3, 2)

plt.imshow(
    sal,
    cmap="gray",
    vmin=0,
    vmax=1
)

plt.title(
    "Continuous Saliency"
)

plt.axis("off")


plt.subplot(1, 3, 3)

plt.imshow(
    noise,
    cmap="gray",
    vmin=0,
    vmax=BASE_NOISE
)

plt.title(
    "Spatial Noise Strength"
)

plt.axis("off")

plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# STEP 6 — SANC SPATIAL NOISE MAP
# ============================================================

# Saliency is in [0, 1]
# High saliency  -> low noise
# Low saliency   -> high noise

BASE_NOISE = 0.20

sanc_noise_map = (
    BASE_NOISE *
    (1.0 - saliency)
)

print("=" * 60)
print("SANC SPATIAL NOISE MAP")
print("=" * 60)

print("Shape:",
      sanc_noise_map.shape)

print("Minimum:",
      sanc_noise_map.min().item())

print("Maximum:",
      sanc_noise_map.max().item())

print("Mean:",
      sanc_noise_map.mean().item())

In [ ]:
# ============================================================
# STEP 6B — VERIFY SANC NOISE MAP
# ============================================================

idx = 0

sal = (
    saliency[idx, 0]
    .detach()
    .cpu()
    .numpy()
)

noise_map = (
    sanc_noise_map[idx, 0]
    .detach()
    .cpu()
    .numpy()
)

plt.figure(
    figsize=(10, 4)
)

plt.subplot(1, 2, 1)

plt.imshow(
    sal,
    cmap="gray",
    vmin=0,
    vmax=1
)

plt.title(
    "Continuous Saliency"
)

plt.axis("off")


plt.subplot(1, 2, 2)

plt.imshow(
    noise_map,
    cmap="gray",
    vmin=0,
    vmax=BASE_NOISE
)

plt.title(
    "SANC Spatial Noise Map"
)

plt.axis("off")

plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# STEP 7 — APPLY SANC SPATIAL NOISE
# ============================================================

# Gaussian noise
random_noise = torch.randn_like(low_light)

# Expand single-channel noise map to RGB
noise_map_rgb = sanc_noise_map.repeat(
    1, 3, 1, 1
)

# Apply spatially controlled noise
sanc_noisy_input = (
    low_light +
    noise_map_rgb * random_noise
)

# Keep valid image range
sanc_noisy_input = torch.clamp(
    sanc_noisy_input,
    0.0,
    1.0
)

print("=" * 60)
print("SANC NOISY INPUT")
print("=" * 60)

print("Original:",
      low_light.shape)

print("Noise map:",
      noise_map_rgb.shape)

print("SANC input:",
      sanc_noisy_input.shape)

print(
    "Range:",
    sanc_noisy_input.min().item(),
    "to",
    sanc_noisy_input.max().item()
)

In [ ]:
# ============================================================
# STEP 7B — VISUALIZE
# ============================================================

idx = 0

original_img = (
    low_light[idx]
    .detach()
    .cpu()
    .permute(1, 2, 0)
    .numpy()
)

sal_img = (
    saliency[idx, 0]
    .detach()
    .cpu()
    .numpy()
)

noisy_img = (
    sanc_noisy_input[idx]
    .detach()
    .cpu()
    .permute(1, 2, 0)
    .numpy()
)

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.imshow(original_img)
plt.title("Low-light")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(
    sal_img,
    cmap="gray",
    vmin=0,
    vmax=1
)
plt.title("Continuous Saliency")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(noisy_img)
plt.title("SANC Spatially Noisy Input")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# STEP 7C — TEST DIFFERENT SANC NOISE STRENGTHS
# ============================================================

idx = 0

image = low_light[idx:idx+1]

sal = saliency[idx:idx+1]

noise_map_base = 1.0 - sal

noise_levels = [
    0.05,
    0.10,
    0.15,
]

plt.figure(
    figsize=(16, 4)
)

# ------------------------------------------------------------
# Original
# ------------------------------------------------------------

original_img = (
    image[0]
    .detach()
    .cpu()
    .permute(1, 2, 0)
    .numpy()
)

plt.subplot(
    1,
    4,
    1
)

plt.imshow(
    original_img
)

plt.title(
    "Low-light"
)

plt.axis("off")


# ------------------------------------------------------------
# Different noise levels
# ------------------------------------------------------------

for plot_idx, base_noise in enumerate(
    noise_levels,
    start=2
):

    noise_map = (
        base_noise *
        noise_map_base
    )

    noise_map_rgb = noise_map.repeat(
        1,
        3,
        1,
        1
    )

    random_noise = torch.randn_like(
        image
    )

    noisy = (
        image +
        noise_map_rgb *
        random_noise
    )

    noisy = torch.clamp(
        noisy,
        0.0,
        1.0
    )

    noisy_img = (
        noisy[0]
        .detach()
        .cpu()
        .permute(1, 2, 0)
        .numpy()
    )

    plt.subplot(
        1,
        4,
        plot_idx
    )

    plt.imshow(
        noisy_img
    )

    plt.title(
        f"BASE_NOISE = {base_noise}"
    )

    plt.axis("off")


plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# STEP 8 — SMOOTH SALIENCY-CONTROLLED NOISE
# ============================================================

import torch.nn.functional as F
import matplotlib.pyplot as plt

idx = 0

image = low_light[idx:idx+1]

sal = saliency[idx:idx+1]

# ------------------------------------------------------------
# Saliency -> noise strength
# ------------------------------------------------------------

noise_strength = 1.0 - sal

noise_levels = [
    0.02,
    0.03,
    0.05
]

plt.figure(figsize=(16, 4))

# ------------------------------------------------------------
# Original
# ------------------------------------------------------------

original_img = (
    image[0]
    .detach()
    .cpu()
    .permute(1, 2, 0)
    .numpy()
)

plt.subplot(1, 4, 1)

plt.imshow(original_img)

plt.title("Low-light")

plt.axis("off")


# ------------------------------------------------------------
# Smooth noise experiments
# ------------------------------------------------------------

for plot_idx, base_noise in enumerate(
    noise_levels,
    start=2
):

    # --------------------------------------------------------
    # Generate ONE noise channel
    # and share it across RGB
    # --------------------------------------------------------

    raw_noise = torch.randn(
        1,
        1,
        image.shape[2],
        image.shape[3],
        device=device
    )

    # Smooth the noise spatially
    smooth_noise = F.avg_pool2d(
        raw_noise,
        kernel_size=9,
        stride=1,
        padding=4
    )

    # Normalize noise approximately
    smooth_noise = (
        smooth_noise /
        (smooth_noise.std() + 1e-8)
    )

    # --------------------------------------------------------
    # Spatially control noise
    # --------------------------------------------------------

    spatial_noise = (
        base_noise *
        noise_strength *
        smooth_noise
    )

    # Share same spatial noise across RGB
    spatial_noise_rgb = spatial_noise.repeat(
        1,
        3,
        1,
        1
    )

    # --------------------------------------------------------
    # Apply noise
    # --------------------------------------------------------

    noisy = (
        image +
        spatial_noise_rgb
    )

    noisy = torch.clamp(
        noisy,
        0.0,
        1.0
    )

    noisy_img = (
        noisy[0]
        .detach()
        .cpu()
        .permute(1, 2, 0)
        .numpy()
    )

    plt.subplot(
        1,
        4,
        plot_idx
    )

    plt.imshow(noisy_img)

    plt.title(
        f"Smooth Noise = {base_noise}"
    )

    plt.axis("off")


plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# SANC RESTORATION — STEP 1
# Imports + Device + Configuration
# ============================================================

import os
import math
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 60)
print("ENVIRONMENT")
print("=" * 60)

print(
    "PyTorch version:",
    torch.__version__
)

print(
    "Device:",
    device
)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "CUDA version:",
        torch.version.cuda
    )

else:

    print(
        "WARNING: CUDA is not available."
    )

In [ ]:
# ============================================================
# STEP 2A — RESTORE ORIGINAL U-NET
# Exact output-layer naming required by checkpoint
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F


class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class OriginalUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super().__init__()

        # Encoder
        self.enc1 = DoubleConv(in_channels, 64)
        self.enc2 = DoubleConv(64, 128)
        self.enc3 = DoubleConv(128, 256)
        self.enc4 = DoubleConv(256, 512)

        self.pool = nn.MaxPool2d(
            kernel_size=2,
            stride=2
        )

        # Bottleneck
        self.bottleneck = DoubleConv(512, 1024)

        # Decoder
        self.up4 = nn.ConvTranspose2d(
            1024, 512,
            kernel_size=2,
            stride=2
        )
        self.dec4 = DoubleConv(1024, 512)

        self.up3 = nn.ConvTranspose2d(
            512, 256,
            kernel_size=2,
            stride=2
        )
        self.dec3 = DoubleConv(512, 256)

        self.up2 = nn.ConvTranspose2d(
            256, 128,
            kernel_size=2,
            stride=2
        )
        self.dec2 = DoubleConv(256, 128)

        self.up1 = nn.ConvTranspose2d(
            128, 64,
            kernel_size=2,
            stride=2
        )
        self.dec1 = DoubleConv(128, 64)

        # IMPORTANT:
        # The trained checkpoint uses "output"
        self.output = nn.Conv2d(
            64,
            out_channels,
            kernel_size=1
        )

    def forward(self, x):

        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        # Bottleneck
        b = self.bottleneck(self.pool(e4))

        # Decoder
        d4 = self.up4(b)
        d4 = torch.cat([d4, e4], dim=1)
        d4 = self.dec4(d4)

        d3 = self.up3(d4)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return torch.sigmoid(self.output(d1))


print("OriginalUNet restored.")

In [ ]:
# ============================================================
# STEP 2B — LOAD TRAINED SALIENCY CHECKPOINT
# ============================================================

BEST_UNET_PATH = "/content/drive/MyDrive/SANC/models/best_unet_saliency.pth"

checkpoint = torch.load(
    BEST_UNET_PATH,
    map_location=device
)

saliency_model = OriginalUNet(
    in_channels=3,
    out_channels=1
).to(device)

saliency_model.load_state_dict(
    checkpoint["model_state_dict"]
)

saliency_model.eval()

print("Saliency model loaded successfully.")
print("Checkpoint:", BEST_UNET_PATH)

if "epoch" in checkpoint:
    print("Epoch:", checkpoint["epoch"])

if "val_dice" in checkpoint:
    print("Validation Dice:", checkpoint["val_dice"])

if "val_iou" in checkpoint:
    print("Validation IoU:", checkpoint["val_iou"])

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
# ============================================================
# STEP 3 — DDPM DIFFUSION SCHEDULE
# ============================================================

import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# Number of diffusion steps
T = 100

# Linear beta schedule
beta_start = 1e-4
beta_end = 0.02

betas = torch.linspace(
    beta_start,
    beta_end,
    T,
    device=device
)

alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

sqrt_alpha_bars = torch.sqrt(alpha_bars)
sqrt_one_minus_alpha_bars = torch.sqrt(1.0 - alpha_bars)

# Previous cumulative alpha
alpha_bars_prev = torch.cat([
    torch.tensor([1.0], device=device),
    alpha_bars[:-1]
])

# DDPM posterior variance
posterior_variance = (
    betas * (1.0 - alpha_bars_prev)
    / (1.0 - alpha_bars)
)

print("=" * 60)
print("DIFFUSION SCHEDULE")
print("=" * 60)
print("T:", T)
print("Beta start:", beta_start)
print("Beta end:", beta_end)
print("alpha_bar[0]:", alpha_bars[0].item())
print("alpha_bar[-1]:", alpha_bars[-1].item())
print("Schedule device:", betas.device)

In [ ]:
# ============================================================
# STEP 4 — DIFFUSION HELPERS
# ============================================================

def extract(values, timesteps, x_shape):
    """
    Extract timestep-specific values and reshape them
    for broadcasting over an image tensor.
    """

    batch_size = timesteps.shape[0]

    out = values.gather(
        0,
        timesteps
    )

    return out.reshape(
        batch_size,
        *((1,) * (len(x_shape) - 1))
    )


def q_sample(x0, t, colored_noise):
    """
    Forward diffusion using SANC-colored noise.

    x_t = sqrt(alpha_bar_t) * x0
          + sqrt(1-alpha_bar_t) * colored_noise
    """

    sqrt_ab = extract(
        sqrt_alpha_bars,
        t,
        x0.shape
    )

    sqrt_one_minus_ab = extract(
        sqrt_one_minus_alpha_bars,
        t,
        x0.shape
    )

    xt = (
        sqrt_ab * x0
        + sqrt_one_minus_ab * colored_noise
    )

    return xt


print("Forward diffusion functions restored.")

In [ ]:
# ============================================================
# STEP 5A — CORRECTED SPATIAL-ADAPTIVE SANC COLORING
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F


class SANCNoiseColoring(nn.Module):
    """
    Structure-Adaptive Noise Coloring

    Creates low-, mid-, and high-frequency components from
    Gaussian noise and locally blends them according to the
    continuous saliency map.

    High saliency:
        -> stronger high-frequency component

    Low saliency:
        -> stronger low-frequency component

    Input:
        noise:
            [B, C, H, W]

        saliency:
            [B, 1, H, W], values in [0, 1]

    Output:
        colored_noise:
            [B, C, H, W]
    """

    def __init__(
        self,
        saliency_power=1.0
    ):
        super().__init__()

        self.saliency_power = saliency_power

    def _frequency_bands(self, H, W, device):

        # ----------------------------------------------------
        # Frequency coordinates
        # ----------------------------------------------------

        fy = torch.fft.fftfreq(
            H,
            device=device
        )

        fx = torch.fft.fftfreq(
            W,
            device=device
        )

        yy, xx = torch.meshgrid(
            fy,
            fx,
            indexing="ij"
        )

        radius = torch.sqrt(
            xx ** 2 + yy ** 2
        )

        # Normalize radial frequency to approximately [0, 1]
        radius = radius / radius.max()

        # ----------------------------------------------------
        # Smooth frequency-band centers
        # ----------------------------------------------------

        low_center = 0.15
        mid_center = 0.45
        high_center = 0.80

        bandwidth = 0.20

        # Smooth Gaussian frequency bands
        low_band = torch.exp(
            -0.5 * (
                (radius - low_center)
                / bandwidth
            ) ** 2
        )

        mid_band = torch.exp(
            -0.5 * (
                (radius - mid_center)
                / bandwidth
            ) ** 2
        )

        high_band = torch.exp(
            -0.5 * (
                (radius - high_center)
                / bandwidth
            ) ** 2
        )

        # ----------------------------------------------------
        # Normalize the bands so they form a partition
        # ----------------------------------------------------

        total = (
            low_band
            + mid_band
            + high_band
            + 1e-8
        )

        low_band = low_band / total
        mid_band = mid_band / total
        high_band = high_band / total

        return (
            low_band,
            mid_band,
            high_band
        )

    def forward(self, noise, saliency):

        B, C, H, W = noise.shape

        # ----------------------------------------------------
        # 1. Match saliency resolution
        # ----------------------------------------------------

        if saliency.shape[-2:] != (H, W):

            saliency = F.interpolate(
                saliency,
                size=(H, W),
                mode="bilinear",
                align_corners=False
            )

        # ----------------------------------------------------
        # 2. Keep saliency continuous in [0,1]
        # ----------------------------------------------------

        M = torch.clamp(
            saliency,
            0.0,
            1.0
        )

        M = M ** self.saliency_power

        # ----------------------------------------------------
        # 3. FFT of Gaussian noise
        # ----------------------------------------------------

        noise_fft = torch.fft.fft2(
            noise,
            dim=(-2, -1)
        )

        # ----------------------------------------------------
        # 4. Construct frequency bands
        # ----------------------------------------------------

        low_band, mid_band, high_band = (
            self._frequency_bands(
                H,
                W,
                noise.device
            )
        )

        # Add dimensions for broadcasting:
        #
        # [H,W] -> [1,1,H,W]

        low_band = low_band[None, None]
        mid_band = mid_band[None, None]
        high_band = high_band[None, None]

        # ----------------------------------------------------
        # 5. Extract frequency components
        # ----------------------------------------------------

        low_fft = (
            noise_fft
            * low_band
        )

        mid_fft = (
            noise_fft
            * mid_band
        )

        high_fft = (
            noise_fft
            * high_band
        )

        # ----------------------------------------------------
        # 6. Return each component to spatial domain
        # ----------------------------------------------------

        low_noise = torch.fft.ifft2(
            low_fft,
            dim=(-2, -1)
        ).real

        mid_noise = torch.fft.ifft2(
            mid_fft,
            dim=(-2, -1)
        ).real

        high_noise = torch.fft.ifft2(
            high_fft,
            dim=(-2, -1)
        ).real

        # ----------------------------------------------------
        # 7. Saliency-dependent local spectral weights
        #
        # M = 0:
        #   low  = 0.55
        #   mid  = 0.30
        #   high = 0.15
        #
        # M = 1:
        #   low  = 0.20
        #   mid  = 0.30
        #   high = 0.50
        #
        # Therefore increasing saliency shifts the local
        # noise spectrum toward higher frequencies.
        # ----------------------------------------------------

        low_weight = (
            0.55
            - 0.35 * M
        )

        mid_weight = (
            0.30
            * torch.ones_like(M)
        )

        high_weight = (
            0.15
            + 0.35 * M
        )

        # ----------------------------------------------------
        # 8. Locally blend the frequency components
        # ----------------------------------------------------

        colored_noise = (
            low_weight * low_noise
            +
            mid_weight * mid_noise
            +
            high_weight * high_noise
        )

        # ----------------------------------------------------
        # 9. Normalize to approximately N(0,1)
        #
        # This keeps the diffusion noise scale compatible
        # with the DDPM schedule.
        # ----------------------------------------------------

        mean = colored_noise.mean(
            dim=(-2, -1),
            keepdim=True
        )

        std = colored_noise.std(
            dim=(-2, -1),
            keepdim=True
        )

        colored_noise = (
            colored_noise - mean
        ) / (std + 1e-6)

        return colored_noise


# ------------------------------------------------------------
# Create SANC module
# ------------------------------------------------------------

sanc = SANCNoiseColoring(
    saliency_power=1.0
).to(device)

print("Corrected SANCNoiseColoring defined successfully.")

In [ ]:
# ============================================================
# STEP 5B — CORRECTED SANC NUMERICAL TEST
# ============================================================

B = 2
C = 3
H = 128
W = 128

# Gaussian noise
test_noise = torch.randn(
    B, C, H, W,
    device=device
)

# Continuous artificial saliency
test_saliency = torch.zeros(
    B, 1, H, W,
    device=device
)

test_saliency[:, :, 32:96, 32:96] = 1.0

# Smooth saliency boundaries
test_saliency = F.avg_pool2d(
    test_saliency,
    kernel_size=15,
    stride=1,
    padding=7
)

# Apply corrected SANC
with torch.no_grad():

    colored_noise = sanc(
        test_noise,
        test_saliency
    )


print("=" * 60)
print("CORRECTED SANC TEST")
print("=" * 60)

print("Input noise shape:   ", test_noise.shape)
print("Saliency shape:      ", test_saliency.shape)
print("Colored noise shape: ", colored_noise.shape)

print()
print("Gaussian mean:",
      test_noise.mean().item())

print("Gaussian std:",
      test_noise.std().item())

print("SANC mean:",
      colored_noise.mean().item())

print("SANC std:",
      colored_noise.std().item())

print()
print("NaN present:",
      torch.isnan(colored_noise).any().item())

print("Inf present:",
      torch.isinf(colored_noise).any().item())


# ------------------------------------------------------------
# Verify local weights explicitly
# ------------------------------------------------------------

M_low = torch.tensor(0.0, device=device)
M_high = torch.tensor(1.0, device=device)

low_low = 0.55 - 0.35 * M_low
mid_low = 0.30
high_low = 0.15 + 0.35 * M_low

low_high = 0.55 - 0.35 * M_high
mid_high = 0.30
high_high = 0.15 + 0.35 * M_high

print()
print("Low-saliency weights:")
print("  Low :", low_low.item())
print("  Mid :", mid_low)
print("  High:", high_low.item())

print()
print("High-saliency weights:")
print("  Low :", low_high.item())
print("  Mid :", mid_high)
print("  High:", high_high.item())

In [ ]:
# ============================================================
# STEP 5C — CORRECTED SANC VISUALIZATION
# ============================================================

import matplotlib.pyplot as plt
import numpy as np


# ------------------------------------------------------------
# Select first sample / first channel
# ------------------------------------------------------------

gaussian_img = (
    test_noise[0, 0]
    .detach()
    .cpu()
    .numpy()
)

sanc_img = (
    colored_noise[0, 0]
    .detach()
    .cpu()
    .numpy()
)

saliency_img = (
    test_saliency[0, 0]
    .detach()
    .cpu()
    .numpy()
)


# ------------------------------------------------------------
# Fourier spectra
# ------------------------------------------------------------

gaussian_fft = np.fft.fftshift(
    np.fft.fft2(gaussian_img)
)

sanc_fft = np.fft.fftshift(
    np.fft.fft2(sanc_img)
)

gaussian_spectrum = np.log1p(
    np.abs(gaussian_fft)
)

sanc_spectrum = np.log1p(
    np.abs(sanc_fft)
)


# ------------------------------------------------------------
# Visualization
# ------------------------------------------------------------

plt.figure(figsize=(16, 4))

plt.subplot(1, 4, 1)

plt.imshow(
    saliency_img,
    cmap="gray"
)

plt.title("Saliency Map")
plt.axis("off")


plt.subplot(1, 4, 2)

plt.imshow(
    gaussian_img,
    cmap="gray"
)

plt.title("Gaussian Noise")
plt.axis("off")


plt.subplot(1, 4, 3)

plt.imshow(
    gaussian_spectrum,
    cmap="gray"
)

plt.title("Gaussian Spectrum")
plt.axis("off")


plt.subplot(1, 4, 4)

plt.imshow(
    sanc_spectrum,
    cmap="gray"
)

plt.title("Corrected SANC Spectrum")
plt.axis("off")


plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# SANC spatial noise
# ------------------------------------------------------------

plt.figure(figsize=(6, 6))

plt.imshow(
    sanc_img,
    cmap="gray"
)

plt.title("Corrected SANC-Colored Noise")
plt.axis("off")

plt.show()

In [ ]:
# ============================================================
# STEP 5D — LOCAL SALIENCY-DEPENDENT SPECTRAL TEST
# ============================================================

# Generate separate noise components
noise = torch.randn(
    1, 3, 128, 128,
    device=device
)

# Two saliency maps:
# one completely low-saliency
# one completely high-saliency

M_low = torch.zeros(
    1, 1, 128, 128,
    device=device
)

M_high = torch.ones(
    1, 1, 128, 128,
    device=device
)

with torch.no_grad():

    noise_low = sanc(
        noise,
        M_low
    )

    noise_high = sanc(
        noise,
        M_high
    )


# ------------------------------------------------------------
# Radial spectrum helper
# ------------------------------------------------------------

def radial_energy(image):

    image = image.detach().cpu().numpy()

    fft = np.fft.fftshift(
        np.fft.fft2(image)
    )

    magnitude = np.abs(fft)

    H, W = image.shape

    y, x = np.indices(
        (H, W)
    )

    cy = H // 2
    cx = W // 2

    radius = np.sqrt(
        (x - cx) ** 2
        +
        (y - cy) ** 2
    )

    max_radius = min(H, W) // 2

    low_mask = radius < max_radius * 0.33
    high_mask = radius > max_radius * 0.66

    low_energy = magnitude[low_mask].mean()
    high_energy = magnitude[high_mask].mean()

    return low_energy, high_energy


low_low, low_high = radial_energy(
    noise_low[0, 0]
)

high_low, high_high = radial_energy(
    noise_high[0, 0]
)


print("=" * 60)
print("LOCAL SALIENCY-DEPENDENT SPECTRAL TEST")
print("=" * 60)

print()
print("Low-saliency noise:")
print("  Low-frequency energy :",
      low_low)
print("  High-frequency energy:",
      low_high)

print()
print("High-saliency noise:")
print("  Low-frequency energy :",
      high_low)
print("  High-frequency energy:",
      high_high)

print()
print("High/Low ratio:")

print(
    "  Low saliency :",
    low_high / (low_low + 1e-8)
)

print(
    "  High saliency:",
    high_high / (high_low + 1e-8)
)

In [ ]:
# ============================================================
# STEP 5E — PREPARE REAL MEDICAL IMAGE TEST
# ============================================================

import os
import glob
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

print("=" * 60)
print("REAL MEDICAL IMAGE SANC TEST")
print("=" * 60)

# ------------------------------------------------------------
# Make sure SANC exists
# ------------------------------------------------------------

assert 'sanc' in globals(), (
    "SANC module not found. Run corrected Step 5A first."
)

# ------------------------------------------------------------
# Make sure saliency model exists
# ------------------------------------------------------------

assert 'saliency_model' in globals(), (
    "Saliency model not found. Run Step 2B first."
)

saliency_model.eval()

print("Saliency model: available")
print("SANC module: available")
print("Device:", device)

# ------------------------------------------------------------
# Check whether train_loader survived the runtime reset
# ------------------------------------------------------------

if 'train_loader' in globals():

    print("train_loader: available")

else:

    print("train_loader: NOT available")

    print()
    print("The runtime reset removed the previous DataLoader.")
    print("We will need to restore the dataset loader before")
    print("running the real-image test.")

In [ ]:
# ============================================================
# STEP 5E-0 — RESTORE SANC DATASET FROM GOOGLE DRIVE
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import glob
import pandas as pd

SANC_ROOT = "/content/drive/MyDrive/SANC"

print("=" * 70)
print("SANC DRIVE DATASET RESTORATION")
print("=" * 70)

print("SANC root:")
print(SANC_ROOT)

print()
print("Exists:", os.path.exists(SANC_ROOT))

if not os.path.exists(SANC_ROOT):
    raise FileNotFoundError(
        "SANC folder was not found in Google Drive."
    )


# ------------------------------------------------------------
# Find CSV metadata files
# ------------------------------------------------------------

csv_files = []

for root, dirs, files in os.walk(SANC_ROOT):

    for file in files:

        if file.lower().endswith(".csv"):

            csv_files.append(
                os.path.join(root, file)
            )


print()
print("CSV files found:", len(csv_files))

for path in csv_files:

    print(" -", path)

In [ ]:
# ============================================================
# STEP 5E-2 — ROBUST DATASET PATH RESOLUTION
# ============================================================

def resolve_ham_path(relative_path):

    """
    HAM metadata stores paths such as:
        images/ISIC_xxxxx.png
        masks/ISIC_xxxxx.png
        low_light/ISIC_xxxxx.png
    """

    return os.path.join(
        HAM_ROOT,
        relative_path
    )


def resolve_origa_path(relative_path):

    """
    ORIGA common metadata stores:
        images/334.png
        masks/334.png

    Low-light files are stored separately in the
    persistent integration directory.
    """

    return os.path.join(
        ORIGA_ROOT,
        relative_path
    )


def resolve_kvasir_path(path):

    """
    Kvasir metadata should use the persistent Drive
    directories regardless of stale /content paths.
    """

    filename = os.path.basename(path)

    # Determine which folder the path refers to
    path_lower = str(path).lower()

    if "low_light" in path_lower:
        return os.path.join(
            KVASIR_ROOT,
            "low_light",
            filename
        )

    elif "mask" in path_lower:
        return os.path.join(
            KVASIR_ROOT,
            "masks",
            filename
        )

    else:
        return os.path.join(
            KVASIR_ROOT,
            "original",
            filename
        )


# ------------------------------------------------------------
# Load metadata
# ------------------------------------------------------------

ham_df = pd.read_csv(HAM_META)
origa_df = pd.read_csv(ORIGA_META)
kvasir_df = pd.read_csv(KVASIR_VAL)


print("=" * 70)
print("METADATA LOADED")
print("=" * 70)

print("HAM:", ham_df.shape)
print("ORIGA:", origa_df.shape)
print("Kvasir validation:", kvasir_df.shape)

In [ ]:
def resolve_ham_path(relative_path):
    """
    Resolve HAM10000 metadata paths against the actual
    persistent Drive directory structure.
    """
    relative_path = str(relative_path)

    # Extract just the filename
    filename = os.path.basename(relative_path)

    if "low_light" in relative_path.lower():
        return os.path.join(
            HAM_ROOT,
            "low_light",
            "low_light",
            filename
        )

    elif "mask" in relative_path.lower():
        return os.path.join(
            HAM_ROOT,
            "masks",
            "masks",
            filename
        )

    else:
        return os.path.join(
            HAM_ROOT,
            "images",
            "images",
            filename
        )

In [ ]:
ham_meta = pd.read_csv(
    "/content/drive/MyDrive/SANC/datasets/ham/HAM preprocessing/metadata.csv"
)

ham_test = ham_meta[
    ham_meta["split"].astype(str).str.lower() == "test"
].copy()

ham_rows = []

for _, row in ham_test.iterrows():

    filename = row["image_id"] + ".png"

    original_path = resolve_ham_path(row["image_path"])
    low_light_path = resolve_ham_path(row["low_light_path"])
    mask_path = resolve_ham_path(row["mask_path"])

    ham_rows.append({
        "filename": filename,
        "original_path": original_path,
        "low_light_path": low_light_path,
        "mask_path": mask_path,
        "dataset": "HAM10000",
        "split": "test"
    })

ham_df = pd.DataFrame(ham_rows)

print("HAM test samples:", len(ham_df))
print(ham_df.head(3).to_string(index=False))

In [ ]:
print("=" * 70)
print("VERIFYING HAM10000 PATHS")
print("=" * 70)

for column in ["original_path", "low_light_path", "mask_path"]:
    exists_count = ham_df[column].apply(os.path.exists).sum()
    total_count = len(ham_df)

    print(
        f"{column:20s}: "
        f"{exists_count}/{total_count}"
    )

In [ ]:
common_df = pd.concat(
    [
        ham_df,
        origa_df,
        kvasir_df
    ],
    ignore_index=True
)

print("=" * 70)
print("FINAL REAL-DATA DATASET")
print("=" * 70)

print("\nTotal samples:", len(common_df))

print("\nSamples by dataset:")
print(common_df["dataset"].value_counts())

print("\nPath verification:")

for dataset_name in common_df["dataset"].unique():

    subset = common_df[
        common_df["dataset"] == dataset_name
    ]

    print(f"\n{dataset_name}")

    for column in [
        "original_path",
        "low_light_path",
        "mask_path"
    ]:
        exists_count = subset[column].apply(os.path.exists).sum()

        print(
            f"  {column:20s}: "
            f"{exists_count}/{len(subset)}"
        )

In [ ]:
print("=" * 70)
print("ORIGA DATAFRAME CHECK")
print("=" * 70)

print("Shape:", origa_df.shape)
print("\nColumns:")# ============================================================
# BUILD ORIGA TEST DATAFRAME
# ============================================================

import os
import pandas as pd

ORIGA_ROOT = "/content/drive/MyDrive/SANC/datasets/ORIGA_processed/integration"

ORIGA_META_PATH = os.path.join(
    ORIGA_ROOT,
    "ORIGA_common_metadata.csv"
)

origa_meta = pd.read_csv(ORIGA_META_PATH)

print("Raw ORIGA metadata:", origa_meta.shape)
print("Columns:", origa_meta.columns.tolist())

# Keep TEST split only
origa_test = origa_meta[
    origa_meta["split"].astype(str).str.lower() == "test"
].copy()

print("ORIGA test samples:", len(origa_test))

origa_rows = []

for _, row in origa_test.iterrows():

    filename = os.path.basename(str(row["filename"]))

    origa_rows.append({
        "filename": filename,

        "original_path": os.path.join(
            ORIGA_ROOT,
            "images",
            filename
        ),

        "low_light_path": os.path.join(
            ORIGA_ROOT,
            "low_light",
            filename
        ),

        "mask_path": os.path.join(
            ORIGA_ROOT,
            "masks",
            filename
        ),

        "dataset": "ORIGA",
        "split": "test"
    })

origa_df = pd.DataFrame(origa_rows)

print("\nORIGA dataframe:")
print("Shape:", origa_df.shape)
print(origa_df.head(3).to_string(index=False))
print(origa_df.columns.tolist())

print("\nDataset/split counts:")
print(origa_df["split"].value_counts(dropna=False))

print("\nMissing values:")
print(origa_df[
    ["filename", "original_path", "low_light_path", "mask_path"]
].isna().sum())

print("\nFirst 5 rows:")
print(origa_df.head().to_string(index=False))

In [ ]:
# ============================================================
# STEP 5F — REAL MEDICAL IMAGE SANC TEST
# ============================================================

from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms
import torch
import numpy as np


class SANCMedicalDataset(Dataset):

    def __init__(self, dataframe, image_size=128):

        self.df = dataframe.reset_index(drop=True)

        self.image_transform = transforms.Compose([
            transforms.Resize(
                (image_size, image_size),
                interpolation=transforms.InterpolationMode.BILINEAR
            ),
            transforms.ToTensor()
        ])

        self.mask_transform = transforms.Compose([
            transforms.Resize(
                (image_size, image_size),
                interpolation=transforms.InterpolationMode.NEAREST
            ),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        original = Image.open(
            row["original_path"]
        ).convert("RGB")

        low_light = Image.open(
            row["low_light_path"]
        ).convert("RGB")

        mask = Image.open(
            row["mask_path"]
        ).convert("L")

        original = self.image_transform(original)
        low_light = self.image_transform(low_light)

        mask = self.mask_transform(mask)

        # Robust binary-mask normalization
        mask = (mask > 0.5).float()

        return {
            "low_light": low_light,
            "original": original,
            "mask": mask,
            "dataset": row["dataset"],
            "filename": row["filename"]
        }


real_dataset = SANCMedicalDataset(
    common_df,
    image_size=128
)

real_loader = DataLoader(
    real_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Dataset size:", len(real_dataset))
print("Number of batches:", len(real_loader))

In [ ]:
# ============================================================
# REAL IMAGE → PREDICTED SALIENCY → SANC
# ============================================================

saliency_model.eval()
sanc.eval()

real_low_light = low_light[0:1]
real_original = original[0:1]
real_mask = mask[0:1]

with torch.no_grad():

    # Predicted saliency from the REAL low-light image
    real_saliency = saliency_model(
        real_low_light
    )

    # Gaussian noise
    real_gaussian_noise = torch.randn_like(
        real_low_light
    )

    # SANC-colored noise
    real_sanc_noise = sanc(
        real_gaussian_noise,
        real_saliency
    )

print("=" * 70)
print("REAL SANC TEST")
print("=" * 70)

print("\nImage:")
print("  low-light :", real_low_light.shape)
print("  original  :", real_original.shape)

print("\nSaliency:")
print("  shape :", real_saliency.shape)
print("  min   :", real_saliency.min().item())
print("  max   :", real_saliency.max().item())
print("  mean  :", real_saliency.mean().item())

print("\nGaussian noise:")
print("  mean :", real_gaussian_noise.mean().item())
print("  std  :", real_gaussian_noise.std().item())

print("\nSANC noise:")
print("  mean :", real_sanc_noise.mean().item())
print("  std  :", real_sanc_noise.std().item())
print("  NaN  :", torch.isnan(real_sanc_noise).any().item())
print("  Inf  :", torch.isinf(real_sanc_noise).any().item())

In [ ]:
# ============================================================
# STEP 5F-3 — LOCAL SANC FREQUENCY ANALYSIS
# ============================================================

M = real_saliency[0, 0]
N = real_sanc_noise[0]

# RGB → grayscale noise for analysis
N_gray = N.mean(dim=0)

high_region = M >= 0.70
low_region = M <= 0.30

print("=" * 70)
print("LOCAL SANC FREQUENCY ANALYSIS")
print("=" * 70)

print("High-saliency pixels:", high_region.sum().item())
print("Low-saliency pixels :", low_region.sum().item())


def local_high_frequency_energy(noise, region):

    dx = noise[:, 1:] - noise[:, :-1]
    dy = noise[1:, :] - noise[:-1, :]

    rx = region[:, 1:] & region[:, :-1]
    ry = region[1:, :] & region[:-1, :]

    energies = []

    if rx.any():
        energies.append((dx[rx] ** 2).mean())

    if ry.any():
        energies.append((dy[ry] ** 2).mean())

    if len(energies) == 0:
        return float("nan")

    return torch.stack(energies).mean().item()


high_energy = local_high_frequency_energy(
    N_gray,
    high_region
)

low_energy = local_high_frequency_energy(
    N_gray,
    low_region
)

print("\nLocal high-frequency energy:")
print("  High-saliency:", high_energy)
print("  Low-saliency :", low_energy)

if np.isfinite(high_energy) and np.isfinite(low_energy):

    ratio = high_energy / (low_energy + 1e-8)

    print("\nHigh / Low ratio:", ratio)

    if ratio > 1:
        print("PASS: high-saliency regions contain stronger high-frequency variation.")
    else:
        print("WARNING: high-saliency regions did not show stronger high-frequency variation.")

In [ ]:
# ============================================================
# STEP 6A — SANC-COLORED FORWARD DIFFUSION
# ============================================================

import torch
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Select one real medical sample
# ------------------------------------------------------------

real_low_light = low_light[0:1]
real_original = original[0:1]

# ------------------------------------------------------------
# Predict saliency from LOW-LIGHT image
# ------------------------------------------------------------

saliency_model.eval()
sanc.eval()

with torch.no_grad():

    real_saliency = saliency_model(
        real_low_light
    )

    gaussian_noise = torch.randn_like(
        real_original
    )

    # SANC uses predicted saliency, NOT GT mask
    colored_noise = sanc(
        gaussian_noise,
        real_saliency
    )


# ------------------------------------------------------------
# Forward diffusion
# ------------------------------------------------------------

@torch.no_grad()
def sanc_forward_diffusion(x0, t, colored_noise):

    sqrt_ab = extract(
        sqrt_alpha_bars,
        t,
        x0.shape
    )

    sqrt_one_minus_ab = extract(
        sqrt_one_minus_alpha_bars,
        t,
        x0.shape
    )

    xt = (
        sqrt_ab * x0
        +
        sqrt_one_minus_ab * colored_noise
    )

    return xt


# ------------------------------------------------------------
# Test several timesteps
# ------------------------------------------------------------

timesteps_to_test = [
    0,
    10,
    25,
    50,
    75,
    99
]

diffusion_results = {}

for timestep in timesteps_to_test:

    t = torch.tensor(
        [timestep],
        device=device,
        dtype=torch.long
    )

    xt = sanc_forward_diffusion(
        real_original,
        t,
        colored_noise
    )

    diffusion_results[timestep] = xt

    print(
        f"t={timestep:2d} | "
        f"min={xt.min().item():.4f} | "
        f"max={xt.max().item():.4f} | "
        f"mean={xt.mean().item():.4f} | "
        f"std={xt.std().item():.4f}"
    )

In [ ]:
# ============================================================
# COMPLETE SANC DIFFUSION VISUALIZATION
# ============================================================

timesteps_to_visualize = [0, 10, 25, 50, 75, 99]

fig, axes = plt.subplots(
    2,
    6,
    figsize=(22, 8)
)

# ------------------------------------------------------------
# Top row
# ------------------------------------------------------------

axes[0, 0].imshow(
    tensor_to_image(real_original)
)
axes[0, 0].set_title("Original $x_0$")
axes[0, 0].axis("off")


axes[0, 1].imshow(
    tensor_to_image(real_low_light)
)
axes[0, 1].set_title("Low-light")
axes[0, 1].axis("off")


axes[0, 2].imshow(
    real_saliency[0, 0].detach().cpu(),
    cmap="gray",
    vmin=0,
    vmax=1
)
axes[0, 2].set_title("Predicted Saliency")
axes[0, 2].axis("off")


axes[0, 3].imshow(noise_display)
axes[0, 3].set_title("SANC Noise")
axes[0, 3].axis("off")


# Leave remaining two panels empty
axes[0, 4].axis("off")
axes[0, 5].axis("off")


# ------------------------------------------------------------
# Bottom row — complete diffusion trajectory
# ------------------------------------------------------------

for ax, timestep in zip(
    axes[1],
    timesteps_to_visualize
):

    ax.imshow(
        tensor_to_image(
            diffusion_results[timestep]
        )
    )

    ax.set_title(f"t = {timestep}")
    ax.axis("off")


plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# REVERSE DIFFUSION — TIME EMBEDDING
# ============================================================
# ============================================================
# REVERSE DIFFUSION — IMPORTS
# ============================================================

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

class TimeEmbedding(nn.Module):

    def __init__(self, embedding_dim):
        super().__init__()

        self.embedding_dim = embedding_dim

        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim, embedding_dim * 2),
            nn.SiLU(),
            nn.Linear(embedding_dim * 2, embedding_dim)
        )

    def forward(self, t):

        half_dim = self.embedding_dim // 2

        exponent = (
            -np.log(10000)
            * torch.arange(
                half_dim,
                device=t.device
            )
            / (half_dim - 1)
        )

        embedding = t.float().unsqueeze(1) * torch.exp(exponent)

        embedding = torch.cat(
            [
                torch.sin(embedding),
                torch.cos(embedding)
            ],
            dim=1
        )

        return self.mlp(embedding)

In [ ]:
# ============================================================
# TIME-CONDITIONED RESIDUAL BLOCK
# ============================================================

class TimeResBlock(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        time_dim
    ):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            3,
            padding=1
        )

        self.norm1 = nn.BatchNorm2d(
            out_channels
        )

        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            3,
            padding=1
        )

        self.norm2 = nn.BatchNorm2d(
            out_channels
        )

        self.time_proj = nn.Linear(
            time_dim,
            out_channels
        )

        self.activation = nn.SiLU()

        self.skip = (
            nn.Conv2d(
                in_channels,
                out_channels,
                1
            )
            if in_channels != out_channels
            else nn.Identity()
        )

    def forward(self, x, t_embedding):

        h = self.conv1(x)
        h = self.norm1(h)

        time_term = self.time_proj(
            t_embedding
        ).unsqueeze(-1).unsqueeze(-1)

        h = self.activation(
            h + time_term
        )

        h = self.norm2(
            self.conv2(h)
        )

        h = h + self.skip(x)

        return self.activation(h)

In [ ]:
# ============================================================
# ATTENTION-BASED FEATURE FUSION
# ============================================================

class AttentionFusion(nn.Module):

    def __init__(self, channels):
        super().__init__()

        self.attention = nn.Sequential(
            nn.Conv2d(
                channels * 2,
                channels,
                1
            ),
            nn.SiLU(),

            nn.Conv2d(
                channels,
                2,
                1
            ),

            nn.Softmax(dim=1)
        )

        self.output = nn.Conv2d(
            channels,
            channels,
            3,
            padding=1
        )

    def forward(
        self,
        illumination_features,
        detail_features
    ):

        combined = torch.cat(
            [
                illumination_features,
                detail_features
            ],
            dim=1
        )

        weights = self.attention(
            combined
        )

        illumination_weight = (
            weights[:, 0:1]
        )

        detail_weight = (
            weights[:, 1:2]
        )

        fused = (
            illumination_weight
            * illumination_features
            +
            detail_weight
            * detail_features
        )

        return self.output(fused)

In [ ]:
# ============================================================
# CONDITIONAL DUAL-BRANCH REVERSE DIFFUSION U-NET
# ============================================================
#
# Input:
#   x_t       : noisy diffusion image      [B, 3, H, W]
#   low_light : low-light conditioning     [B, 3, H, W]
#   saliency  : predicted saliency map     [B, 1, H, W]
#   t         : diffusion timestep         [B]
#
# Total input channels = 3 + 3 + 1 = 7
#
# Output:
#   predicted colored noise               [B, 3, H, W]
#
# Architecture:
#   Shared encoder
#       ↓
#   Illumination branch ─┐
#                        ├→ Attention Fusion → Noise prediction
#   Detail branch ───────┘
# ============================================================


class DualBranchReverseUNet(nn.Module):

    def __init__(
        self,
        in_channels=7,
        base_channels=64,
        time_dim=128
    ):
        super().__init__()

        # ----------------------------------------------------
        # Time embedding
        # ----------------------------------------------------

        self.time_embedding = TimeEmbedding(time_dim)

        # ----------------------------------------------------
        # SHARED ENCODER
        # ----------------------------------------------------
        #
        # 7 → 64 → 128 → 256 → 512
        #

        self.enc1 = TimeResBlock(
            in_channels,
            base_channels,
            time_dim
        )

        self.enc2 = TimeResBlock(
            base_channels,
            base_channels * 2,
            time_dim
        )

        self.enc3 = TimeResBlock(
            base_channels * 2,
            base_channels * 4,
            time_dim
        )

        self.bottleneck = TimeResBlock(
            base_channels * 4,
            base_channels * 8,
            time_dim
        )

        # ----------------------------------------------------
        # UPSAMPLING
        # ----------------------------------------------------
        #
        # 512 → 256
        # 256 → 128
        # 128 → 64
        #

        self.up3 = nn.ConvTranspose2d(
            base_channels * 8,
            base_channels * 4,
            kernel_size=2,
            stride=2
        )

        self.up2 = nn.ConvTranspose2d(
            base_channels * 4,
            base_channels * 2,
            kernel_size=2,
            stride=2
        )

        self.up1 = nn.ConvTranspose2d(
            base_channels * 2,
            base_channels,
            kernel_size=2,
            stride=2
        )

        # ====================================================
        # ILLUMINATION RESTORATION BRANCH
        # ====================================================
        #
        # At decoder level 3:
        #   up3 = 256 channels
        #   skip e3 = 256 channels
        #   concat = 512 channels
        #

        self.illum_dec3 = TimeResBlock(
            base_channels * 8,
            base_channels * 4,
            time_dim
        )

        # Level 2:
        #   up2 = 128
        #   skip e2 = 128
        #   concat = 256

        self.illum_dec2 = TimeResBlock(
            base_channels * 4,
            base_channels * 2,
            time_dim
        )

        # Level 1:
        #   up1 = 64
        #   skip e1 = 64
        #   concat = 128

        self.illum_dec1 = TimeResBlock(
            base_channels * 2,
            base_channels,
            time_dim
        )

        # ====================================================
        # DETAIL PRESERVATION BRANCH
        # ====================================================

        self.detail_dec3 = TimeResBlock(
            base_channels * 8,
            base_channels * 4,
            time_dim
        )

        self.detail_dec2 = TimeResBlock(
            base_channels * 4,
            base_channels * 2,
            time_dim
        )

        self.detail_dec1 = TimeResBlock(
            base_channels * 2,
            base_channels,
            time_dim
        )

        # ----------------------------------------------------
        # ATTENTION-BASED FUSION
        # ----------------------------------------------------

        self.fusion = AttentionFusion(
            base_channels
        )

        # ----------------------------------------------------
        # FINAL NOISE PREDICTION
        # ----------------------------------------------------

        self.output = nn.Conv2d(
            base_channels,
            3,
            kernel_size=1
        )

    def forward(
        self,
        x_t,
        low_light,
        saliency,
        t
    ):

        # ----------------------------------------------------
        # Combine diffusion state + conditioning
        # ----------------------------------------------------

        x = torch.cat(
            [
                x_t,
                low_light,
                saliency
            ],
            dim=1
        )

        # x = [B, 7, H, W]

        # ----------------------------------------------------
        # Time embedding
        # ----------------------------------------------------

        t_emb = self.time_embedding(t)

        # ----------------------------------------------------
        # SHARED ENCODER
        # ----------------------------------------------------

        e1 = self.enc1(
            x,
            t_emb
        )

        # [B, 64, H, W]

        e2 = self.enc2(
            F.avg_pool2d(e1, 2),
            t_emb
        )

        # [B, 128, H/2, W/2]

        e3 = self.enc3(
            F.avg_pool2d(e2, 2),
            t_emb
        )

        # [B, 256, H/4, W/4]

        b = self.bottleneck(
            F.avg_pool2d(e3, 2),
            t_emb
        )

        # [B, 512, H/8, W/8]

        # ====================================================
        # ILLUMINATION BRANCH
        # ====================================================

        u3_i = self.up3(b)

        # [B, 256, H/4, W/4]

        u3_i = torch.cat(
            [
                u3_i,
                e3
            ],
            dim=1
        )

        # [B, 512, H/4, W/4]

        illum3 = self.illum_dec3(
            u3_i,
            t_emb
        )

        # [B, 256, H/4, W/4]

        u2_i = self.up2(illum3)

        # [B, 128, H/2, W/2]

        u2_i = torch.cat(
            [
                u2_i,
                e2
            ],
            dim=1
        )

        # [B, 256, H/2, W/2]

        illum2 = self.illum_dec2(
            u2_i,
            t_emb
        )

        # [B, 128, H/2, W/2]

        u1_i = self.up1(illum2)

        # [B, 64, H, W]

        u1_i = torch.cat(
            [
                u1_i,
                e1
            ],
            dim=1
        )

        # [B, 128, H, W]

        illum1 = self.illum_dec1(
            u1_i,
            t_emb
        )

        # [B, 64, H, W]

        # ====================================================
        # DETAIL BRANCH
        # ====================================================

        u3_d = self.up3(b)

        u3_d = torch.cat(
            [
                u3_d,
                e3
            ],
            dim=1
        )

        detail3 = self.detail_dec3(
            u3_d,
            t_emb
        )

        u2_d = self.up2(detail3)

        u2_d = torch.cat(
            [
                u2_d,
                e2
            ],
            dim=1
        )

        detail2 = self.detail_dec2(
            u2_d,
            t_emb
        )

        u1_d = self.up1(detail2)

        u1_d = torch.cat(
            [
                u1_d,
                e1
            ],
            dim=1
        )

        detail1 = self.detail_dec1(
            u1_d,
            t_emb
        )

        # ====================================================
        # ATTENTION FUSION
        # ====================================================

        fused = self.fusion(
            illum1,
            detail1
        )

        # [B, 64, H, W]

        # ====================================================
        # PREDICT DIFFUSION NOISE
        # ====================================================

        predicted_noise = self.output(
            fused
        )

        # [B, 3, H, W]

        return predicted_noise

In [ ]:
# ============================================================
# REVERSE DIFFUSION — IMPORTS + DEVICE
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print(
    "Device:",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "CPU"
)

In [ ]:
reverse_model = DualBranchReverseUNet(
    in_channels=7,
    base_channels=64,
    time_dim=128
).to(device)

print(
    "Parameters:",
    sum(p.numel() for p in reverse_model.parameters())
)

In [ ]:
# ============================================================
# RESTORE FORWARD DIFFUSION HELPERS
# ============================================================

def extract(values, timesteps, x_shape):
    batch_size = timesteps.shape[0]

    out = values.gather(0, timesteps)

    return out.reshape(
        batch_size,
        *((1,) * (len(x_shape) - 1))
    )


def q_sample(x0, t, colored_noise):

    sqrt_ab = extract(
        sqrt_alpha_bars,
        t,
        x0.shape
    )

    sqrt_one_minus_ab = extract(
        sqrt_one_minus_alpha_bars,
        t,
        x0.shape
    )

    return (
        sqrt_ab * x0
        + sqrt_one_minus_ab * colored_noise
    )

print("q_sample restored.")

In [ ]:
# ============================================================
# REVERSE MODEL FORWARD-PASS SANITY CHECK
# ============================================================

reverse_model.eval()

with torch.no_grad():

    # Random timestep
    t_test = torch.tensor([50], device=device).long()

    # Forward diffusion state
    x_t_test = q_sample(
        real_original,
        t_test,
        sanc_noise
    )

    # Predict noise
    predicted_noise_test = reverse_model(
        x_t_test,
        real_low_light,
        real_saliency,
        t_test
    )

print("x_t shape:", x_t_test.shape)
print("Low-light shape:", real_low_light.shape)
print("Saliency shape:", real_saliency.shape)
print("Predicted noise shape:", predicted_noise_test.shape)

print(
    "NaN:",
    torch.isnan(predicted_noise_test).any().item()
)

print(
    "Inf:",
    torch.isinf(predicted_noise_test).any().item()
)

In [ ]:
print(reverse_model)

In [ ]:
del reverse_model
torch.cuda.empty_cache()

In [ ]:
reverse_model = DualBranchReverseUNet(
    in_channels=7,
    base_channels=64,
    time_dim=128
).to(device)

print("Reverse model created successfully.")
print(
    "Parameters:",
    sum(p.numel() for p in reverse_model.parameters())
)

In [ ]:
print("Illumination decoder:")
print(reverse_model.illum_dec3)

print("\nDetail decoder:")
print(reverse_model.detail_dec3)

In [ ]:
# ============================================================
# FORWARD-PASS SANITY CHECK
# ============================================================

reverse_model.eval()

with torch.no_grad():

    t_test = torch.tensor(
        [50],
        device=device,
        dtype=torch.long
    )

    x_t_test = q_sample(
        real_original,
        t_test,
        sanc_noise
    )

    predicted_noise_test = reverse_model(
        x_t_test,
        real_low_light,
        real_saliency,
        t_test
    )

print("x_t shape:", x_t_test.shape)
print("Low-light shape:", real_low_light.shape)
print("Saliency shape:", real_saliency.shape)
print(
    "Predicted noise shape:",
    predicted_noise_test.shape
)

print(
    "NaN:",
    torch.isnan(predicted_noise_test).any().item()
)

print(
    "Inf:",
    torch.isinf(predicted_noise_test).any().item()
)

In [ ]:
# ============================================================
# RESTORE FINAL COMMON DATAFRAME
# ============================================================

import os
import pandas as pd

# ------------------------------------------------------------
# Persistent paths
# ------------------------------------------------------------

SANC_ROOT = "/content/drive/MyDrive/SANC"

HAM_ROOT = (
    "/content/drive/MyDrive/SANC/datasets/"
    "ham/HAM preprocessing"
)

ORIGA_ROOT = (
    "/content/drive/MyDrive/SANC/datasets/"
    "ORIGA_processed/integration"
)

KVASIR_ROOT = (
    "/content/drive/MyDrive/SANC/datasets/"
    "kvasir"
)

KVASIR_META = (
    "/content/drive/MyDrive/SANC/datasets/"
    "metadata"
)

# ------------------------------------------------------------
# HAM10000
# ------------------------------------------------------------

ham_meta = pd.read_csv(
    os.path.join(HAM_ROOT, "metadata.csv")
)

ham_df = ham_meta[
    [
        "image_id",
        "image_path",
        "mask_path",
        "low_light_path",
        "dataset",
        "split"
    ]
].copy()

ham_df = ham_df.rename(
    columns={
        "image_id": "filename"
    }
)

# Use the finalized HAM paths
ham_df["original"] = ham_df["image_path"]
ham_df["low_light"] = ham_df["low_light_path"]
ham_df["mask"] = ham_df["mask_path"]

ham_df["dataset"] = "HAM10000"

ham_df = ham_df[
    [
        "filename",
        "original",
        "low_light",
        "mask",
        "dataset",
        "split"
    ]
]


# ------------------------------------------------------------
# ORIGA
# ------------------------------------------------------------

origa_meta = pd.read_csv(
    os.path.join(
        ORIGA_ROOT,
        "ORIGA_common_metadata.csv"
    )
)

origa_df = origa_meta[
    [
        "filename",
        "original",
        "low_light",
        "mask",
        "dataset",
        "split"
    ]
].copy()

origa_df["dataset"] = "ORIGA"


# ------------------------------------------------------------
# Kvasir-SEG
# ------------------------------------------------------------

kvasir_train = pd.read_csv(
    os.path.join(
        KVASIR_META,
        "kvasir_train.csv"
    )
)

kvasir_val = pd.read_csv(
    os.path.join(
        KVASIR_META,
        "kvasir_val.csv"
    )
)

kvasir_test = pd.read_csv(
    os.path.join(
        KVASIR_META,
        "kvasir_test.csv"
    )
)


def get_filename_column(df):

    possible = [
        "filename",
        "image",
        "image_id",
        "id"
    ]

    for col in possible:
        if col in df.columns:
            return col

    raise ValueError(
        f"Could not find filename column. "
        f"Available columns: {list(df.columns)}"
    )


def prepare_kvasir_split(df, split_name):

    df = df.copy()

    filename_col = get_filename_column(df)

    df["filename"] = df[filename_col].astype(str)

    df["original"] = df["filename"].apply(
        lambda x: os.path.join(
            KVASIR_ROOT,
            "original",
            x
        )
    )

    df["low_light"] = df["filename"].apply(
        lambda x: os.path.join(
            KVASIR_ROOT,
            "low_light",
            x
        )
    )

    df["mask"] = df["filename"].apply(
        lambda x: os.path.join(
            KVASIR_ROOT,
            "masks",
            x
        )
    )

    df["dataset"] = "Kvasir-SEG"
    df["split"] = split_name

    return df[
        [
            "filename",
            "original",
            "low_light",
            "mask",
            "dataset",
            "split"
        ]
    ]


kvasir_train_df = prepare_kvasir_split(
    kvasir_train,
    "train"
)

kvasir_val_df = prepare_kvasir_split(
    kvasir_val,
    "val"
)

kvasir_test_df = prepare_kvasir_split(
    kvasir_test,
    "test"
)


# ------------------------------------------------------------
# Combine all datasets
# ------------------------------------------------------------

combined_df = pd.concat(
    [
        ham_df,
        origa_df,
        kvasir_train_df,
        kvasir_val_df,
        kvasir_test_df
    ],
    ignore_index=True
)


# ------------------------------------------------------------
# Final split distribution
# ------------------------------------------------------------

print("Combined samples:", len(combined_df))

print("\nSplit distribution:")
print(
    combined_df["split"]
    .value_counts()
    .sort_index()
)

print("\nDataset distribution:")
print(
    combined_df["dataset"]
    .value_counts()
)


# ------------------------------------------------------------
# Expected result
# ------------------------------------------------------------

assert len(combined_df) == 2150, (
    f"Expected 2150 candidates, "
    f"got {len(combined_df)}"
)

print("\ncombined_df restored successfully.")

In [ ]:
# ============================================================
# SANC MEDICAL DATASET
# ============================================================

from torch.utils.data import Dataset
from PIL import Image
import numpy as np
import torch


class SANCMedicalDataset(Dataset):

    def __init__(self, dataframe, image_size=128):
        self.df = dataframe.reset_index(drop=True)
        self.image_size = image_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        # ----------------------------------------------------
        # Load images
        # ----------------------------------------------------

        low_light = Image.open(
            row["low_light"]
        ).convert("RGB")

        original = Image.open(
            row["original"]
        ).convert("RGB")

        mask = Image.open(
            row["mask"]
        ).convert("L")

        # ----------------------------------------------------
        # Resize
        # ----------------------------------------------------

        low_light = low_light.resize(
            (self.image_size, self.image_size),
            Image.Resampling.BILINEAR
        )

        original = original.resize(
            (self.image_size, self.image_size),
            Image.Resampling.BILINEAR
        )

        mask = mask.resize(
            (self.image_size, self.image_size),
            Image.Resampling.NEAREST
        )

        # ----------------------------------------------------
        # Convert to NumPy
        # ----------------------------------------------------

        low_light = np.array(
            low_light,
            dtype=np.float32,
            copy=True
        ) / 255.0

        original = np.array(
            original,
            dtype=np.float32,
            copy=True
        ) / 255.0

        mask = np.array(
            mask,
            dtype=np.float32,
            copy=True
        )

        # ----------------------------------------------------
        # Robust binary mask handling
        # ----------------------------------------------------

        mask = (mask > 0).astype(np.float32)

        # ----------------------------------------------------
        # HWC → CHW
        # ----------------------------------------------------

        low_light = torch.from_numpy(
            low_light
        ).permute(2, 0, 1)

        original = torch.from_numpy(
            original
        ).permute(2, 0, 1)

        mask = torch.from_numpy(
            mask
        ).unsqueeze(0)

        # ----------------------------------------------------
        # Return
        # ----------------------------------------------------

        return {
            "low_light": low_light,
            "original": original,
            "mask": mask,
            "dataset": row["dataset"],
            "filename": row["filename"],
            "split": row["split"]
        }


print("SANCMedicalDataset restored successfully.")

In [ ]:
# ============================================================
# CREATE DATASETS AND LOADERS
# ============================================================

BATCH_SIZE = 8

train_dataset = SANCMedicalDataset(
    train_data
)

val_dataset = SANCMedicalDataset(
    val_data
)

test_dataset = SANCMedicalDataset(
    test_data
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:
# ============================================================
# FIX FINAL DATAFRAME PATHS
# ============================================================

import os

# ------------------------------------------------------------
# HAM10000
# ------------------------------------------------------------

ham_mask = combined_df["dataset"] == "HAM10000"

combined_df.loc[ham_mask, "filename"] = (
    combined_df.loc[ham_mask, "filename"]
    .astype(str)
    .apply(os.path.basename)
)

combined_df.loc[ham_mask, "original"] = (
    combined_df.loc[ham_mask, "filename"]
    .apply(lambda x: os.path.join(
        HAM_ROOT,
        "images",
        "images",
        x
    ))
)

combined_df.loc[ham_mask, "low_light"] = (
    combined_df.loc[ham_mask, "filename"]
    .apply(lambda x: os.path.join(
        HAM_ROOT,
        "low_light",
        "low_light",
        x
    ))
)

combined_df.loc[ham_mask, "mask"] = (
    combined_df.loc[ham_mask, "filename"]
    .apply(lambda x: os.path.join(
        HAM_ROOT,
        "masks",
        "masks",
        x
    ))
)


# ------------------------------------------------------------
# ORIGA
# ------------------------------------------------------------

origa_mask = combined_df["dataset"] == "ORIGA"

combined_df.loc[origa_mask, "filename"] = (
    combined_df.loc[origa_mask, "filename"]
    .astype(str)
    .apply(os.path.basename)
)

combined_df.loc[origa_mask, "original"] = (
    combined_df.loc[origa_mask, "filename"]
    .apply(lambda x: os.path.join(
        ORIGA_ROOT,
        "images",
        x
    ))
)

combined_df.loc[origa_mask, "low_light"] = (
    combined_df.loc[origa_mask, "filename"]
    .apply(lambda x: os.path.join(
        ORIGA_ROOT,
        "low_light",
        x
    ))
)

combined_df.loc[origa_mask, "mask"] = (
    combined_df.loc[origa_mask, "filename"]
    .apply(lambda x: os.path.join(
        ORIGA_ROOT,
        "masks",
        x
    ))
)


# ------------------------------------------------------------
# Kvasir-SEG
# ------------------------------------------------------------

kvasir_mask = combined_df["dataset"] == "Kvasir-SEG"

combined_df.loc[kvasir_mask, "filename"] = (
    combined_df.loc[kvasir_mask, "filename"]
    .astype(str)
    .apply(os.path.basename)
)

combined_df.loc[kvasir_mask, "original"] = (
    combined_df.loc[kvasir_mask, "filename"]
    .apply(lambda x: os.path.join(
        KVASIR_ROOT,
        "original",
        x
    ))
)

combined_df.loc[kvasir_mask, "low_light"] = (
    combined_df.loc[kvasir_mask, "filename"]
    .apply(lambda x: os.path.join(
        KVASIR_ROOT,
        "low_light",
        x
    ))
)

combined_df.loc[kvasir_mask, "mask"] = (
    combined_df.loc[kvasir_mask, "filename"]
    .apply(lambda x: os.path.join(
        KVASIR_ROOT,
        "masks",
        x
    ))
)

print("Paths normalized successfully.")

In [ ]:
# ============================================================
# VERIFY ALL DATASET PATHS
# ============================================================

required_columns = [
    "original",
    "low_light",
    "mask"
]

for column in required_columns:

    exists = combined_df[column].apply(
        os.path.exists
    )

    print(
        f"{column}: "
        f"{exists.sum()}/{len(exists)} files exist"
    )

In [ ]:
# ============================================================
# FIND WHICH DATASET HAS MISSING FILES
# ============================================================

for dataset_name in combined_df["dataset"].unique():

    subset = combined_df[
        combined_df["dataset"] == dataset_name
    ]

    print(f"\n{dataset_name}: {len(subset)} samples")

    for column in ["original", "low_light", "mask"]:

        exists = subset[column].apply(os.path.exists)

        print(
            f"  {column}: "
            f"{exists.sum()}/{len(exists)} exist"
        )

In [ ]:
# ============================================================
# DEFINITIVE HAM10000 PATH FIX
# ============================================================

import os

ham_mask = combined_df["dataset"] == "HAM10000"

# HAM filenames are ISIC_XXXXXXX.png
combined_df.loc[ham_mask, "filename"] = (
    combined_df.loc[ham_mask, "filename"]
    .astype(str)
    .apply(lambda x: os.path.splitext(os.path.basename(x))[0] + ".png")
)

# Actual HAM directories
combined_df.loc[ham_mask, "original"] = (
    combined_df.loc[ham_mask, "filename"]
    .apply(lambda x: os.path.join(
        HAM_ROOT,
        "images",
        "images",
        x
    ))
)

combined_df.loc[ham_mask, "low_light"] = (
    combined_df.loc[ham_mask, "filename"]
    .apply(lambda x: os.path.join(
        HAM_ROOT,
        "low_light",
        "low_light",
        x
    ))
)

combined_df.loc[ham_mask, "mask"] = (
    combined_df.loc[ham_mask, "filename"]
    .apply(lambda x: os.path.join(
        HAM_ROOT,
        "masks",
        "masks",
        x
    ))
)

print("HAM paths updated.")

In [ ]:
# ============================================================
# TEST HAM FILE
# ============================================================

ham_test = combined_df[
    combined_df["dataset"] == "HAM10000"
].iloc[0]

print("Filename:", ham_test["filename"])
print()
print("Original:", ham_test["original"])
print("Exists:", os.path.exists(ham_test["original"]))
print()
print("Low-light:", ham_test["low_light"])
print("Exists:", os.path.exists(ham_test["low_light"]))
print()
print("Mask:", ham_test["mask"])
print("Exists:", os.path.exists(ham_test["mask"]))

In [ ]:
# ============================================================
# FINAL PATH VALIDATION
# ============================================================

for dataset_name in combined_df["dataset"].unique():

    subset = combined_df[
        combined_df["dataset"] == dataset_name
    ]

    print(f"\n{dataset_name}: {len(subset)} samples")

    for column in ["original", "low_light", "mask"]:

        exists = subset[column].apply(os.path.exists)

        print(
            f"  {column}: "
            f"{exists.sum()}/{len(exists)} exist"
        )

In [ ]:
# ============================================================
# CREATE FINAL LOADERS
# ============================================================

train_data = combined_df[
    combined_df["split"] == "train"
].reset_index(drop=True)

val_data = combined_df[
    combined_df["split"] == "val"
].reset_index(drop=True)

test_data = combined_df[
    combined_df["split"] == "test"
].reset_index(drop=True)

BATCH_SIZE = 8

train_dataset = SANCMedicalDataset(train_data)
val_dataset = SANCMedicalDataset(val_data)
test_dataset = SANCMedicalDataset(test_data)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:
# ============================================================
# TEST ONE TRAINING BATCH
# ============================================================

batch = next(iter(train_loader))

print("Low-light:", batch["low_light"].shape)
print("Original:", batch["original"].shape)
print("Mask:", batch["mask"].shape)
print("Dataset:", batch["dataset"][:3])
print("Filename:", batch["filename"][:3])

In [ ]:
# ============================================================
# REVERSE DIFFUSION — ONE BATCH SANITY CHECK
# ============================================================

reverse_model.train()
saliency_model.eval()

batch = next(iter(train_loader))

low_light = batch["low_light"].to(device)
original = batch["original"].to(device)

# 1. Predict saliency
with torch.no_grad():
    saliency = saliency_model(low_light)

# 2. Generate Gaussian noise
gaussian_noise = torch.randn_like(original)

# 3. SANC-colored noise
colored_noise = sanc_noise_coloring(
    gaussian_noise,
    saliency
)

# 4. Random timestep
t = torch.randint(
    0,
    T,
    (original.size(0),),
    device=device
).long()

# 5. Forward diffusion
x_t = q_sample(
    original,
    t,
    colored_noise
)

# 6. Predict noise
predicted_noise = reverse_model(
    x_t,
    low_light,
    saliency,
    t
)

# 7. Loss
loss = F.mse_loss(
    predicted_noise,
    colored_noise
)

print("Original:", original.shape)
print("Saliency:", saliency.shape)
print("SANC noise:", colored_noise.shape)
print("x_t:", x_t.shape)
print("Predicted noise:", predicted_noise.shape)

print()
print("Loss:", loss.item())
print("NaN:", torch.isnan(loss).item())
print("Inf:", torch.isinf(loss).item())

In [ ]:
# ============================================================
# TRAIN CONDITIONAL REVERSE DIFFUSION MODEL
# ============================================================

import torch
import torch.nn.functional as F

optimizer = torch.optim.AdamW(
    reverse_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

NUM_EPOCHS = 10

reverse_model.train()
saliency_model.eval()

for epoch in range(NUM_EPOCHS):

    running_loss = 0.0

    for batch in train_loader:

        low_light = batch["low_light"].to(device)
        original = batch["original"].to(device)

        # ----------------------------------------------------
        # 1. Predict continuous saliency
        # ----------------------------------------------------

        with torch.no_grad():
            saliency = saliency_model(low_light)

        # ----------------------------------------------------
        # 2. Generate Gaussian noise
        # ----------------------------------------------------

        gaussian_noise = torch.randn_like(original)

        # ----------------------------------------------------
        # 3. Apply SANC spectral coloring
        # ----------------------------------------------------

        colored_noise = sanc_noise_coloring(
            gaussian_noise,
            saliency
        )

        # ----------------------------------------------------
        # 4. Sample random diffusion timestep
        # ----------------------------------------------------

        t = torch.randint(
            low=0,
            high=T,
            size=(original.size(0),),
            device=device
        ).long()

        # ----------------------------------------------------
        # 5. Forward diffusion
        # ----------------------------------------------------

        x_t = q_sample(
            original,
            t,
            colored_noise
        )

        # ----------------------------------------------------
        # 6. Predict colored noise
        # ----------------------------------------------------

        predicted_noise = reverse_model(
            x_t,
            low_light,
            saliency,
            t
        )

        # ----------------------------------------------------
        # 7. Diffusion noise-prediction loss
        # ----------------------------------------------------

        loss = F.mse_loss(
            predicted_noise,
            colored_noise
        )

        # ----------------------------------------------------
        # 8. Backpropagation
        # ----------------------------------------------------

        optimizer.zero_grad(
            set_to_none=True
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    epoch_loss = (
        running_loss /
        len(train_loader)
    )

    print(
        f"Epoch {epoch + 1:02d}/{NUM_EPOCHS} "
        f"| Train Loss: {epoch_loss:.6f}"
    )

In [ ]:
# ============================================================
# RESTORE SANC NOISE COLORING MODULE
# ============================================================

class SANCNoiseColoring(nn.Module):

    def __init__(self, saliency_power=1.0):
        super().__init__()
        self.saliency_power = saliency_power

    def _frequency_bands(self, H, W, device):

        fy = torch.fft.fftfreq(
            H,
            device=device
        )

        fx = torch.fft.fftfreq(
            W,
            device=device
        )

        yy, xx = torch.meshgrid(
            fy,
            fx,
            indexing="ij"
        )

        radius = torch.sqrt(
            xx ** 2 + yy ** 2
        )

        radius = radius / (
            radius.max() + 1e-8
        )

        # Smooth frequency bands
        low = torch.exp(
            -0.5 * (
                (radius - 0.15) / 0.20
            ) ** 2
        )

        mid = torch.exp(
            -0.5 * (
                (radius - 0.45) / 0.20
            ) ** 2
        )

        high = torch.exp(
            -0.5 * (
                (radius - 0.80) / 0.20
            ) ** 2
        )

        # Normalize so bands approximately partition energy
        total = low + mid + high + 1e-8

        low = low / total
        mid = mid / total
        high = high / total

        return low, mid, high

    def forward(self, noise, saliency):

        B, C, H, W = noise.shape

        # ----------------------------------------------------
        # Resize saliency if necessary
        # ----------------------------------------------------

        saliency = F.interpolate(
            saliency,
            size=(H, W),
            mode="bilinear",
            align_corners=False
        )

        saliency = torch.clamp(
            saliency,
            0.0,
            1.0
        )

        saliency = saliency ** self.saliency_power

        # ----------------------------------------------------
        # Frequency bands
        # ----------------------------------------------------

        low_band, mid_band, high_band = (
            self._frequency_bands(
                H,
                W,
                noise.device
            )
        )

        low_band = low_band[None, None, :, :]
        mid_band = mid_band[None, None, :, :]
        high_band = high_band[None, None, :, :]

        # ----------------------------------------------------
        # FFT of Gaussian noise
        # ----------------------------------------------------

        noise_fft = torch.fft.fft2(
            noise
        )

        # ----------------------------------------------------
        # Extract frequency components
        # ----------------------------------------------------

        low_fft = (
            noise_fft * low_band
        )

        mid_fft = (
            noise_fft * mid_band
        )

        high_fft = (
            noise_fft * high_band
        )

        # ----------------------------------------------------
        # Convert back to spatial domain
        # ----------------------------------------------------

        low_noise = torch.fft.ifft2(
            low_fft
        ).real

        mid_noise = torch.fft.ifft2(
            mid_fft
        ).real

        high_noise = torch.fft.ifft2(
            high_fft
        ).real

        # ----------------------------------------------------
        # Saliency-dependent spectral weighting
        # ----------------------------------------------------
        #
        # Background:
        #   low = 0.55
        #   mid = 0.30
        #   high = 0.15
        #
        # High saliency:
        #   low = 0.20
        #   mid = 0.30
        #   high = 0.50
        #

        low_weight = (
            0.55
            - 0.35 * saliency
        )

        mid_weight = (
            torch.ones_like(saliency)
            * 0.30
        )

        high_weight = (
            0.15
            + 0.35 * saliency
        )

        # ----------------------------------------------------
        # Spatially adaptive spectral mixture
        # ----------------------------------------------------

        colored_noise = (
            low_weight * low_noise
            + mid_weight * mid_noise
            + high_weight * high_noise
        )

        # ----------------------------------------------------
        # Per-sample/channel normalization
        # ----------------------------------------------------

        mean = colored_noise.mean(
            dim=(2, 3),
            keepdim=True
        )

        std = colored_noise.std(
            dim=(2, 3),
            keepdim=True
        )

        colored_noise = (
            colored_noise - mean
        ) / (
            std + 1e-8
        )

        return colored_noise


# ------------------------------------------------------------
# Create SANC module
# ------------------------------------------------------------

sanc_noise_coloring = SANCNoiseColoring(
    saliency_power=1.0
).to(device)

print("SANC noise coloring restored successfully.")

In [ ]:
# ============================================================
# REVERSE DIFFUSION — COMPLETE SANITY CHECK
# ============================================================

from torch.utils.data import DataLoader

# ------------------------------------------------------------
# 1. Recreate data splits
# ------------------------------------------------------------

train_data = combined_df[
    combined_df["split"] == "train"
].reset_index(drop=True)

# ------------------------------------------------------------
# 2. Recreate training dataset
# ------------------------------------------------------------

train_dataset = SANCMedicalDataset(train_data)

# ------------------------------------------------------------
# 3. Recreate training loader
# ------------------------------------------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0
)

# ------------------------------------------------------------
# 4. Get one batch
# ------------------------------------------------------------

batch = next(iter(train_loader))

low_light = batch["low_light"].to(device)
original = batch["original"].to(device)

# ------------------------------------------------------------
# 5. Predict saliency
# ------------------------------------------------------------

saliency_model.eval()
reverse_model.eval()

with torch.no_grad():

    saliency = saliency_model(
        low_light
    )

# ------------------------------------------------------------
# 6. Generate Gaussian noise
# ------------------------------------------------------------

gaussian_noise = torch.randn_like(
    original
)

# ------------------------------------------------------------
# 7. Generate SANC-colored noise
# ------------------------------------------------------------

with torch.no_grad():

    colored_noise = sanc_noise_coloring(
        gaussian_noise,
        saliency
    )

# ------------------------------------------------------------
# 8. Sample random timestep
# ------------------------------------------------------------

t = torch.randint(
    0,
    T,
    (original.size(0),),
    device=device
).long()

# ------------------------------------------------------------
# 9. Forward diffusion
# ------------------------------------------------------------

with torch.no_grad():

    x_t = q_sample(
        original,
        t,
        colored_noise
    )

# ------------------------------------------------------------
# 10. Reverse U-Net prediction
# ------------------------------------------------------------

with torch.no_grad():

    predicted_noise = reverse_model(
        x_t,
        low_light,
        saliency,
        t
    )

# ------------------------------------------------------------
# 11. Calculate loss
# ------------------------------------------------------------

loss = F.mse_loss(
    predicted_noise,
    colored_noise
)

# ------------------------------------------------------------
# 12. Report
# ------------------------------------------------------------

print("========== REVERSE DIFFUSION SANITY CHECK ==========")

print("Original:          ", original.shape)
print("Low-light:         ", low_light.shape)
print("Saliency:          ", saliency.shape)
print("SANC noise:        ", colored_noise.shape)
print("x_t:               ", x_t.shape)
print("Predicted noise:   ", predicted_noise.shape)

print()
print("Timestep:", t.tolist())
print("Loss:", loss.item())
print("NaN:", torch.isnan(loss).item())
print("Inf:", torch.isinf(loss).item())

In [ ]:
# ============================================================
# TRAIN CONDITIONAL REVERSE DIFFUSION MODEL
# ============================================================

optimizer = torch.optim.AdamW(
    reverse_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

NUM_EPOCHS = 10

saliency_model.eval()

for epoch in range(NUM_EPOCHS):

    reverse_model.train()

    running_loss = 0.0

    for batch in train_loader:

        low_light = batch["low_light"].to(device)
        original = batch["original"].to(device)

        # ----------------------------------------------------
        # Predict saliency
        # ----------------------------------------------------

        with torch.no_grad():
            saliency = saliency_model(
                low_light
            )

        # ----------------------------------------------------
        # Generate Gaussian noise
        # ----------------------------------------------------

        gaussian_noise = torch.randn_like(
            original
        )

        # ----------------------------------------------------
        # SANC-colored noise
        # ----------------------------------------------------

        with torch.no_grad():
            colored_noise = sanc_noise_coloring(
                gaussian_noise,
                saliency
            )

        # ----------------------------------------------------
        # Random timestep
        # ----------------------------------------------------

        t = torch.randint(
            0,
            T,
            (original.size(0),),
            device=device
        ).long()

        # ----------------------------------------------------
        # Forward diffusion
        # ----------------------------------------------------

        x_t = q_sample(
            original,
            t,
            colored_noise
        )

        # ----------------------------------------------------
        # Reverse model predicts noise
        # ----------------------------------------------------

        predicted_noise = reverse_model(
            x_t,
            low_light,
            saliency,
            t
        )

        # ----------------------------------------------------
        # Noise prediction loss
        # ----------------------------------------------------

        loss = F.mse_loss(
            predicted_noise,
            colored_noise
        )

        # ----------------------------------------------------
        # Optimization
        # ----------------------------------------------------

        optimizer.zero_grad(
            set_to_none=True
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    epoch_loss = (
        running_loss /
        len(train_loader)
    )

    print(
        f"Epoch {epoch + 1:02d}/{NUM_EPOCHS} "
        f"| Train Loss: {epoch_loss:.6f}"
    )

In [ ]:
# ============================================================
# SAVE TRAINED REVERSE MODEL
# ============================================================

reverse_checkpoint_path = (
    "/content/drive/MyDrive/SANC/models/"
    "reverse_diffusion_model.pth"
)

torch.save(
    {
        "model_state_dict": reverse_model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "num_epochs": NUM_EPOCHS
    },
    reverse_checkpoint_path
)

print("Saved:", reverse_checkpoint_path)

In [ ]:
# ============================================================
# TEST TRAINED REVERSE DIFFUSION
# ============================================================

import matplotlib.pyplot as plt
import torch


# ------------------------------------------------------------
# 1. Find a valid test sample
# ------------------------------------------------------------

valid_test_data = test_data[
    test_data["original"].apply(os.path.exists) &
    test_data["low_light"].apply(os.path.exists) &
    test_data["mask"].apply(os.path.exists)
].reset_index(drop=True)

print("Valid test samples:", len(valid_test_data))


# ------------------------------------------------------------
# 2. Select one test sample
# ------------------------------------------------------------

sample_dataset = SANCMedicalDataset(
    valid_test_data.iloc[[0]]
)

sample = sample_dataset[0]

real_original = sample["original"].unsqueeze(0).to(device)
real_low_light = sample["low_light"].unsqueeze(0).to(device)


# ------------------------------------------------------------
# 3. Predict saliency
# ------------------------------------------------------------

saliency_model.eval()
reverse_model.eval()

with torch.no_grad():

    real_saliency = saliency_model(
        real_low_light
    )


# ------------------------------------------------------------
# 4. Start reverse diffusion from random noise
# ------------------------------------------------------------

x_t = torch.randn_like(
    real_original
)

trajectory = {}


# ------------------------------------------------------------
# 5. Reverse diffusion
# ------------------------------------------------------------

with torch.no_grad():

    for timestep in reversed(range(T)):

        t = torch.full(
            (1,),
            timestep,
            device=device,
            dtype=torch.long
        )

        # Predict SANC-colored noise
        predicted_noise = reverse_model(
            x_t,
            real_low_light,
            real_saliency,
            t
        )

        # DDPM-style reverse step
        beta_t = extract(
            betas,
            t,
            x_t.shape
        )

        alpha_t = extract(
            alphas,
            t,
            x_t.shape
        )

        alpha_bar_t = extract(
            alpha_bars,
            t,
            x_t.shape
        )

        mean = (
            1.0 / torch.sqrt(alpha_t)
        ) * (
            x_t
            - (
                beta_t /
                torch.sqrt(
                    1.0 - alpha_bar_t
                )
            ) * predicted_noise
        )

        posterior_var = extract(
            posterior_variance,
            t,
            x_t.shape
        )

        noise = torch.randn_like(x_t)

        nonzero_mask = (
            (t != 0)
            .float()
            .view(-1, 1, 1, 1)
        )

        x_t = (
            mean
            + nonzero_mask
            * torch.sqrt(
                posterior_var
            )
            * noise
        )

        # Store selected timesteps
        if timestep in [99, 75, 50, 25, 10, 0]:
            trajectory[timestep] = (
                x_t.detach().clone()
            )


final_output = x_t.detach().clone()


# ------------------------------------------------------------
# 6. Check numerical validity
# ------------------------------------------------------------

print("\n========== REVERSE TEST ==========")

print(
    "Filename:",
    sample["filename"]
)

print(
    "Dataset:",
    sample["dataset"]
)

print(
    "Original:",
    real_original.shape
)

print(
    "Low-light:",
    real_low_light.shape
)

print(
    "Saliency:",
    real_saliency.shape
)

print(
    "Final output:",
    final_output.shape
)

print(
    "NaN:",
    torch.isnan(final_output).any().item()
)

print(
    "Inf:",
    torch.isinf(final_output).any().item()
)


# ------------------------------------------------------------
# 7. Visualization helper
# ------------------------------------------------------------

def display_tensor(tensor):

    tensor = tensor.detach().cpu()

    if tensor.dim() == 4:
        tensor = tensor[0]

    image = tensor.permute(
        1, 2, 0
    ).numpy()

    return np.clip(
        image,
        0,
        1
    )


# ------------------------------------------------------------
# 8. Prepare saliency
# ------------------------------------------------------------

saliency_display = (
    real_saliency[0, 0]
    .detach()
    .cpu()
    .numpy()
)


# ------------------------------------------------------------
# 9. Plot result
# ------------------------------------------------------------

fig, axes = plt.subplots(
    2,
    6,
    figsize=(18, 7)
)

# Top row
axes[0, 0].imshow(
    display_tensor(real_original)
)
axes[0, 0].set_title("Original")

axes[0, 1].imshow(
    display_tensor(real_low_light)
)
axes[0, 1].set_title("Low-light")

axes[0, 2].imshow(
    saliency_display,
    cmap="gray"
)
axes[0, 2].set_title("Predicted Saliency")

axes[0, 3].imshow(
    display_tensor(trajectory[99])
)
axes[0, 3].set_title("Generated t=99")

axes[0, 4].imshow(
    display_tensor(trajectory[50])
)
axes[0, 4].set_title("Generated t=50")

axes[0, 5].imshow(
    display_tensor(final_output)
)
axes[0, 5].set_title("Generated x₀")

# Bottom row
for ax, timestep in zip(
    axes[1],
    [99, 75, 50, 25, 10, 0]
):

    ax.imshow(
        display_tensor(
            trajectory[timestep]
        )
    )

    ax.set_title(
        f"Reverse t = {timestep}"
    )

# Remove axes
for ax in axes.flat:
    ax.axis("off")

plt.tight_layout()
plt.show()

**# After Mid Review**

In [ ]:

from google.colab import drive
drive.mount('/content/drive')


In [ ]:

import os

SANC_ROOT = "/content/drive/MyDrive/SANC"

print("SANC folder exists:", os.path.isdir(SANC_ROOT))

if os.path.isdir(SANC_ROOT):
    for name in os.listdir(SANC_ROOT):
        print(name)
else:
    print("SANC was not found at the expected path.")
    print("MyDrive entries:")
    print(os.listdir("/content/drive/MyDrive")[:40])


In [ ]:

import os

model_dir = os.path.join(SANC_ROOT, "models")

print("Model directory exists:", os.path.isdir(model_dir))

if os.path.isdir(model_dir):
    for name in sorted(os.listdir(model_dir)):
        path = os.path.join(model_dir, name)
        if os.path.isfile(path):
            print(f"{name} — {os.path.getsize(path) / 1024**2:.2f} MB")


In [ ]:

# ============================================================
# SANC SALIENCY DEBUGGING — CONSOLIDATED DATASET SETUP
# Replaces notebook cells 224–234
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from PIL import Image
from torch.utils.data import Dataset, DataLoader

# ------------------------------------------------------------
# 1. CONFIGURATION AND DRIVE PATHS
# ------------------------------------------------------------

SANC_ROOT = "/content/drive/MyDrive/SANC"

HAM_ROOT = os.path.join(
    SANC_ROOT, "datasets", "ham", "HAM preprocessing"
)
ORIGA_ROOT = os.path.join(
    SANC_ROOT, "datasets", "ORIGA_processed", "integration"
)
KVASIR_ROOT = os.path.join(
    SANC_ROOT, "datasets", "kvasir"
)
KVASIR_META = os.path.join(
    SANC_ROOT, "datasets", "metadata"
)

required_csvs = {
    "HAM metadata": os.path.join(HAM_ROOT, "metadata.csv"),
    "ORIGA metadata": os.path.join(
        ORIGA_ROOT, "ORIGA_common_metadata.csv"
    ),
    "Kvasir train": os.path.join(KVASIR_META, "kvasir_train.csv"),
    "Kvasir val": os.path.join(KVASIR_META, "kvasir_val.csv"),
    "Kvasir test": os.path.join(KVASIR_META, "kvasir_test.csv"),
}

print("=" * 60)
print("CHECKING REQUIRED METADATA")
print("=" * 60)

missing_csvs = [
    f"{name}: {path}"
    for name, path in required_csvs.items()
    if not os.path.isfile(path)
]

if missing_csvs:
    raise FileNotFoundError(
        "Required CSV files not found:\n" +
        "\n".join(missing_csvs) +
        "\n\nCheck SANC_ROOT and your shared Drive shortcut."
    )

for name, path in required_csvs.items():
    print(f"FOUND: {name}")

# ------------------------------------------------------------
# 2. REBUILD HAM10000 DATAFRAME
# ------------------------------------------------------------

ham_meta = pd.read_csv(required_csvs["HAM metadata"])

ham_required = [
    "image_id", "image_path", "mask_path",
    "low_light_path", "dataset", "split"
]

missing_cols = set(ham_required) - set(ham_meta.columns)
if missing_cols:
    raise ValueError(
        f"HAM metadata missing columns: {sorted(missing_cols)}"
    )

ham_df = ham_meta[ham_required].copy()

ham_df = ham_df.rename(columns={"image_id": "filename"})
ham_df["original"] = ham_df["image_path"]
ham_df["low_light"] = ham_df["low_light_path"]
ham_df["mask"] = ham_df["mask_path"]
ham_df["dataset"] = "HAM10000"

ham_df = ham_df[
    ["filename", "original", "low_light", "mask", "dataset", "split"]
].copy()

# ------------------------------------------------------------
# 3. REBUILD ORIGA DATAFRAME
# ------------------------------------------------------------

origa_meta = pd.read_csv(required_csvs["ORIGA metadata"])

origa_required = [
    "filename", "original", "low_light",
    "mask", "dataset", "split"
]

missing_cols = set(origa_required) - set(origa_meta.columns)
if missing_cols:
    raise ValueError(
        f"ORIGA metadata missing columns: {sorted(missing_cols)}"
    )

origa_df = origa_meta[origa_required].copy()
origa_df["dataset"] = "ORIGA"

# ------------------------------------------------------------
# 4. REBUILD KVASIR DATAFRAMES
# ------------------------------------------------------------

def get_filename_column(df):
    for col in ["filename", "image", "image_id", "id"]:
        if col in df.columns:
            return col
    raise ValueError(
        f"No filename column found. Columns: {list(df.columns)}"
    )


def prepare_kvasir_split(df, split_name):
    df = df.copy()
    filename_col = get_filename_column(df)

    df["filename"] = df[filename_col].astype(str).apply(
        lambda x: os.path.basename(x)
    )

    df["original"] = df["filename"].apply(
        lambda x: os.path.join(KVASIR_ROOT, "original", x)
    )
    df["low_light"] = df["filename"].apply(
        lambda x: os.path.join(KVASIR_ROOT, "low_light", x)
    )
    df["mask"] = df["filename"].apply(
        lambda x: os.path.join(KVASIR_ROOT, "masks", x)
    )

    df["dataset"] = "Kvasir-SEG"
    df["split"] = split_name

    return df[
        ["filename", "original", "low_light", "mask", "dataset", "split"]
    ].copy()


kvasir_train_df = prepare_kvasir_split(
    pd.read_csv(required_csvs["Kvasir train"]), "train"
)
kvasir_val_df = prepare_kvasir_split(
    pd.read_csv(required_csvs["Kvasir val"]), "val"
)
kvasir_test_df = prepare_kvasir_split(
    pd.read_csv(required_csvs["Kvasir test"]), "test"
)

# ------------------------------------------------------------
# 5. COMBINE ALL DATASETS
# ------------------------------------------------------------

combined_df = pd.concat(
    [ham_df, origa_df, kvasir_train_df,
     kvasir_val_df, kvasir_test_df],
    ignore_index=True
)

# ------------------------------------------------------------
# 6. NORMALIZE PATHS AS IN THE ORIGINAL NOTEBOOK
# ------------------------------------------------------------

# HAM10000 paths
ham_idx = combined_df["dataset"].eq("HAM10000")

combined_df.loc[ham_idx, "filename"] = (
    combined_df.loc[ham_idx, "filename"]
    .astype(str)
    .apply(lambda x: os.path.splitext(os.path.basename(x))[0] + ".png")
)

for column, subdir in [
    ("original", os.path.join("images", "images")),
    ("low_light", os.path.join("low_light", "low_light")),
    ("mask", os.path.join("masks", "masks")),
]:
    combined_df.loc[ham_idx, column] = (
        combined_df.loc[ham_idx, "filename"]
        .apply(lambda x: os.path.join(HAM_ROOT, subdir, x))
    )

# ORIGA paths
origa_idx = combined_df["dataset"].eq("ORIGA")

combined_df.loc[origa_idx, "filename"] = (
    combined_df.loc[origa_idx, "filename"]
    .astype(str)
    .apply(lambda x: os.path.basename(x))
)

for column, subdir in [
    ("original", "images"),
    ("low_light", "low_light"),
    ("mask", "masks"),
]:
    combined_df.loc[origa_idx, column] = (
        combined_df.loc[origa_idx, "filename"]
        .apply(lambda x: os.path.join(ORIGA_ROOT, subdir, x))
    )

# Kvasir paths
kvasir_idx = combined_df["dataset"].eq("Kvasir-SEG")

combined_df.loc[kvasir_idx, "filename"] = (
    combined_df.loc[kvasir_idx, "filename"]
    .astype(str)
    .apply(lambda x: os.path.basename(x))
)

for column, subdir in [
    ("original", "original"),
    ("low_light", "low_light"),
    ("mask", "masks"),
]:
    combined_df.loc[kvasir_idx, column] = (
        combined_df.loc[kvasir_idx, "filename"]
        .apply(lambda x: os.path.join(KVASIR_ROOT, subdir, x))
    )

# ------------------------------------------------------------
# 7. CHECK DUPLICATES AND DATA SPLITS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("DATAFRAME SUMMARY")
print("=" * 60)

print("Total rows:", len(combined_df))
print("\nSamples per dataset:")
print(combined_df["dataset"].value_counts(dropna=False))

print("\nSamples per split:")
print(combined_df["split"].value_counts(dropna=False))

print("\nDataset x split:")
print(pd.crosstab(combined_df["dataset"], combined_df["split"]))

duplicate_count = combined_df.duplicated(
    subset=["dataset", "filename"]
).sum()
print("\nDuplicate dataset/filename rows:", duplicate_count)

# ------------------------------------------------------------
# 8. VERIFY EVERY IMAGE, LOW-LIGHT IMAGE AND MASK PATH
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("FILE EXISTENCE CHECK")
print("=" * 60)

for dataset_name, subset in combined_df.groupby("dataset"):
    print(f"\n{dataset_name}: {len(subset)} rows")

    for col in ["original", "low_light", "mask"]:
        exists = subset[col].map(
            lambda p: isinstance(p, str) and os.path.isfile(p)
        )
        print(f"  {col}: {exists.sum()}/{len(subset)} files found")

# Show missing files, without hiding them
missing_records = []

for col in ["original", "low_light", "mask"]:
    exists = combined_df[col].map(
        lambda p: isinstance(p, str) and os.path.isfile(p)
    )

    for idx in combined_df.index[~exists]:
        missing_records.append({
            "dataset": combined_df.at[idx, "dataset"],
            "split": combined_df.at[idx, "split"],
            "filename": combined_df.at[idx, "filename"],
            "missing_type": col,
            "path": combined_df.at[idx, col],
        })

missing_df = pd.DataFrame(missing_records)

if len(missing_df):
    print("\nFIRST 20 MISSING PATHS:")
    display(missing_df.head(20))
else:
    print("\nAll referenced files exist.")

# ------------------------------------------------------------
# 9. DATASET CLASS — SAME PREPROCESSING AS ORIGINAL NOTEBOOK
# ------------------------------------------------------------

class SANCMedicalDataset(Dataset):
    def __init__(self, dataframe, image_size=128):
        self.df = dataframe.reset_index(drop=True).copy()
        self.image_size = image_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        low_light = Image.open(row["low_light"]).convert("RGB")
        original = Image.open(row["original"]).convert("RGB")
        mask = Image.open(row["mask"]).convert("L")

        low_light = low_light.resize(
            (self.image_size, self.image_size),
            Image.Resampling.BILINEAR
        )
        original = original.resize(
            (self.image_size, self.image_size),
            Image.Resampling.BILINEAR
        )
        mask = mask.resize(
            (self.image_size, self.image_size),
            Image.Resampling.NEAREST
        )

        low_light = np.array(
            low_light, dtype=np.float32, copy=True
        ) / 255.0
        original = np.array(
            original, dtype=np.float32, copy=True
        ) / 255.0
        mask = np.array(
            mask, dtype=np.float32, copy=True
        )

        # Preserve the original notebook's mask convention.
        mask = (mask > 0).astype(np.float32)

        low_light = torch.from_numpy(low_light).permute(2, 0, 1)
        original = torch.from_numpy(original).permute(2, 0, 1)
        mask = torch.from_numpy(mask).unsqueeze(0)

        return {
            "low_light": low_light,
            "original": original,
            "mask": mask,
            "dataset": row["dataset"],
            "filename": row["filename"],
            "split": row["split"],
        }

# ------------------------------------------------------------
# 10. BUILD LOADERS FROM VALID PATHS ONLY
# ------------------------------------------------------------

valid_rows = ~combined_df.index.isin(
    missing_df.index if False else []
)

# Check all three paths for each row
for col in ["original", "low_light", "mask"]:
    combined_df[col + "_exists"] = combined_df[col].map(
        lambda p: isinstance(p, str) and os.path.isfile(p)
    )

valid_rows = combined_df[
    combined_df[
        ["original_exists", "low_light_exists", "mask_exists"]
    ].all(axis=1)
].copy()

print("\nValid complete samples:", len(valid_rows), "/", len(combined_df))

train_data = valid_rows[valid_rows["split"] == "train"].copy()
val_data = valid_rows[valid_rows["split"] == "val"].copy()
test_data = valid_rows[valid_rows["split"] == "test"].copy()

BATCH_SIZE = 8

train_dataset = SANCMedicalDataset(train_data)
val_dataset = SANCMedicalDataset(val_data)
test_dataset = SANCMedicalDataset(test_data)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE,
    shuffle=True, num_workers=0
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=0
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=0
)

print("\n" + "=" * 60)
print("FINAL LOADER SUMMARY")
print("=" * 60)
print("Train:", len(train_dataset), "| batches:", len(train_loader))
print("Validation:", len(val_dataset), "| batches:", len(val_loader))
print("Test:", len(test_dataset), "| batches:", len(test_loader))

# ------------------------------------------------------------
# 11. TEST ONE BATCH
# ------------------------------------------------------------

if len(train_dataset) == 0:
    print("\nNo valid training samples. Fix the missing paths first.")
else:
    batch = next(iter(train_loader))

    print("\nBatch tensor shapes:")
    print("Low-light:", tuple(batch["low_light"].shape))
    print("Original:", tuple(batch["original"].shape))
    print("Mask:", tuple(batch["mask"].shape))
    print("Datasets:", batch["dataset"][:min(8, len(batch["dataset"]))])
    print("Filenames:", batch["filename"][:min(8, len(batch["filename"]))])

    # Visualize the first sample in the batch
    fig, ax = plt.subplots(1, 3, figsize=(12, 4))

    ax[0].imshow(batch["original"][0].permute(1, 2, 0).numpy())
    ax[0].set_title("Original")

    ax[1].imshow(batch["low_light"][0].permute(1, 2, 0).numpy())
    ax[1].set_title("Low-light")

    ax[2].imshow(batch["mask"][0, 0].numpy(), cmap="gray", vmin=0, vmax=1)
    ax[2].set_title("Binary mask")

    for a in ax:
        a.axis("off")

    plt.tight_layout()
    plt.show()

print("\nDataset setup and loader diagnostics finished.")


In [ ]:

import os
import torch

checkpoint_path = os.path.join(
    "/content/drive/MyDrive/SANC",
    "models",
    "best_unet_saliency.pth"
)

print("Checkpoint exists:", os.path.isfile(checkpoint_path))

if not os.path.isfile(checkpoint_path):
    raise FileNotFoundError(checkpoint_path)

print("File size (MB):", round(os.path.getsize(checkpoint_path) / 1024**2, 2))

# Weights-only loading is safer for inspecting checkpoint files.
try:
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True
    )
except TypeError:
    # Compatibility fallback for older PyTorch versions.
    checkpoint = torch.load(checkpoint_path, map_location="cpu")

print("Checkpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("Top-level keys:", list(checkpoint.keys())[:30])

    state = checkpoint.get(
        "model_state_dict",
        checkpoint.get("state_dict", checkpoint)
    )

    if isinstance(state, dict):
        print("\nFirst parameter names and shapes:")
        for name, value in list(state.items())[:12]:
            if torch.is_tensor(value):
                print(name, tuple(value.shape))
else:
    print("This checkpoint is not a state-dictionary-style object.")


In [ ]:

import torch
import numpy as np

checkpoint_path = "/content/drive/MyDrive/SANC/models/best_unet_saliency.pth"

checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)

print("=" * 55)
print("SAVED CHECKPOINT METRICS")
print("=" * 55)
for key in ["epoch", "val_dice", "val_iou", "val_loss"]:
    print(f"{key}: {checkpoint.get(key, 'Not saved')}")

print("\n" + "=" * 55)
print("MASK ENCODING CHECK")
print("=" * 55)

batch = next(iter(train_loader))
masks = batch["mask"] if isinstance(batch, dict) else batch[2]

print("Mask shape:", tuple(masks.shape))
print("Mask min/max:", masks.min().item(), masks.max().item())
print("Unique values in first mask (up to 30):",
      torch.unique(masks[0]).cpu().numpy()[:30])
print("Foreground pixels in first mask:",
      (masks[0] > 0.5).sum().item())
print("Total pixels in first mask:", masks[0].numel())

print("\n" + "=" * 55)
print("MODEL VARIABLES")
print("=" * 55)

for name in ["model", "unet", "saliency_model", "best_model"]:
    obj = globals().get(name)
    if isinstance(obj, torch.nn.Module):
        print(f"{name}: {type(obj).__name__}")


In [ ]:

import torch
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_path = "/content/drive/MyDrive/SANC/models/best_unet_saliency.pth"

# Recreate the exact model architecture
model = UNet(in_channels=3, out_channels=1).to(device)

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=True
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Checkpoint loaded successfully.")
print("Device:", device)
print("Saved validation Dice:", checkpoint.get("val_dice"))
print("Saved validation IoU:", checkpoint.get("val_iou"))

# Get one test batch
batch = next(iter(test_loader))

images = batch["low_light"].to(device)
masks = batch["mask"].to(device)

with torch.no_grad():
    logits = model(images)
    probabilities = torch.sigmoid(logits)
    predictions = (probabilities >= 0.5).float()

print("Input shape:", tuple(images.shape))
print("Output shape:", tuple(logits.shape))
print("Prediction probability range:",
      probabilities.min().item(), probabilities.max().item())
print("Predicted foreground fraction:",
      predictions.mean().item())
print("Ground-truth foreground fraction:",
      masks.mean().item())

# Visualize up to 3 samples
n = min(3, images.shape[0])
fig, axes = plt.subplots(n, 4, figsize=(12, 3.5 * n))

if n == 1:
    axes = axes.reshape(1, -1)

for i in range(n):
    axes[i, 0].imshow(images[i].detach().cpu().permute(1, 2, 0).clamp(0, 1))
    axes[i, 0].set_title("Low-light input")

    axes[i, 1].imshow(masks[i, 0].detach().cpu(), cmap="gray", vmin=0, vmax=1)
    axes[i, 1].set_title("Ground-truth mask")

    axes[i, 2].imshow(probabilities[i, 0].detach().cpu(), cmap="viridis", vmin=0, vmax=1)
    axes[i, 2].set_title("Predicted probability")

    axes[i, 3].imshow(predictions[i, 0].detach().cpu(), cmap="gray", vmin=0, vmax=1)
    axes[i, 3].set_title("Thresholded prediction")

    for ax in axes[i]:
        ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

from pathlib import Path
from collections import Counter

paths = []
for _, row in combined_df.iterrows():
    for col in ["original", "low_light", "mask"]:
        paths.append((str(row["dataset"]), col, Path(str(row[col]))))

issues = []
for dataset, kind, path in paths:
    try:
        if not path.exists():
            issues.append((dataset, kind, str(path), "missing"))
        elif path.stat().st_size == 0:
            issues.append((dataset, kind, str(path), "empty"))
    except OSError as e:
        issues.append((dataset, kind, str(path), str(e)))

print("Missing/empty files:", len(issues))
print("Breakdown:", Counter((x[0], x[1], x[3]) for x in issues))
for item in issues[:20]:
    print(item)


In [ ]:

from torch.utils.data import DataLoader, Subset
from collections import defaultdict
import torch

# Find test samples using the existing dataset and split indices
test_df = combined_df[combined_df["split"] == "test"].reset_index(drop=True)

bad_path = (
    "/content/drive/MyDrive/SANC/datasets/kvasir/masks/"
    "cju89y9h0puti0818i5yw29e6.jpg"
)

valid_indices = [
    i for i, row in test_df.iterrows()
    if row["mask"] != bad_path
]

print("Test samples originally:", len(test_df))
print("Test samples evaluated:", len(valid_indices))
print("Excluded:", len(test_df) - len(valid_indices))

# Build a test-only dataset using the same dataset class
clean_test_dataset = SANCMedicalDataset(test_df.iloc[valid_indices].reset_index(drop=True))
clean_test_loader = DataLoader(
    clean_test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0
)

stats = defaultdict(lambda: {"n": 0, "dice": 0.0, "iou": 0.0})
model.eval()

with torch.inference_mode():
    for batch_idx, batch in enumerate(clean_test_loader):
        x = batch["low_light"].to(device)
        y = (batch["mask"].to(device) >= 0.5).float()
        names = batch["dataset"]

        pred = (torch.sigmoid(model(x)) >= 0.5).float()

        dims = (1, 2, 3)
        inter = (pred * y).sum(dim=dims)
        p_sum = pred.sum(dim=dims)
        y_sum = y.sum(dim=dims)
        union = p_sum + y_sum - inter

        dice = (2 * inter + 1e-7) / (p_sum + y_sum + 1e-7)
        iou = (inter + 1e-7) / (union + 1e-7)

        for name, d, j in zip(names, dice.cpu().tolist(), iou.cpu().tolist()):
            stats[str(name)]["n"] += 1
            stats[str(name)]["dice"] += d
            stats[str(name)]["iou"] += j

for name, s in sorted(stats.items()):
    print(
        f"{name}: N={s['n']}, "
        f"Mean Dice={s['dice']/s['n']:.4f}, "
        f"Mean IoU={s['iou']/s['n']:.4f}"
    )


In [ ]:

import os
import pandas as pd

save_dir = "/content/drive/MyDrive/SANC/results"
os.makedirs(save_dir, exist_ok=True)

results_df = pd.DataFrame([
    {"Dataset": "HAM10000", "Images_Evaluated": 130,
     "Mean_Dice": 0.8891, "Mean_IoU": 0.8180},
    {"Dataset": "Kvasir-SEG", "Images_Evaluated": 149,
     "Mean_Dice": 0.7565, "Mean_IoU": 0.6423},
    {"Dataset": "ORIGA", "Images_Evaluated": 130,
     "Mean_Dice": 0.9368, "Mean_IoU": 0.8848},
])

csv_path = os.path.join(save_dir, "saliency_test_results_diagnostic.csv")
results_df.to_csv(csv_path, index=False)

print("Results saved to:", csv_path)
print(results_df.to_string(index=False))


In [ ]:

import shutil

source = "/content/drive/MyDrive/SANC/models/best_unet_saliency.pth"
backup = "/content/drive/MyDrive/SANC/models/best_unet_saliency_backup.pth"

if not os.path.exists(backup):
    shutil.copy2(source, backup)
    print("Checkpoint backup saved:", backup)
else:
    print("Backup already exists:", backup)


In [ ]:

from pathlib import Path

root = Path("/content/drive/MyDrive")
image_id = "cju89y9h0puti0818i5yw29e6"

print("Searching for files with matching image ID...")
matches = []

for p in root.rglob("*"):
    if p.is_file() and image_id in p.name and "mask" in str(p.parent).lower():
        try:
            size = p.stat().st_size
            if size > 0:
                matches.append((str(p), size))
        except OSError:
            pass

if matches:
    for path, size in matches:
        print(f"{size} bytes | {path}")
else:
    print("No non-empty mask candidate found.")


In [ ]:

import zipfile
from pathlib import Path

base = Path("/content/drive/MyDrive/SANC/datasets/kvasir")
target = "cju89y9h0puti0818i5yw29e6"

for zip_path in [
    base / "kvasir-seg.zip",
    base / "Kvasir_SEG_processed.zip",
]:
    print(f"\nArchive: {zip_path.name}")

    if not zip_path.exists():
        print("Archive not found.")
        continue

    try:
        with zipfile.ZipFile(zip_path, "r") as z:
            matches = [
                name for name in z.namelist()
                if target in Path(name).name
            ]

            if matches:
                for name in matches:
                    info = z.getinfo(name)
                    print(f"FOUND: {name} ({info.file_size} bytes)")
            else:
                print("No matching filename in this archive.")
    except Exception as e:
        print("Could not inspect archive:", repr(e))


In [ ]:

def high_frequency_ratio(x):
    H, W = x.shape[-2:]

    spectrum = torch.fft.fft2(x, dim=(-2, -1))
    power = spectrum.abs().square()

    fy = torch.fft.fftfreq(H, device=x.device)
    fx = torch.fft.fftfreq(W, device=x.device)
    yy, xx = torch.meshgrid(fy, fx, indexing="ij")
    radius = torch.sqrt(xx**2 + yy**2)

    hf_mask = radius > 0.60 * radius.max()

    # Sum over spatial dimensions, preserving batch and channels
    hf_energy = (power * hf_mask[None, None, :, :]).sum(
        dim=(-2, -1)
    )
    total_energy = power.sum(dim=(-2, -1))

    ratio = hf_energy / (total_energy + 1e-8)
    return ratio.mean().item()


In [ ]:
for name, output in [
    ("Low saliency", out_low),
    ("High saliency", out_high),
    ("Mixed saliency", out_mixed),
]:
    print(
        f"{name:16s} | "
        f"mean={output.mean().item():.5f} | "
        f"std={output.std().item():.5f} | "
        f"HF ratio={high_frequency_ratio(output):.5f} | "
        f"finite={torch.isfinite(output).all().item()}"
    )

print(
    "Low vs high outputs differ:",
    not torch.allclose(out_low, out_high)
)

In [ ]:

# STEP 6 — Reverse U-Net shape and numerical test
reverse_model.eval()

B, H, W = 2, 128, 128

x_t = torch.randn(B, 3, H, W, device=device)
low_light = torch.randn(B, 3, H, W, device=device)
saliency = torch.rand(B, 1, H, W, device=device)
t = torch.randint(0, T, (B,), device=device)

with torch.no_grad():
    predicted_noise = reverse_model(
        x_t, low_light, saliency, t
    )

print("Output shape:", tuple(predicted_noise.shape))
print("Expected shape:", tuple(x_t.shape))
print("All values finite:", torch.isfinite(predicted_noise).all().item())
print("Output mean:", predicted_noise.mean().item())
print("Output std:", predicted_noise.std().item())


In [ ]:

@torch.no_grad()
def reverse_diffusion_sample(low_light, reverse_model, saliency_model):
    reverse_model.eval()
    saliency_model.eval()

    batch_size = low_light.shape[0]
    device = low_light.device

    # Estimate the structural saliency condition
    saliency = saliency_model(low_light)

    # Start from Gaussian noise
    x = torch.randn_like(low_light)

    for step in reversed(range(T)):
        t = torch.full(
            (batch_size,), step, device=device, dtype=torch.long
        )

        beta_t = betas[step]
        alpha_t = alphas[step]
        alpha_bar_t = alpha_bars[step]

        # Predict the noise at the current diffusion step
        predicted_noise = reverse_model(x, low_light, saliency, t)

        # Standard DDPM reverse mean (approximation for SANC)
        mean = (
            x - (beta_t / torch.sqrt(1.0 - alpha_bar_t))
            * predicted_noise
        ) / torch.sqrt(alpha_t)

        if step > 0:
            # Standard Gaussian reverse-step noise
            x = mean + torch.sqrt(beta_t) * torch.randn_like(x)
        else:
            x = mean

    return x


In [ ]:

batch = next(iter(val_loader))
low_light = batch["low_light"].to(device)

enhanced = reverse_diffusion_sample(
    low_light, reverse_model, saliency_model
)

print("Input shape:", tuple(low_light.shape))
print("Output shape:", tuple(enhanced.shape))
print("Output finite:", torch.isfinite(enhanced).all().item())
print("Output range:", enhanced.min().item(), enhanced.max().item())


In [ ]:

import torch
import os

checkpoint_path = "/content/drive/MyDrive/SANC/models/reverse_diffusion_model.pth"

checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

print("Checkpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("Checkpoint keys:", list(checkpoint.keys())[:20])

    if "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    elif "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
    else:
        # Check whether the dictionary itself contains model weights
        state_dict = checkpoint

    if isinstance(state_dict, dict) and any(
        torch.is_tensor(v) for v in state_dict.values()
    ):
        try:
            result = reverse_model.load_state_dict(state_dict, strict=True)
            print("Checkpoint loaded successfully:", result)
        except Exception as e:
            print("Could not load directly:", str(e)[:1000])
    else:
        print("No model state dictionary found under the expected keys.")
else:
    print("Checkpoint is not a dictionary; inspect its format before loading.")


In [ ]:

# The trained checkpoint is now loaded.
# Run inference on the same validation batch.

reverse_model.eval()
saliency_model.eval()

with torch.no_grad():
    enhanced = reverse_diffusion_sample(
        low_light, reverse_model, saliency_model
    )

print("Output finite:", torch.isfinite(enhanced).all().item())
print("Output min:", enhanced.min().item())
print("Output max:", enhanced.max().item())
print("Output mean:", enhanced.mean().item())
print("Output std:", enhanced.std().item())


In [ ]:

import matplotlib.pyplot as plt

idx = 0

input_img = low_light[idx].detach().cpu().permute(1, 2, 0).numpy()
output_img = enhanced[idx].detach().cpu().permute(1, 2, 0).numpy()

# Clamp only for visualization, not for model evaluation
output_display = output_img.clip(0, 1)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))

axes[0].imshow(input_img.clip(0, 1))
axes[0].set_title("Low-light Input")
axes[0].axis("off")

axes[1].imshow(output_display)
axes[1].set_title("SANC Enhanced Output")
axes[1].axis("off")

plt.tight_layout()
plt.show()


In [ ]:

checkpoint = torch.load(
    "/content/drive/MyDrive/SANC/models/reverse_diffusion_model.pth",
    map_location="cpu",
    weights_only=False
)

print("Epochs recorded:", checkpoint.get("num_epochs", "Not recorded"))
print("Checkpoint keys:", list(checkpoint.keys()))


In [ ]:

reverse_model.eval()
saliency_model.eval()

with torch.no_grad():
    saliency = saliency_model(low_light)

    # Reproduce the first reverse step from the current sampler
    x = torch.randn_like(low_light)
    t = torch.zeros(low_light.size(0), device=device, dtype=torch.long)

    predicted_noise = reverse_model(x, low_light, saliency, t)

    beta_t = betas[0]
    alpha_t = alphas[0]
    alpha_bar_t = alpha_bars[0]

    mean = (
        x - (beta_t / torch.sqrt(1.0 - alpha_bar_t)) * predicted_noise
    ) / torch.sqrt(alpha_t)

    print("x std:", x.std().item())
    print("Predicted noise std:", predicted_noise.std().item())
    print("Reverse mean std:", mean.std().item())
    print("Reverse mean range:", mean.min().item(), mean.max().item())
    print("All finite:", torch.isfinite(mean).all().item())


In [ ]:
print("Number of diffusion steps:", T)
print("First alpha_bar:", alpha_bars[0].item())
print("Final alpha_bar:", alpha_bars[-1].item())
print("Final signal coefficient:", torch.sqrt(alpha_bars[-1]).item())
print(
    "Final noise coefficient:",
    torch.sqrt(1.0 - alpha_bars[-1]).item()
)

In [ ]:
print("T:", T)
print("betas shape:", betas.shape)
print("alphas shape:", alphas.shape)
print("alpha_bars shape:", alpha_bars.shape)

print("\nSchedule values:")
for i in [0, 1, T // 2, T - 2, T - 1]:
    print(
        f"t={i:3d} | "
        f"beta={betas[i].item():.6f} | "
        f"alpha={alphas[i].item():.6f} | "
        f"alpha_bar={alpha_bars[i].item():.6f}"
    )

print("\nConsistency checks:")
print(
    "alpha = 1 - beta:",
    torch.allclose(alphas, 1.0 - betas, atol=1e-6)
)
print(
    "alpha_bar is cumulative product:",
    torch.allclose(
        alpha_bars,
        torch.cumprod(alphas, dim=0),
        atol=1e-5
    )
)


In [ ]:
reverse_model.eval()
saliency_model.eval()

batch = next(iter(val_loader))
low_light = batch["low_light"].to(device)

with torch.no_grad():
    saliency = saliency_model(low_light)
    x = torch.randn_like(low_light)

    print("Starting from Gaussian noise:")
    print(f"std={x.std().item():.4f}")

    checkpoints = {99, 75, 50, 25, 10, 1, 0}

    for step in reversed(range(T)):
        t = torch.full(
            (low_light.size(0),),
            step,
            device=device,
            dtype=torch.long
        )

        predicted_noise = reverse_model(
            x, low_light, saliency, t
        )

        beta_t = betas[step]
        alpha_t = alphas[step]
        alpha_bar_t = alpha_bars[step]

        mean = (
            x - (beta_t / torch.sqrt(1.0 - alpha_bar_t))
            * predicted_noise
        ) / torch.sqrt(alpha_t)

        if step > 0:
            x = mean + torch.sqrt(beta_t) * torch.randn_like(x)
        else:
            x = mean

        if step in checkpoints:
            print(
                f"After step {step:2d}: "
                f"mean={x.mean().item():.4f}, "
                f"std={x.std().item():.4f}, "
                f"min={x.min().item():.3f}, "
                f"max={x.max().item():.3f}, "
                f"finite={torch.isfinite(x).all().item()}"
            )

In [ ]:
reverse_model.eval()
saliency_model.eval()

batch = next(iter(val_loader))
low_light = batch["low_light"].to(device)
original = batch["original"].to(device)

with torch.no_grad():
    enhanced = reverse_diffusion_sample(
        low_light,
        reverse_model,
        saliency_model
    )

print("Low-light input:")
print(
    f"min={low_light.min().item():.4f}, "
    f"max={low_light.max().item():.4f}, "
    f"mean={low_light.mean().item():.4f}, "
    f"std={low_light.std().item():.4f}"
)

print("\nOriginal target:")
print(
    f"min={original.min().item():.4f}, "
    f"max={original.max().item():.4f}, "
    f"mean={original.mean().item():.4f}, "
    f"std={original.std().item():.4f}"
)

print("\nEnhanced output before clipping:")
print(
    f"min={enhanced.min().item():.4f}, "
    f"max={enhanced.max().item():.4f}, "
    f"mean={enhanced.mean().item():.4f}, "
    f"std={enhanced.std().item():.4f}"
)

print(
    "\nFraction outside [0, 1]:",
    ((enhanced < 0) | (enhanced > 1)).float().mean().item()
)

In [ ]:
reverse_model.eval()
saliency_model.eval()

batch = next(iter(val_loader))
low_light = batch["low_light"].to(device)
original = batch["original"].to(device)

with torch.no_grad():
    saliency = saliency_model(low_light)

    noise = torch.randn_like(original)
    colored_noise = sanc_noise_coloring(noise, saliency)

    # Test a middle timestep
    t = torch.full(
        (original.size(0),),
        T // 2,
        device=device,
        dtype=torch.long
    )

    x_t = q_sample(original, t, colored_noise)

    predicted_noise = reverse_model(
        x_t, low_light, saliency, t
    )

    mse = F.mse_loss(predicted_noise, colored_noise)

    print("Original range:",
          original.min().item(), original.max().item())
    print("Low-light range:",
          low_light.min().item(), low_light.max().item())
    print("Colored noise std:", colored_noise.std().item())
    print("Predicted noise std:", predicted_noise.std().item())
    print("Noise prediction MSE:", mse.item())
    print("Predictions finite:",
          torch.isfinite(predicted_noise).all().item())

In [ ]:
print("Checking DDPM posterior variance...")

alpha_bar_prev = torch.cat([
    torch.ones_like(alpha_bars[:1]),
    alpha_bars[:-1]
])

posterior_variance = (
    betas * (1.0 - alpha_bar_prev)
    / (1.0 - alpha_bars)
)

print("Posterior variance at t=0:", posterior_variance[0].item())
print("Posterior variance at t=1:", posterior_variance[1].item())
print("Posterior variance at t=50:", posterior_variance[50].item())
print("Posterior variance at t=99:", posterior_variance[99].item())

print(
    "All finite:",
    torch.isfinite(posterior_variance).all().item()
)

In [ ]:
# Build posterior variance from the existing schedule
alpha_bar_prev = torch.cat([
    torch.ones_like(alpha_bars[:1]),
    alpha_bars[:-1]
])

posterior_variance = (
    betas * (1.0 - alpha_bar_prev)
    / (1.0 - alpha_bars)
)

@torch.no_grad()
def reverse_diffusion_sample_v2(low_light, reverse_model, saliency_model):
    reverse_model.eval()
    saliency_model.eval()

    batch_size = low_light.shape[0]
    saliency = saliency_model(low_light)
    x = torch.randn_like(low_light)

    for step in reversed(range(T)):
        t = torch.full(
            (batch_size,), step,
            device=low_light.device,
            dtype=torch.long
        )

        beta_t = betas[step]
        alpha_t = alphas[step]
        alpha_bar_t = alpha_bars[step]

        predicted_noise = reverse_model(
            x, low_light, saliency, t
        )

        mean = (
            x - (beta_t / torch.sqrt(1.0 - alpha_bar_t))
            * predicted_noise
        ) / torch.sqrt(alpha_t)

        if step > 0:
            x = mean + torch.sqrt(
                posterior_variance[step]
            ) * torch.randn_like(x)
        else:
            x = mean

    return x

In [ ]:
batch = next(iter(val_loader))
low_light = batch["low_light"].to(device)
original = batch["original"].to(device)

# Run the revised sampler
enhanced_v2 = reverse_diffusion_sample_v2(
    low_light, reverse_model, saliency_model
)

print("Target mean/std:",
      original.mean().item(), original.std().item())
print("Revised output mean/std:",
      enhanced_v2.mean().item(), enhanced_v2.std().item())
print("Revised output range:",
      enhanced_v2.min().item(), enhanced_v2.max().item())
print("All finite:",
      torch.isfinite(enhanced_v2).all().item())

In [ ]:
checkpoint_path = (
    "/content/drive/MyDrive/SANC/models/reverse_diffusion_model.pth"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint keys:", checkpoint.keys())
print("Saved epochs:", checkpoint.get("num_epochs", "Not recorded"))
print("Last recorded training loss:", checkpoint.get("loss", "Not recorded"))


In [ ]:
@torch.no_grad()
def reverse_sample_from_start(
    low_light, reverse_model, saliency_model, start_from_input=False
):
    reverse_model.eval()
    saliency_model.eval()

    batch_size = low_light.shape[0]
    saliency = saliency_model(low_light)

    if start_from_input:
        x = low_light.clone()
    else:
        x = torch.randn_like(low_light)

    for step in reversed(range(T)):
        t = torch.full(
            (batch_size,), step,
            device=low_light.device,
            dtype=torch.long
        )

        predicted_noise = reverse_model(
            x, low_light, saliency, t
        )

        beta_t = betas[step]
        alpha_t = alphas[step]
        alpha_bar_t = alpha_bars[step]

        mean = (
            x - beta_t / torch.sqrt(1.0 - alpha_bar_t)
            * predicted_noise
        ) / torch.sqrt(alpha_t)

        if step > 0:
            x = mean + torch.sqrt(
                posterior_variance[step]
            ) * torch.randn_like(x)
        else:
            x = mean

    return x


batch = next(iter(val_loader))
low_light = batch["low_light"].to(device)
original = batch["original"].to(device)

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

random_start_output = reverse_sample_from_start(
    low_light, reverse_model, saliency_model,
    start_from_input=False
)

input_start_output = reverse_sample_from_start(
    low_light, reverse_model, saliency_model,
    start_from_input=True
)

for name, image in [
    ("Random start", random_start_output),
    ("Low-light start", input_start_output),
]:
    print(
        f"{name}: mean={image.mean().item():.4f}, "
        f"std={image.std().item():.4f}, "
        f"range=[{image.min().item():.3f}, "
        f"{image.max().item():.3f}]"
    )

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:

# MEMBER 1 — DIFFUSION SCHEDULE AUDIT
# Run after the existing schedule cell.

print("T =", T)
print("betas shape:", tuple(betas.shape))
print("alphas shape:", tuple(alphas.shape))
print("alpha_bars shape:", tuple(alpha_bars.shape))

print("\nSelected timesteps:")
for i in sorted(set([0, 1, T // 2, T - 2, T - 1])):
    print(
        f"t={i:3d} | "
        f"beta={betas[i].item():.6f} | "
        f"alpha={alphas[i].item():.6f} | "
        f"alpha_bar={alpha_bars[i].item():.6f}"
    )

# Check schedule consistency
print("\nConsistency checks:")
print("All betas positive:", (betas > 0).all().item())
print("All betas below 1:", (betas < 1).all().item())

print(
    "alpha = 1 - beta:",
    torch.allclose(alphas, 1 - betas, atol=1e-6)
)

print(
    "alpha_bar = cumulative product:",
    torch.allclose(
        alpha_bars,
        torch.cumprod(alphas, dim=0),
        atol=1e-6
    )
)

print("\nTerminal distribution diagnostics:")
print("alpha_bar_T:", alpha_bars[-1].item())
print("Signal coefficient:",
      torch.sqrt(alpha_bars[-1]).item())
print("Noise coefficient:",
      torch.sqrt(1 - alpha_bars[-1]).item())


In [ ]:

# MEMBER 1 — SANC NUMERICAL AUDIT
# Requires the existing `sanc` module.

torch.manual_seed(42)

B, C, H, W = 4, 3, 128, 128

noise = torch.randn(B, C, H, W, device=device)

saliency_low = torch.zeros(B, 1, H, W, device=device)
saliency_high = torch.ones(B, 1, H, W, device=device)

with torch.no_grad():
    colored_low = sanc(noise, saliency_low)
    colored_high = sanc(noise, saliency_high)

for name, z in [
    ("Gaussian input", noise),
    ("SANC, low saliency", colored_low),
    ("SANC, high saliency", colored_high),
]:
    print(
        f"{name:24s} | "
        f"mean={z.mean().item():.5f} | "
        f"std={z.std().item():.5f} | "
        f"finite={torch.isfinite(z).all().item()}"
    )

print(
    "\nLow/high saliency outputs differ:",
    not torch.allclose(colored_low, colored_high)
)


In [ ]:

# MEMBER 1 — SCHEDULE AUDIT

print("T =", T)
print("betas shape:", tuple(betas.shape))
print("alphas shape:", tuple(alphas.shape))
print("alpha_bars shape:", tuple(alpha_bars.shape))

print("\nSelected timesteps:")
for i in sorted(set([0, 1, T // 2, T - 2, T - 1])):
    print(
        f"t={i:3d} | "
        f"beta={betas[i].item():.6f} | "
        f"alpha={alphas[i].item():.6f} | "
        f"alpha_bar={alpha_bars[i].item():.6f}"
    )

print("\nConsistency:")
print("Betas valid:",
      ((betas > 0) & (betas < 1)).all().item())
print("alpha = 1-beta:",
      torch.allclose(alphas, 1-betas, atol=1e-6))
print("alpha_bar cumulative product:",
      torch.allclose(
          alpha_bars, torch.cumprod(alphas, dim=0),
          atol=1e-6
      ))

print("\nTerminal coefficients:")
print("Final alpha_bar:", alpha_bars[-1].item())
print("Signal coefficient:",
      torch.sqrt(alpha_bars[-1]).item())
print("Noise coefficient:",
      torch.sqrt(1-alpha_bars[-1]).item())


In [ ]:
# STEP 1: Check which combined DataFrame exists

for name in ["combined_df", "combined_all_df"]:
    obj = globals().get(name)

    if obj is not None:
        print(f"\n{name} exists")
        print("Shape:", obj.shape)
        print("Columns:", obj.columns.tolist())
    else:
        print(f"{name} does not exist")

In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/SANC/datasets")

paths = {
    "HAM metadata": ROOT / "ham/HAM preprocessing/metadata.csv",
    "ORIGA metadata": ROOT / "ORIGA_processed/integration/ORIGA_common_metadata.csv",
    "Kvasir train split": ROOT / "metadata/kvasir_train.csv",
    "Kvasir validation split": ROOT / "metadata/kvasir_val.csv",
    "Kvasir test split": ROOT / "metadata/kvasir_test.csv",
}

for name, path in paths.items():
    print(f"{name}: {'FOUND' if path.is_file() else 'MISSING'}")
    print(f"  {path}")

In [ ]:

import pandas as pd
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/SANC/datasets")

# --------------------------------------------------
# 1. Load HAM10000 metadata
# --------------------------------------------------
ham_root = ROOT / "ham" / "HAM preprocessing"
ham = pd.read_csv(ham_root / "metadata.csv")

HAM_COMMON = ham.rename(columns={
    "image_id": "filename",
    "image_path": "original",
    "low_light_path": "low_light",
    "mask_path": "mask"
}).copy()

HAM_COMMON["dataset"] = "HAM10000"

# --------------------------------------------------
# 2. Load ORIGA metadata
# --------------------------------------------------
origa_root = ROOT / "ORIGA_processed" / "integration"
origa = pd.read_csv(origa_root / "ORIGA_common_metadata.csv")

origa_common_df = origa.rename(columns={
    "filename": "filename",
    "original": "original",
    "low_light": "low_light",
    "mask": "mask",
    "split": "split"
}).copy()

origa_common_df["dataset"] = "ORIGA"

# Resolve stale paths using the actual integration folders
for col, folder in [
    ("original", "images"),
    ("low_light", "low_light"),
    ("mask", "masks")
]:
    origa_common_df[col] = origa_common_df["filename"].apply(
        lambda f: str(origa_root / folder / Path(str(f)).name)
    )

# --------------------------------------------------
# 3. Load Kvasir train/validation/test split CSVs
# --------------------------------------------------
kvasir_root = ROOT / "kvasir"
metadata_root = ROOT / "metadata"

def load_kvasir_split(csv_name, split_name):
    df = pd.read_csv(metadata_root / csv_name).copy()

    # Find the filename column
    candidates = ["filename", "image", "image_id", "id"]
    filename_col = next(
        (c for c in candidates if c in df.columns), None
    )
    if filename_col is None:
        raise ValueError(
            f"No filename column in {csv_name}. "
            f"Columns: {df.columns.tolist()}"
        )

    df["filename"] = df[filename_col].astype(str).apply(
        lambda x: Path(x).name
    )
    df["original"] = df["filename"].apply(
        lambda x: str(kvasir_root / "original" / x)
    )
    df["low_light"] = df["filename"].apply(
        lambda x: str(kvasir_root / "low_light" / x)
    )
    df["mask"] = df["filename"].apply(
        lambda x: str(kvasir_root / "masks" / x)
    )
    df["dataset"] = "Kvasir-SEG"
    df["split"] = split_name

    return df[[
        "filename", "original", "low_light",
        "mask", "dataset", "split"
    ]]

kvasir_train_df = load_kvasir_split("kvasir_train.csv", "train")
kvasir_val_df = load_kvasir_split("kvasir_val.csv", "val")
kvasir_test_df = load_kvasir_split("kvasir_test.csv", "test")

kvasir_common_df = pd.concat(
    [kvasir_train_df, kvasir_val_df, kvasir_test_df],
    ignore_index=True
)

# Standardize columns for all datasets
required_cols = [
    "filename", "original", "low_light",
    "mask", "dataset", "split"
]

for name, df in [
    ("HAM", HAM_COMMON),
    ("ORIGA", origa_common_df),
    ("Kvasir", kvasir_common_df)
]:
    missing = set(required_cols) - set(df.columns)
    if missing:
        raise ValueError(f"{name} metadata missing columns: {missing}")

    # Keep only the common fields
    if name == "HAM":
        HAM_COMMON = df[required_cols].copy()
    elif name == "ORIGA":
        origa_common_df = df[required_cols].copy()
    else:
        kvasir_common_df = df[required_cols].copy()

# --------------------------------------------------
# 4. Combine metadata
# --------------------------------------------------
combined_all_df = pd.concat(
    [kvasir_common_df, origa_common_df, HAM_COMMON],
    ignore_index=True
)

print("Total samples:", len(combined_all_df))
print("\nDataset counts:")
print(combined_all_df["dataset"].value_counts())

print("\nDataset and split counts:")
print(combined_all_df.groupby(["dataset", "split"]).size())

# --------------------------------------------------
# 5. Inspect test counts and corrupted mask record
# --------------------------------------------------
test_df = combined_all_df[
    combined_all_df["split"].astype(str).str.strip().str.lower()
    == "test"
].reset_index(drop=True)

bad_filename = "cju89y9h0puti0818i5yw29e6.jpg"

bad_rows = test_df[
    test_df["filename"].astype(str).str.endswith(bad_filename)
]

valid_test_df = test_df[
    ~test_df["filename"].astype(str).str.endswith(bad_filename)
].copy()

print("\nOfficial test samples:", len(test_df))
print("Corrupted-mask record found:", len(bad_rows))
print("Samples after excluding that record:", len(valid_test_df))
print("Excluded:", len(test_df) - len(valid_test_df))

print("\nTest samples after exclusion, by dataset:")
print(valid_test_df["dataset"].value_counts())


In [ ]:
from pathlib import Path

model_dir = Path("/content/drive/MyDrive/SANC/models")

print("Model directory exists:", model_dir.exists())

if model_dir.exists():
    for p in model_dir.iterdir():
        if p.is_file():
            print(p.name, f"({p.stat().st_size:,} bytes)")

In [ ]:
import torch
from pathlib import Path

checkpoint_path = Path(
    "/content/drive/MyDrive/SANC/models/best_unet_saliency.pth"
)

print("Checkpoint exists:", checkpoint_path.is_file())

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

print("\nCheckpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("Checkpoint keys:", list(checkpoint.keys()))

    for key, value in checkpoint.items():
        if isinstance(value, (int, float, str, bool)):
            print(f"{key}: {value}")
        elif isinstance(value, dict):
            print(f"{key}: dictionary with {len(value)} entries")
        elif torch.is_tensor(value):
            print(f"{key}: tensor, shape={tuple(value.shape)}")

In [ ]:
print("UNet defined:", "UNet" in globals())
print("torch available:", "torch" in globals())

In [ ]:
# ============================================
# U-NET FOR SALIENCY ESTIMATION
# ============================================

import torch
import torch.nn as nn


class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):

    def __init__(self, in_channels=3, out_channels=1):
        super().__init__()

        # -------------------------
        # Encoder
        # -------------------------
        self.enc1 = DoubleConv(3, 64)
        self.enc2 = DoubleConv(64, 128)
        self.enc3 = DoubleConv(128, 256)
        self.enc4 = DoubleConv(256, 512)

        self.pool = nn.MaxPool2d(
            kernel_size=2,
            stride=2
        )

        # -------------------------
        # Bottleneck
        # -------------------------
        self.bottleneck = DoubleConv(
            512,
            1024
        )

        # -------------------------
        # Decoder
        # -------------------------
        self.up4 = nn.ConvTranspose2d(
            1024,
            512,
            kernel_size=2,
            stride=2
        )

        self.dec4 = DoubleConv(
            1024,
            512
        )

        self.up3 = nn.ConvTranspose2d(
            512,
            256,
            kernel_size=2,
            stride=2
        )

        self.dec3 = DoubleConv(
            512,
            256
        )

        self.up2 = nn.ConvTranspose2d(
            256,
            128,
            kernel_size=2,
            stride=2
        )

        self.dec2 = DoubleConv(
            256,
            128
        )

        self.up1 = nn.ConvTranspose2d(
            128,
            64,
            kernel_size=2,
            stride=2
        )

        self.dec1 = DoubleConv(
            128,
            64
        )

        # -------------------------
        # Output
        # -------------------------
        self.output = nn.Conv2d(
            64,
            out_channels,
            kernel_size=1
        )

    def forward(self, x):

        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        # Bottleneck
        b = self.bottleneck(
            self.pool(e4)
        )

        # Decoder
        d4 = self.up4(b)
        d4 = torch.cat([d4, e4], dim=1)
        d4 = self.dec4(d4)

        d3 = self.up3(d4)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.output(d1)


print("U-Net defined successfully.")

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_path = (
    "/content/drive/MyDrive/SANC/models/best_unet_saliency.pth"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=True
)

model = UNet(in_channels=3, out_channels=1).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Model restored successfully.")
print("Device:", device)
print("Checkpoint epoch:", checkpoint["epoch"])
print("Validation Dice:", round(checkpoint["val_dice"], 4))
print("Validation IoU:", round(checkpoint["val_iou"], 4))

In [ ]:
print("UNet available:", "UNet" in globals())
print("Model available:", "model" in globals())
print("Device:", device)
print("Model mode:", "eval" if not model.training else "train")

In [ ]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import numpy as np


class SANCMedicalDataset(Dataset):

    def __init__(self, dataframe):

        self.df = dataframe.reset_index(drop=True)

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        # Load original image
        original = Image.open(
            row["original"]
        ).convert("RGB")

        # Load low-light image
        low_light = Image.open(
            row["low_light"]
        ).convert("RGB")

        # Load binary mask
        mask = Image.open(
            row["mask"]
        ).convert("L")

        # Convert images to numpy arrays
        original = np.array(original)
        low_light = np.array(low_light)
        mask = np.array(mask)

        # Convert to PyTorch tensors
        original = torch.from_numpy(
            original
        ).permute(2, 0, 1).float() / 255.0

        low_light = torch.from_numpy(
            low_light
        ).permute(2, 0, 1).float() / 255.0

        mask = torch.from_numpy(
            mask
        ).float()

        # Convert mask to binary 0/1
        mask = (mask > 0).float()

        # Add channel dimension to mask
        mask = mask.unsqueeze(0)

        return {
            "original": original,
            "low_light": low_light,
            "mask": mask,
            "dataset": row["dataset"],
            "filename": row["filename"],
            "split": row["split"]
        }

In [ ]:
from pathlib import Path

bad_filename = "cju89y9h0puti0818i5yw29e6.jpg"

test_df = combined_all_df[
    combined_all_df["split"].astype(str).str.strip().str.lower() == "test"
].reset_index(drop=True)

valid_test_df = test_df[
    ~test_df["filename"].astype(str).str.endswith(bad_filename)
].reset_index(drop=True)

print("Official test samples:", len(test_df))
print("Evaluation samples:", len(valid_test_df))
print("\nSamples per dataset:")
print(valid_test_df["dataset"].value_counts())

missing = []
for _, row in valid_test_df.iterrows():
    for col in ["original", "low_light", "mask"]:
        if not Path(str(row[col])).is_file():
            missing.append((row["dataset"], row["filename"], col, row[col]))

print("\nMissing files:", len(missing))
for item in missing[:20]:
    print(item)

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/SANC/datasets")

checks = {
    "HAM metadata": ROOT / "ham/HAM preprocessing/metadata.csv",
    "HAM images": ROOT / "ham/HAM preprocessing/images/images",
    "HAM low_light": ROOT / "ham/HAM preprocessing/low_light/low_light",
    "HAM masks": ROOT / "ham/HAM preprocessing/masks/masks",
    "ORIGA images": ROOT / "ORIGA_processed/integration/images",
    "ORIGA low_light": ROOT / "ORIGA_processed/integration/low_light",
    "ORIGA masks": ROOT / "ORIGA_processed/integration/masks",
    "Kvasir original": ROOT / "kvasir/original",
    "Kvasir low_light": ROOT / "kvasir/low_light",
    "Kvasir masks": ROOT / "kvasir/masks",
}

for name, path in checks.items():
    count = sum(1 for p in path.iterdir() if p.is_file()) if path.is_dir() else 0
    print(f"{name}: exists={path.exists()}, files={count}")

In [ ]:
print("HAM metadata paths:")
print(HAM_COMMON[["filename", "original", "low_light", "mask"]].head(3).to_string(index=False))

print("\nORIGA metadata paths:")
print(origa_common_df[["filename", "original", "low_light", "mask"]].head(3).to_string(index=False))

print("\nKvasir metadata paths:")
print(kvasir_common_df[["filename", "original", "low_light", "mask"]].head(3).to_string(index=False))

In [ ]:
from pathlib import Path

root = Path("/content/drive/MyDrive/SANC")

for folder in ["models", "results", "datasets"]:
    path = root / folder
    print(f"\n{folder}: {'FOUND' if path.exists() else 'NOT FOUND'}")
    if path.exists():
        for item in sorted(path.iterdir()):
            print(" ", item.name)

In [ ]:
import torch
from pathlib import Path

model_dir = Path("/content/drive/MyDrive/SANC/models")

files = [
    "reverse_diffusion_model.pth",
    "best_unet_saliency.pth",
    "best_attention_unet.pth",
    "best_deeplab_saliency.pth",
    "best_unet_bce_dice.pth",
    "best_unetpp_saliency.pth",
]

for name in files:
    path = model_dir / name
    print(f"\n--- {name} ---")

    if not path.exists():
        print("Missing")
        continue

    try:
        checkpoint = torch.load(
            path, map_location="cpu", weights_only=False
        )

        if isinstance(checkpoint, dict):
            print("Checkpoint keys:", list(checkpoint.keys()))

            state = checkpoint.get(
                "model_state_dict",
                checkpoint.get("state_dict", checkpoint)
            )

            if isinstance(state, dict):
                print("Number of parameter entries:", len(state))
                for key, value in list(state.items())[:5]:
                    if torch.is_tensor(value):
                        print(f"  {key}: {tuple(value.shape)}")
        else:
            print("Loaded object type:", type(checkpoint))

    except Exception as e:
        print("Load error:", str(e))

In [ ]:
import inspect
import torch

print("=== DIFFUSION SCHEDULE ===")
for name in ["T", "beta_start", "beta_end"]:
    print(f"{name}:", globals().get(name, "NOT DEFINED"))

for name in ["betas", "alphas", "alpha_bars"]:
    value = globals().get(name)
    if torch.is_tensor(value):
        print(
            f"{name}: shape={tuple(value.shape)}, "
            f"first={value[0].item():.6f}, "
            f"last={value[-1].item():.6f}"
        )
    else:
        print(f"{name}: NOT DEFINED")

print("\n=== MODEL AND DIFFUSION FUNCTIONS ===")
for name in [
    "DualBranchReverseUNet",
    "sanc",
    "sanc_noise_coloring",
    "q_sample",
    "reverse_diffusion_sample",
    "reverse_diffusion_sample_v2",
]:
    obj = globals().get(name)

    print(f"\n--- {name} ---")
    if obj is None:
        print("Not defined in current session")
    else:
        try:
            print(inspect.getsource(obj)[:3500])
        except (OSError, TypeError):
            print("Defined, but source is unavailable to inspect.")

In [ ]:
import torch

print("Model class:", "DualBranchReverseUNet" in globals())
print("Diffusion schedule:", "betas" in globals())
print("Forward diffusion:", "q_sample" in globals())

if "betas" in globals() and torch.is_tensor(betas):
    print("Number of timesteps:", len(betas))
    print("First beta:", betas[0].item())
    print("Last beta:", betas[-1].item())

if "DualBranchReverseUNet" in globals():
    model = DualBranchReverseUNet()
    checkpoint = torch.load(
        "/content/drive/MyDrive/SANC/models/reverse_diffusion_model.pth",
        map_location="cpu",
        weights_only=False
    )
    result = model.load_state_dict(
        checkpoint["model_state_dict"], strict=True
    )
    print("Checkpoint compatibility:", result)

In [ ]:
import torch

checkpoint_path = (
    "/content/drive/MyDrive/SANC/models/reverse_diffusion_model.pth"
)

# Instantiate a fresh model; do not touch the training model
reverse_model_check = DualBranchReverseUNet().to(device)

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

# Strict loading verifies that parameter names and tensor shapes match
result = reverse_model_check.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

reverse_model_check.eval()

print("Checkpoint loaded successfully")
print("Device:", device)
print("Strict loading:", result)
print("Saved training metadata:", {
    key: value
    for key, value in checkpoint.items()
    if key not in ("model_state_dict", "optimizer_state_dict")
})

In [ ]:

# ============================================
# MEMBER 4 — STEP 2A
# VERIFY EXISTING SANC COMPONENTS
# ============================================

import torch

print("========== MEMBER 4 ENVIRONMENT CHECK ==========")

print("Device:", device)
print("Diffusion timesteps:", T)

print("\nSaliency model:")
print("Available:", "saliency_model" in globals())

print("\nSANC module:")
print("Available:", "sanc" in globals())

print("\nReverse model:")
print("Available:", "reverse_model" in globals())

# Verify actual saliency output
saliency_model.eval()

batch = next(iter(val_loader))
low_light = batch["low_light"].to(device)

with torch.no_grad():
    saliency = saliency_model(low_light)

print("\n========== SALIENCY OUTPUT CHECK ==========")

print("Model class:", type(saliency_model).__name__)
print("Output shape:", tuple(saliency.shape))
print("Minimum:", saliency.min().item())
print("Maximum:", saliency.max().item())
print("Mean:", saliency.mean().item())

print(
    "Is probability map:",
    saliency.min().item() >= 0
    and saliency.max().item() <= 1
)

print("\n========== CHECK COMPLETE ==========")


In [ ]:
# ============================================
# MEMBER 4 — STEP 2B
# VALIDATE EXISTING SANC NOISE COLORING
# ============================================

import torch

saliency_model.eval()

with torch.no_grad():

    noise = torch.randn_like(low_light)

    colored_noise = sanc(noise, saliency)

print("========== SANC VALIDATION ==========")

print("Input noise shape:", tuple(noise.shape))
print("Saliency shape:", tuple(saliency.shape))
print("Colored noise shape:", tuple(colored_noise.shape))

print("Noise mean:", noise.mean().item())
print("Noise std:", noise.std().item())

print("Colored noise mean:", colored_noise.mean().item())
print("Colored noise std:", colored_noise.std().item())

print(
    "All outputs finite:",
    torch.isfinite(colored_noise).all().item()
)

print(
    "Output shape matches input:",
    colored_noise.shape == noise.shape
)

In [ ]:
# ============================================
# MEMBER 4 — STEP 3
# UA-TD-SANC: UNCERTAINTY-AWARE,
# TIMESTEP-DEPENDENT NOISE COLORING
# ============================================

import torch
import torch.nn.functional as F

def uncertainty_aware_sanc(
    noise,
    saliency,
    timestep,
    T,
    lambda_u=0.5
):
    """
    Uncertainty-Aware, Timestep-Dependent SANC.

    Original SANC remains unchanged.
    """

    eps = 1e-8

    # 1. Keep saliency in valid probability range
    M = torch.clamp(saliency, 0.0, 1.0)

    # 2. Calculate normalized entropy uncertainty
    uncertainty = -(
        M * torch.log(M + eps)
        + (1 - M) * torch.log(1 - M + eps)
    ) / torch.log(torch.tensor(2.0, device=M.device))

    uncertainty = torch.clamp(uncertainty, 0.0, 1.0)

    # 3. Normalize diffusion timestep
    t = torch.as_tensor(
        timestep,
        device=noise.device,
        dtype=torch.float32
    )

    if t.ndim == 0:
        t = t.expand(noise.shape[0])

    t = t.reshape(-1, 1, 1, 1) / (T - 1)

    # 4. Calculate uncertainty-adjusted saliency
    effective_saliency = M * (
        1.0 - lambda_u * t * uncertainty
    )

    effective_saliency = torch.clamp(
        effective_saliency,
        0.0,
        1.0
    )

    # 5. Apply the ORIGINAL SANC module
    colored_noise = sanc(
        noise,
        effective_saliency
    )

    return colored_noise, effective_saliency, uncertainty


print("UA-TD-SANC extension defined successfully.")

In [ ]:
# ============================================
# MEMBER 4 — STEP 4
# TEST UA-TD-SANC AT DIFFERENT TIMESTEPS
# ============================================

test_noise = torch.randn_like(low_light)

with torch.no_grad():

    results = {}

    for timestep in [0, 25, 50, 75, 99]:

        colored, effective_map, uncertainty = (
            uncertainty_aware_sanc(
                test_noise,
                saliency,
                timestep,
                T
            )
        )

        results[timestep] = colored

        print(
            f"Timestep {timestep:2d} | "
            f"Mean effective saliency: "
            f"{effective_map.mean().item():.4f} | "
            f"Mean uncertainty: "
            f"{uncertainty.mean().item():.4f} | "
            f"Noise std: {colored.std().item():.4f} | "
            f"Finite: "
            f"{torch.isfinite(colored).all().item()}"
        )

In [ ]:
# ============================================
# MEMBER 4 — STEP 5
# VISUALIZE UNCERTAINTY-AWARE SALIENCY
# ============================================

import matplotlib.pyplot as plt

with torch.no_grad():

    _, effective_t0, uncertainty_map = uncertainty_aware_sanc(
        test_noise, saliency, 0, T
    )

    _, effective_t99, _ = uncertainty_aware_sanc(
        test_noise, saliency, 99, T
    )

# Visualize first image in batch
original = saliency[0, 0].cpu().numpy()
uncertainty_img = uncertainty_map[0, 0].cpu().numpy()
early = effective_t0[0, 0].cpu().numpy()
late = effective_t99[0, 0].cpu().numpy()

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

images = [
    (original, "Original Saliency"),
    (uncertainty_img, "Entropy Uncertainty"),
    (early, "Effective Saliency (t=0)"),
    (late, "Effective Saliency (t=99)")
]

for ax, (img, title) in zip(axes, images):
    ax.imshow(img, cmap="gray", vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis("off")

plt.suptitle("Original SANC vs UA-TD-SANC")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# MEMBER 4 — STEP 6B
# CORRECTED FACTORIAL ABLATION
# ============================================

lambda_u = 0.5
lambda_t = 0.5
timestep = 99

t_norm = timestep / (T - 1)

M = torch.clamp(saliency, 0, 1)
U = uncertainty_map

M_A = M
M_B = M * (1 - lambda_u * U)
M_C = M * (1 - lambda_t * t_norm)

M_D = M * (1 - lambda_u * U) * (
    1 - lambda_t * t_norm
)

variants = {
    "Original SANC": M_A,
    "UA-SANC": M_B,
    "TD-SANC": M_C,
    "Combined UA-TD-SANC": M_D
}

print("========== CORRECTED ABLATION ==========")

with torch.no_grad():
    for name, effective_map in variants.items():

        output = sanc(test_noise, effective_map)

        print(
            f"{name:24s} | "
            f"Saliency: {effective_map.mean().item():.5f} | "
            f"Noise std: {output.std().item():.5f} | "
            f"Finite: {torch.isfinite(output).all().item()}"
        )

In [ ]:
# ============================================
# MEMBER 4 — STEP 7
# INSPECT REVERSE DIFFUSION CHECKPOINT
# ============================================

import os
import torch

REVERSE_MODEL_PATH = (
    "/content/drive/MyDrive/SANC/models/"
    "reverse_diffusion_model.pth"
)

print("Checkpoint exists:", os.path.isfile(REVERSE_MODEL_PATH))

checkpoint = torch.load(
    REVERSE_MODEL_PATH,
    map_location=device,
    weights_only=False
)

print("\nCheckpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):

    print("\nTop-level keys:")
    print(list(checkpoint.keys()))

    for key, value in checkpoint.items():
        if isinstance(value, dict):
            print(f"\n{key}: {len(value)} entries")
            print("First keys:", list(value.keys())[:8])
        elif torch.is_tensor(value):
            print(f"{key}: Tensor {tuple(value.shape)}")
        else:
            print(f"{key}: {type(value).__name__}")

In [ ]:
import os

REVERSE_MODEL_PATH = (
    "/content/drive/MyDrive/SANC/models/"
    "reverse_diffusion_model.pth"
)

print("Checkpoint exists:", os.path.exists(REVERSE_MODEL_PATH))

In [ ]:
# ============================================
# MEMBER 4 — STEP 8
# RESTORE EXISTING REVERSE MODEL
# ============================================

import torch

reverse_model_check = DualBranchReverseUNet(
    in_channels=7,
    base_channels=64,
    time_dim=128
).to(device)

checkpoint = torch.load(
    REVERSE_MODEL_PATH,
    map_location=device,
    weights_only=True
)

result = reverse_model_check.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

reverse_model_check.eval()

print("Reverse model loaded successfully!")
print("Strict loading:", result)
print("Training epochs:", checkpoint["num_epochs"])
print(
    "Parameters:",
    sum(p.numel() for p in reverse_model_check.parameters())
)

In [ ]:
reverse_model_check.eval()

print("Model device:", next(reverse_model_check.parameters()).device)
print("Model training mode:", reverse_model_check.training)
print("Model parameter count:", sum(
    p.numel() for p in reverse_model_check.parameters()
))
print("T:", T)
print("Saliency model available:", saliency_model is not None)
print("SANC available:", sanc is not None)


In [ ]:
batch = next(iter(val_loader))

print("Available keys:", batch.keys())

for key, value in batch.items():
    if torch.is_tensor(value):
        print(f"{key}: shape={value.shape}, dtype={value.dtype}")
    else:
        print(f"{key}: type={type(value)}")

In [ ]:
# Extract the batch
low_light = batch["low_light"].to(device)
original = batch["original"].to(device)
mask = batch["mask"].to(device)

# Generate saliency map from low-light images
with torch.no_grad():
    saliency = saliency_model(low_light)
    saliency = torch.clamp(saliency, 0, 1)

# Create a test diffusion state
timestep = torch.full(
    (low_light.shape[0],),
    50,
    device=device,
    dtype=torch.long
)

# Use a random noisy state for a forward-pass test
x_t = torch.randn_like(low_light)

print("Low-light:", low_light.shape)
print("Original:", original.shape)
print("Saliency:", saliency.shape)
print("Noisy state:", x_t.shape)
print("Timestep:", timestep.shape)

In [ ]:
import inspect

print("Forward method:")
print(inspect.signature(reverse_model_check.forward))

print("\nForward method implementation:")
print(inspect.getsource(reverse_model_check.forward))

In [ ]:
import inspect

print(inspect.signature(reverse_model_check.forward))

In [ ]:
print(inspect.getsource(reverse_model_check.forward))

In [ ]:
reverse_model_check.eval()

with torch.no_grad():
    predicted_noise = reverse_model_check(
        x_t,
        low_light,
        saliency,
        timestep
    )

print("Predicted noise shape:", predicted_noise.shape)
print("Expected shape:", x_t.shape)
print("Output mean:", predicted_noise.mean().item())
print("Output std:", predicted_noise.std().item())
print("All values finite:", torch.isfinite(predicted_noise).all().item())

assert predicted_noise.shape == x_t.shape
assert torch.isfinite(predicted_noise).all()

print("\nReverse model forward pass successful!")

In [ ]:
import inspect

print("SANC class:", type(sanc).__name__)
print("\nForward method source:\n")

print(inspect.getsource(type(sanc).forward))

In [ ]:
import torch
import torch.nn.functional as F

def multiscale_sobel(images, eps=1e-6):
    """
    Extract multiscale structural features using Sobel filters.

    Input:
        images: (B, 3, H, W), RGB images

    Output:
        structural_map: (B, 1, H, W), normalized to [0, 1]
    """

    # 1. Convert RGB to grayscale
    gray = (
        0.299 * images[:, 0:1]
        + 0.587 * images[:, 1:2]
        + 0.114 * images[:, 2:3]
    )

    # 2. Define Sobel filters
    sobel_x = torch.tensor(
        [[-1, 0, 1],
         [-2, 0, 2],
         [-1, 0, 1]],
        dtype=gray.dtype,
        device=gray.device
    ).view(1, 1, 3, 3)

    sobel_y = torch.tensor(
        [[-1, -2, -1],
         [ 0,  0,  0],
         [ 1,  2,  1]],
        dtype=gray.dtype,
        device=gray.device
    ).view(1, 1, 3, 3)

    def edge_strength(x):
        gx = F.conv2d(x, sobel_x, padding=1)
        gy = F.conv2d(x, sobel_y, padding=1)

        magnitude = torch.sqrt(gx.square() + gy.square() + eps)

        # Normalize each image independently
        minimum = magnitude.amin(dim=(-2, -1), keepdim=True)
        maximum = magnitude.amax(dim=(-2, -1), keepdim=True)

        return (magnitude - minimum) / (maximum - minimum + eps)

    # 3. Extract edges at three scales
    edge_fine = edge_strength(gray)

    gray_medium = F.avg_pool2d(
        gray, kernel_size=3, stride=1, padding=1
    )
    edge_medium = edge_strength(gray_medium)

    gray_coarse = F.avg_pool2d(
        gray, kernel_size=7, stride=1, padding=3
    )
    edge_coarse = edge_strength(gray_coarse)

    # 4. Combine all three scales
    structural_map = (
        edge_fine + edge_medium + edge_coarse
    ) / 3.0

    return structural_map.clamp(0, 1)

In [ ]:
# Get one validation batch
batch = next(iter(val_loader))

low_light = batch["low_light"].to(device)

# Extract structural features
structural_map = multiscale_sobel(low_light)

print("Input shape:", low_light.shape)
print("Structural map shape:", structural_map.shape)

print("Minimum:", structural_map.min().item())
print("Maximum:", structural_map.max().item())
print("Mean:", structural_map.mean().item())

print("All values finite:", torch.isfinite(structural_map).all().item())

In [ ]:
# 1. Get the original U-Net saliency map
saliency_model.eval()

with torch.no_grad():
    saliency_map = saliency_model(low_light)

# 2. Ensure both maps have valid values
saliency_map = saliency_map.clamp(0, 1)
structural_map = structural_map.clamp(0, 1)

# 3. Combine learned saliency and Sobel structure
lambda_s = 0.30

combined_saliency = (
    (1 - lambda_s) * saliency_map
    + lambda_s * structural_map
)

combined_saliency = combined_saliency.clamp(0, 1)

# 4. Print diagnostics
print("Original saliency shape:", saliency_map.shape)
print("Structural map shape:", structural_map.shape)
print("Combined map shape:", combined_saliency.shape)

print("\nMean values:")
print("Original saliency:", saliency_map.mean().item())
print("Structural map:", structural_map.mean().item())
print("Combined saliency:", combined_saliency.mean().item())

print("\nCombined map range:")
print("Minimum:", combined_saliency.min().item())
print("Maximum:", combined_saliency.max().item())

print(
    "All values finite:",
    torch.isfinite(combined_saliency).all().item()
)

In [ ]:
import matplotlib.pyplot as plt

# Select the first image in the batch
idx = 0

input_image = low_light[idx].detach().cpu().permute(1, 2, 0)
original_map = saliency_map[idx, 0].detach().cpu()
sobel_map = structural_map[idx, 0].detach().cpu()
combined_map = combined_saliency[idx, 0].detach().cpu()

# Plot all four
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

axes[0].imshow(input_image.clamp(0, 1))
axes[0].set_title("Low-light Input")

axes[1].imshow(original_map, cmap="hot", vmin=0, vmax=1)
axes[1].set_title("U-Net Saliency")

axes[2].imshow(sobel_map, cmap="gray", vmin=0, vmax=1)
axes[2].set_title("Multiscale Sobel")

axes[3].imshow(combined_map, cmap="hot", vmin=0, vmax=1)
axes[3].set_title("Combined Saliency")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Numerical stability
eps = 1e-8

# Clamp saliency probabilities
M = saliency_map.clamp(eps, 1 - eps)

# Calculate normalized binary entropy
uncertainty_map = -(
    M * torch.log(M)
    + (1 - M) * torch.log(1 - M)
) / torch.log(torch.tensor(2.0, device=device))

uncertainty_map = uncertainty_map.clamp(0, 1)

# Apply uncertainty-aware adjustment
lambda_u = 0.5

uncertainty_aware_saliency = (
    combined_saliency * (1 - lambda_u * uncertainty_map)
)

uncertainty_aware_saliency = uncertainty_aware_saliency.clamp(0, 1)

# Diagnostics
print("Uncertainty mean:", uncertainty_map.mean().item())
print("Uncertainty range:",
      uncertainty_map.min().item(),
      uncertainty_map.max().item())

print("\nCombined saliency mean:",
      combined_saliency.mean().item())

print("Uncertainty-aware saliency mean:",
      uncertainty_aware_saliency.mean().item())

print("All values finite:",
      torch.isfinite(uncertainty_aware_saliency).all().item())

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

axes[0].imshow(saliency_map[0, 0].detach().cpu(),
               cmap="hot", vmin=0, vmax=1)
axes[0].set_title("Original U-Net Saliency")

axes[1].imshow(uncertainty_map[0, 0].detach().cpu(),
               cmap="viridis", vmin=0, vmax=1)
axes[1].set_title("Entropy Uncertainty")

axes[2].imshow(combined_saliency[0, 0].detach().cpu(),
               cmap="hot", vmin=0, vmax=1)
axes[2].set_title("Combined Saliency")

axes[3].imshow(uncertainty_aware_saliency[0, 0].detach().cpu(),
               cmap="hot", vmin=0, vmax=1)
axes[3].set_title("Uncertainty-aware Saliency")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Timestep control strength
lambda_t = 0.05

# Test three timesteps
test_timesteps = [0, 50, 99]

# Use uncertainty-aware saliency
S_u = uncertainty_aware_saliency

for t in test_timesteps:

    # Normalize timestep
    tau = t / (T - 1)

    # Original SANC frequency weights
    low_weight = 0.55 - 0.35 * S_u
    mid_weight = 0.30 * torch.ones_like(S_u)
    high_weight = 0.15 + 0.35 * S_u

    # Apply timestep adjustment
    adjusted_low = low_weight + lambda_t * tau
    adjusted_mid = mid_weight
    adjusted_high = high_weight - lambda_t * tau

    # Clamp and normalize
    total = (
        adjusted_low.clamp(min=0)
        + adjusted_mid.clamp(min=0)
        + adjusted_high.clamp(min=0)
    )

    adjusted_low = adjusted_low.clamp(min=0) / (total + 1e-8)
    adjusted_mid = adjusted_mid.clamp(min=0) / (total + 1e-8)
    adjusted_high = adjusted_high.clamp(min=0) / (total + 1e-8)

    print(f"\nTimestep: {t}")
    print("Low frequency mean:", adjusted_low.mean().item())
    print("Mid frequency mean:", adjusted_mid.mean().item())
    print("High frequency mean:", adjusted_high.mean().item())

    print(
        "Weight sum:",
        (adjusted_low + adjusted_mid + adjusted_high).mean().item()
    )

In [ ]:
import inspect

forward_source = inspect.getsource(type(sanc).forward)

print(forward_source)

In [ ]:
import inspect

print(inspect.getsource(type(sanc)._frequency_bands))

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ExtendedSANCNoiseColoring(nn.Module):

    def __init__(
        self,
        original_sanc,
        lambda_t=0.05
    ):
        super().__init__()

        # Reuse the original SANC implementation
        self.original_sanc = original_sanc

        # Timestep control strength
        self.lambda_t = lambda_t

    def forward(
        self,
        noise,
        saliency,
        timestep,
        T
    ):

        B, C, H, W = noise.shape

        # 1. Match saliency resolution
        if saliency.shape[-2:] != (H, W):
            saliency = F.interpolate(
                saliency,
                size=(H, W),
                mode="bilinear",
                align_corners=False
            )

        # 2. Apply the original SANC saliency processing
        M = torch.clamp(saliency, 0.0, 1.0)
        M = M ** self.original_sanc.saliency_power

        # 3. Fourier transform
        noise_fft = torch.fft.fft2(
            noise,
            dim=(-2, -1)
        )

        # 4. Reuse the original frequency bands
        low_band, mid_band, high_band = (
            self.original_sanc._frequency_bands(
                H,
                W,
                noise.device
            )
        )

        low_band = low_band[None, None]
        mid_band = mid_band[None, None]
        high_band = high_band[None, None]

        # 5. Extract frequency components
        low_noise = torch.fft.ifft2(
            noise_fft * low_band,
            dim=(-2, -1)
        ).real

        mid_noise = torch.fft.ifft2(
            noise_fft * mid_band,
            dim=(-2, -1)
        ).real

        high_noise = torch.fft.ifft2(
            noise_fft * high_band,
            dim=(-2, -1)
        ).real

        # 6. Original saliency-dependent weights
        low_weight = 0.55 - 0.35 * M
        mid_weight = 0.30 * torch.ones_like(M)
        high_weight = 0.15 + 0.35 * M

        # 7. Prepare timestep
        t = torch.as_tensor(
            timestep,
            device=noise.device,
            dtype=torch.float32
        )

        if t.ndim == 0:
            t = t.expand(B)

        t = t.reshape(B, 1, 1, 1)
        tau = t / (T - 1)

        # 8. Apply timestep-dependent frequency control
        low_weight = low_weight + self.lambda_t * tau
        high_weight = high_weight - self.lambda_t * tau

        # 9. Ensure nonnegative weights and normalize
        low_weight = low_weight.clamp(min=0)
        mid_weight = mid_weight.clamp(min=0)
        high_weight = high_weight.clamp(min=0)

        total = low_weight + mid_weight + high_weight + 1e-8

        low_weight = low_weight / total
        mid_weight = mid_weight / total
        high_weight = high_weight / total

        # 10. Blend frequency components
        colored_noise = (
            low_weight * low_noise
            + mid_weight * mid_noise
            + high_weight * high_noise
        )

        # 11. Preserve original SANC normalization
        mean = colored_noise.mean(
            dim=(-2, -1),
            keepdim=True
        )

        std = colored_noise.std(
            dim=(-2, -1),
            keepdim=True
        )

        colored_noise = (
            colored_noise - mean
        ) / (std + 1e-6)

        return colored_noise

In [ ]:
extended_sanc = ExtendedSANCNoiseColoring(
    original_sanc=sanc,
    lambda_t=0.05
)

extended_sanc.eval()

print("Extended SANC initialized successfully.")
print("Device:", device)
print("Timestep control:", extended_sanc.lambda_t)

In [ ]:
# Generate the same input noise for every timestep
torch.manual_seed(42)

test_noise = torch.randn(
    8, 3, 128, 128,
    device=device
)

# Get the frequency masks
low_band, mid_band, high_band = sanc._frequency_bands(
    128, 128, device
)

# Reshape masks for broadcasting
low_band = low_band[None, None]
mid_band = mid_band[None, None]
high_band = high_band[None, None]

results = []

for t in [0, 50, 99]:

    with torch.no_grad():
        output = extended_sanc(
            noise=test_noise,
            saliency=uncertainty_aware_saliency,
            timestep=t,
            T=T
        )

    # Fourier transform of output
    output_fft = torch.fft.fft2(
        output,
        dim=(-2, -1)
    )

    power = output_fft.abs().square()

    # Measure energy in each frequency band
    low_energy = (power * low_band).sum()
    mid_energy = (power * mid_band).sum()
    high_energy = (power * high_band).sum()

    total_energy = low_energy + mid_energy + high_energy

    low_ratio = (low_energy / total_energy).item()
    mid_ratio = (mid_energy / total_energy).item()
    high_ratio = (high_energy / total_energy).item()

    results.append((t, low_ratio, mid_ratio, high_ratio))

    print(f"\nTimestep: {t}")
    print(f"Low-frequency energy:  {low_ratio:.4f}")
    print(f"Mid-frequency energy:  {mid_ratio:.4f}")
    print(f"High-frequency energy: {high_ratio:.4f}")

In [ ]:
# Generate random Gaussian noise
test_noise = torch.randn(
    8, 3, 128, 128,
    device=device
)

# Test at three diffusion timesteps
for t in [0, 50, 99]:

    with torch.no_grad():
        output = extended_sanc(
            noise=test_noise,
            saliency=uncertainty_aware_saliency,
            timestep=t,
            T=T
        )

    print(f"\nTimestep: {t}")
    print("Output shape:", output.shape)
    print("Mean:", output.mean().item())
    print("Standard deviation:", output.std().item())
    print("All finite:", torch.isfinite(output).all().item())

In [ ]:
# ============================================================
# MEMBER 4 — AUDIT A1
# NOISE-MODULE EQUIVALENCE CHECK (read-only, no training)
# ============================================================
import torch
torch.manual_seed(0)

# 1. Required objects
for name in ["T", "sanc", "sanc_noise_coloring", "extended_sanc",
             "saliency_model", "uncertainty_aware_saliency", "low_light"]:
    print(f"{name:28s}", "OK" if name in globals() else "MISSING")
print("Reverse model variable(s):",
      [n for n in ["reverse_model", "reverse_model_check"] if n in globals()])

# 2. Original U-Net saliency M for the same batch used by the extension cells
saliency_model.eval()
with torch.no_grad():
    M = saliency_model(low_light)
print("\nSaliency model:", type(saliency_model).__name__,
      f"| range [{M.min().item():.5f}, {M.max().item():.5f}]",
      "| shapes match S_u:", M.shape == uncertainty_aware_saliency.shape)

# 3. Same Gaussian noise through every colouring path
z = torch.randn_like(low_light)
with torch.no_grad():
    eps_train = sanc_noise_coloring(z, M)   # instance used in Cell 236 training
    variants = {
        "sanc (STEP 5A), M":    sanc(z, M),
        "extended, t=0,  M":    extended_sanc(z, M, 0, T),
        "extended, t=0,  S_u":  extended_sanc(z, uncertainty_aware_saliency, 0, T),
        "extended, t=99, S_u":  extended_sanc(z, uncertainty_aware_saliency, T - 1, T),
    }

print("\nDifference from training-time SANC noise:")
for name, eps in variants.items():
    d = eps - eps_train
    print(f"  {name:22s} | max |d| = {d.abs().max().item():.2e}"
          f" | RMS d = {d.pow(2).mean().sqrt().item():.4f}")

In [ ]:
# ============================================================
# MEMBER 4 — AUDIT A2
# FROZEN-MODEL COMPATIBILITY WITH EXTENDED NOISE (read-only)
# ============================================================
import torch
import torch.nn.functional as F

model = reverse_model_check
model.eval()
saliency_model.eval()

# Same validation batch that was used to build S_u
x0 = batch["original"].to(device)
assert torch.equal(batch["low_light"].to(device), low_light), \
    "batch and low_light do not match - stop and report"

S_u = uncertainty_aware_saliency
with torch.no_grad():
    M = saliency_model(low_light)   # reverse model is always conditioned on M

B = x0.shape[0]
timesteps = [0, 10, 25, 50, 75, 99]
seeds = [0, 1, 2, 3]
names = ["Original SANC", "C only", "A+B only", "A+B+C"]

pred_mse = {n: {t: [] for t in timesteps} for n in names}
target_gap = {n: {t: [] for t in timesteps} for n in names}

with torch.no_grad():
    for t in timesteps:
        tt = torch.full((B,), t, device=device, dtype=torch.long)
        for s in seeds:
            g = torch.Generator(device=device).manual_seed(s)
            z = torch.randn(x0.shape, generator=g, device=device)

            eps = {
                "Original SANC": sanc_noise_coloring(z, M),
                "C only":        extended_sanc(z, M, t, T),
                "A+B only":      extended_sanc(z, S_u, 0, T),
                "A+B+C":         extended_sanc(z, S_u, t, T),
            }

            for n in names:
                x_t = q_sample(x0, tt, eps[n])
                pred = model(x_t, low_light, M, tt)
                pred_mse[n][t].append(F.mse_loss(pred, eps[n]).item())
                target_gap[n][t].append(
                    F.mse_loss(eps[n], eps["Original SANC"]).item())

mean = lambda v: sum(v) / len(v)

print("Noise-prediction MSE of the frozen model (mean of 4 seeds, 8 images)")
print("Ratio in brackets = variant MSE / Original SANC MSE\n")
print(f"{'t':>3} | " + " | ".join(f"{n:>18}" for n in names))
for t in timesteps:
    base = mean(pred_mse["Original SANC"][t])
    row = []
    for n in names:
        m = mean(pred_mse[n][t])
        row.append(f"{m:.5f} ({m / base:4.2f}x)")
    print(f"{t:>3} | " + " | ".join(f"{r:>18}" for r in row))

print("\nTarget gap: MSE between the variant noise and the original SANC noise")
print(f"{'t':>3} | " + " | ".join(f"{n:>10}" for n in names[1:]))
for t in timesteps:
    print(f"{t:>3} | " + " | ".join(
        f"{mean(target_gap[n][t]):>10.5f}" for n in names[1:]))

In [ ]:
# ============================================================
# MEMBER 4 — AUDIT A3
# FORWARD-CHAIN VALIDITY OF TIMESTEP-DEPENDENT COLOURING
# Condition: (1-ab_t) P_t(f) - a_t (1-ab_{t-1}) P_{t-1}(f) >= 0
# (read-only arithmetic)
# ============================================================
import torch

H, W = low_light.shape[-2:]
low_b, mid_b, high_b = sanc_noise_coloring._frequency_bands(H, W, device)

def power_spectrum(S, tau, lam):
    # Same weight rule as ExtendedSANCNoiseColoring (clamp, then normalise)
    wl = max(0.55 - 0.35 * S + lam * tau, 0.0)
    wm = 0.30
    wh = max(0.15 + 0.35 * S - lam * tau, 0.0)
    tot = wl + wm + wh
    P = ((wl * low_b + wm * mid_b + wh * high_b) / tot) ** 2
    return P / P.mean()          # matches unit-variance normalisation

# Saliency levels covering the real S_u distribution
q = torch.quantile(S_u.flatten().float()[::4],
                   torch.tensor([0.5, 0.99], device=device))
S_levels = [0.0, q[0].item(), q[1].item(), 1.0]
print("S_u levels tested:", [round(s, 4) for s in S_levels], "\n")

print(f"{'lambda_t':>8} | {'worst increment / beta_t':>24} | {'at S':>6} | {'at t':>4} | result")
for lam in [0.0, 0.05, 0.10, 0.15, 0.50]:
    worst, where = float("inf"), None
    for S in S_levels:
        for t in range(1, T):
            P_t  = power_spectrum(S, t / (T - 1), lam)
            P_tm = power_spectrum(S, (t - 1) / (T - 1), lam)
            inc = ((1 - alpha_bars[t]) * P_t
                   - alphas[t] * (1 - alpha_bars[t - 1]) * P_tm)
            val = (inc.min() / betas[t]).item()
            if val < worst:
                worst, where = val, (S, t)
    verdict = "VALID" if worst >= 0 else "INVALID"
    print(f"{lam:>8.2f} | {worst:>24.4f} | {where[0]:>6.3f} | {where[1]:>4d} | {verdict}")

In [ ]:
# ============================================================
# MEMBER 4 — STEP 9
# EXTENDED SALIENCY FUNCTION (COMPONENTS A + B)
# Reuses multiscale_sobel; formulas identical to the existing
# structural-fusion and uncertainty cells.
# ============================================================
import torch

def compute_extended_saliency(low_light, saliency_model,
                              lambda_s=0.30, lambda_u=0.5, eps=1e-8):
    with torch.no_grad():
        M = saliency_model(low_light).clamp(0, 1)          # reverse-model condition
    E = multiscale_sobel(low_light).clamp(0, 1)             # Component A input
    S = ((1 - lambda_s) * M + lambda_s * E).clamp(0, 1)     # Component A
    # Handover formula, NaN-safe in float32
    # (clamp(eps, 1-eps) fails: 1 - 1e-8 == 1.0 in float32)
    U = (-(M * torch.log(M + eps) + (1 - M) * torch.log(1 - M + eps))
        / torch.log(torch.tensor(2.0, device=M.device))).clamp(0, 1)
    S_u = (S * (1 - lambda_u * U)).clamp(0, 1)
                # Component B
    assert torch.isfinite(U).all() and torch.isfinite(S_u).all(), "non-finite U or S_u"
    return {"M": M, "E": E, "S": S, "U": U, "S_u": S_u}

# Equivalence check against the existing cell outputs (same batch)
saliency_model.eval()
out = compute_extended_saliency(low_light, saliency_model)
checks = {
    "E   vs structural_map":             (out["E"],   structural_map),
    "S   vs combined_saliency":          (out["S"],   combined_saliency),
    "U   vs uncertainty_map":            (out["U"],   uncertainty_map),
    "S_u vs uncertainty_aware_saliency": (out["S_u"], uncertainty_aware_saliency),
}
for name, (a, b) in checks.items():
    print(f"{name:36s} | max |diff| = {(a - b).abs().max().item():.2e}")
print("\nS_u mean:", round(out["S_u"].mean().item(), 7), "(expected ~0.1182441)")

In [ ]:
# ============================================================
# MEMBER 4 — STEP 10
# FACTORIAL ABLATION NOISE FUNCTION (A, B, C ON/OFF)
# Replaces the old lambda_t = 0.5 factorial ablation (invalid
# forward chain per Audit A3). Reverse model is always
# conditioned on M; only the noise colouring changes.
# ============================================================
import torch

def ablation_noise(z, sal, t, use_A, use_B, use_C, lambda_u=0.5):
    base = sal["S"] if use_A else sal["M"]                     # Component A
    cmap = (base * (1 - lambda_u * sal["U"])).clamp(0, 1) \
           if use_B else base                                  # Component B
    if use_C:
        return extended_sanc(z, cmap, t, T)                    # Component C
    return sanc_noise_coloring(z, cmap)                        # training-time SANC

ARMS = [(a, b, c) for a in (0, 1) for b in (0, 1) for c in (0, 1)]
def arm_name(a, b, c):
    on = [n for n, f in zip("ABC", (a, b, c)) if f]
    return "SANC (baseline)" if not on else "+" + "+".join(on)

# ---------------- Sanity checks (same batch, t = 99) ----------------
sal = compute_extended_saliency(low_light, saliency_model)
t_chk = T - 1
g = torch.Generator(device=device).manual_seed(0)
z = torch.randn(low_light.shape, generator=g, device=device)

with torch.no_grad():
    eps = {arm_name(*arm): ablation_noise(z, sal, t_chk, *arm) for arm in ARMS}
    ref_base = sanc_noise_coloring(z, sal["M"])
    ref_full = extended_sanc(z, sal["S_u"], t_chk, T)

print("Consistency:")
print("  baseline arm vs training SANC   | max |d| =",
      f"{(eps['SANC (baseline)'] - ref_base).abs().max().item():.2e}")
print("  +A+B+C arm vs extended(S_u)     | max |d| =",
      f"{(eps['+A+B+C'] - ref_full).abs().max().item():.2e}")

# Spectral shares, same method as the existing spectral-energy test
low_b, mid_b, high_b = sanc_noise_coloring._frequency_bands(*low_light.shape[-2:], device)
def shares(x):
    p = torch.fft.fft2(x, dim=(-2, -1)).abs().square()
    e = [(p * b[None, None]).sum() for b in (low_b, mid_b, high_b)]
    tot = sum(e)
    return [(v / tot).item() for v in e]

print(f"\nt = {t_chk}")
print(f"{'arm':>16} | {'mean':>9} | {'std':>7} | {'low':>6} | {'mid':>6} | {'high':>6} | RMS vs baseline")
for name, x in eps.items():
    lo, mi, hi = shares(x)
    rms = (x - eps["SANC (baseline)"]).pow(2).mean().sqrt().item()
    print(f"{name:>16} | {x.mean().item():>9.2e} | {x.std().item():>7.4f} | "
          f"{lo:>6.4f} | {mi:>6.4f} | {hi:>6.4f} | {rms:.4f}")

In [ ]:
# ============================================================
# MEMBER 4 — STEP 11
# REGIONAL BEHAVIOUR OF COMPONENTS A AND B (read-only)
# ============================================================
import torch

mask = batch["mask"].to(device)
sal = compute_extended_saliency(low_light, saliency_model)

maps = {
    "M (SANC)":   sal["M"],
    "A: S":       sal["S"],
    "B: M(1-lU)": (sal["M"] * (1 - 0.5 * sal["U"])).clamp(0, 1),
    "A+B: S_u":   sal["S_u"],
}

E = sal["E"]
edge_thr = torch.quantile(E.flatten()[::4].float(), 0.90)
regions = {
    "inside mask":      mask > 0.5,
    "outside mask":     mask <= 0.5,
    "edges (top 10%)":  E >= edge_thr,
    "uncertain (U>.5)": sal["U"] > 0.5,
}

print("Pixel share per region:")
for r, sel in regions.items():
    print(f"  {r:18s} {sel.float().mean().item():.4f}")

print(f"\n{'map':>12} | " + " | ".join(f"{r:>16}" for r in regions))
for name, m in maps.items():
    vals = [m[sel].mean().item() if sel.any() else float('nan')
            for sel in regions.values()]
    print(f"{name:>12} | " + " | ".join(f"{v:>16.4f}" for v in vals))

In [ ]:
# ============================================================
# MEMBER 4 — STEP 12
# CROSS-DATASET COMPATIBILITY + REGIONAL CHECK (read-only)
# Uses TRAINING images so the test split stays untouched.
# ============================================================
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

def dataset_batch(name, n=16, split="train", seed=0):
    df = combined_df[(combined_df["dataset"] == name) &
                     (combined_df["split"] == split)]
    df = df.sample(n=n, random_state=seed).reset_index(drop=True)
    b = next(iter(DataLoader(SANCMedicalDataset(df), batch_size=n, shuffle=False)))
    return {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in b.items()}

reverse_model_check.eval()
saliency_model.eval()

for ds in ["Kvasir-SEG", "HAM10000", "ORIGA"]:
    b = dataset_batch(ds)
    ll, x0, mask = b["low_light"], b["original"], b["mask"]
    sal = compute_extended_saliency(ll, saliency_model)
    B = ll.shape[0]

    # ---- Compatibility: full A+B+C vs baseline SANC ----
    print("=" * 72)
    print(f"{ds}  (n = {B} training images)")
    print("  t  | MSE baseline | MSE A+B+C | ratio")
    with torch.no_grad():
        for t in [10, 25, 50, 99]:
            tt = torch.full((B,), t, device=device, dtype=torch.long)
            mb, mf = [], []
            for s in [0, 1]:
                g = torch.Generator(device=device).manual_seed(s)
                z = torch.randn(x0.shape, generator=g, device=device)
                for arm, store in [((0, 0, 0), mb), ((1, 1, 1), mf)]:
                    e = ablation_noise(z, sal, t, *arm)
                    p = reverse_model_check(q_sample(x0, tt, e), ll, sal["M"], tt)
                    store.append(F.mse_loss(p, e).item())
            base, full = sum(mb) / len(mb), sum(mf) / len(mf)
            print(f"  {t:>2} |   {base:.5f}    |  {full:.5f}  | {full / base:.2f}x")

    # ---- Regional behaviour ----
    E = sal["E"]
    regions = {
        "inside mask": mask > 0.5,
        "edges top10%": E >= torch.quantile(E.flatten()[::4].float(), 0.90),
        "uncertain": sal["U"] > 0.5,
    }
    maps = {
        "M (SANC)": sal["M"],
        "A: S": sal["S"],
        "B: M(1-lU)": (sal["M"] * (1 - 0.5 * sal["U"])).clamp(0, 1),
        "A+B: S_u": sal["S_u"],
    }
    print(f"\n  {'map':>11} | " + " | ".join(f"{r:>12}" for r in regions))
    for name, m in maps.items():
        vals = [m[sel].mean().item() if sel.any() else float("nan")
                for sel in regions.values()]
        print(f"  {name:>11} | " + " | ".join(f"{v:>12.4f}" for v in vals))
print("=" * 72)

In [ ]:
# ============================================================
# MEMBER 4 — STEP 13
# MEMBER 3 DDIM SAMPLER + FIRST ABLATION SMOKE TEST
# (smoke test only: 8 Kvasir validation images - NOT a result)
# ============================================================
import torch
from types import SimpleNamespace

# ---- verbatim from SANC_Member3_Evaluation.ipynb, section 2 ----
def make_noise(kind, shape, saliency, gen=None, sanc=None):
    dev = saliency.device
    z = torch.randn(shape, generator=gen, device=dev) if gen is not None else torch.randn(shape, device=dev)
    if kind == "gaussian":
        return z
    if kind == "blue":
        return blue_colour(z)
    if kind == "sanc":
        return sanc(z, saliency)
    raise ValueError(kind)

@torch.no_grad()
def ddim_sample(model, low, sal, kind, sanc, sched, t0, n_steps, gen=None, clip_x0=True, init="lowlight"):
    """Deterministic (eta=0) sampler, started at step t0 from the low-light image."""
    model.eval()
    B = low.shape[0]
    ts = torch.linspace(t0, 0, max(2, n_steps)).round().long().tolist()
    ts = sorted(set(ts), reverse=True)
    eps0 = make_noise(kind, low.shape, sal, gen, sanc)
    x0_init = low if init == "lowlight" else torch.zeros_like(low)
    ab0 = sched.ab[t0]
    x = ab0.sqrt() * x0_init + (1.0 - ab0).sqrt() * eps0
    for i, t in enumerate(ts):
        tt = torch.full((B,), t, device=low.device, dtype=torch.long)
        eps = model(x, low, sal, tt)
        ab = sched.ab[t]
        x0 = (x - (1.0 - ab).sqrt() * eps) / ab.sqrt()
        if clip_x0:
            x0 = x0.clamp(0.0, 1.0)
            eps = (x - ab.sqrt() * x0) / (1.0 - ab).sqrt()
        if i + 1 < len(ts):
            abp = sched.ab[ts[i + 1]]
            x = abp.sqrt() * x0 + (1.0 - abp).sqrt() * eps
        else:
            x = x0
    return x
# ---- end verbatim ----

sched_m3 = SimpleNamespace(ab=alpha_bars)   # ddim_sample only uses sched.ab (same schedule)

class ArmNoise:
    """Noise callable for ddim_sample. Ignores the saliency it is given (M, used for
    model conditioning) and returns the colouring of the chosen ablation arm."""
    def __init__(self, sal_dict, t0, arm):
        self.sal, self.t0, self.arm = sal_dict, t0, arm
    def __call__(self, z, _saliency):
        return ablation_noise(z, self.sal, self.t0, *self.arm)

def psnr_batch(out, ref):
    mse = (out.clamp(0, 1) - ref).pow(2).mean(dim=(1, 2, 3))
    return 10 * torch.log10(1.0 / mse)

# ---- smoke test on the validation batch ----
x0 = batch["original"].to(device)
sal = compute_extended_saliency(low_light, saliency_model)
M = sal["M"]
T0, STEPS, SEED = 99, 25, 42
gen = lambda: torch.Generator(device=device).manual_seed(SEED)

# 1. Integration check: baseline arm == Member 3's own SANC call
out_m3 = ddim_sample(reverse_model_check, low_light, M, "sanc", sanc_noise_coloring,
                     sched_m3, T0, STEPS, gen())
out_arm = ddim_sample(reverse_model_check, low_light, M, "sanc",
                      ArmNoise(sal, T0, (0, 0, 0)), sched_m3, T0, STEPS, gen())
print("Integration check | baseline arm vs Member 3 SANC call | max |d| =",
      f"{(out_arm - out_m3).abs().max().item():.2e}")

# 2. All 8 arms, same seed
print(f"\nLow-light input  | PSNR {psnr_batch(low_light, x0).mean().item():6.3f} dB"
      f" | mean {low_light.mean().item():.4f}")
print(f"Reference        |                  | mean {x0.mean().item():.4f}\n")
print(f"{'arm':>16} | {'PSNR (dB)':>9} | {'d vs base':>9} | {'out mean':>8}")
base_psnr = None
for arm in ARMS:
    out = ddim_sample(reverse_model_check, low_light, M, "sanc",
                      ArmNoise(sal, T0, arm), sched_m3, T0, STEPS, gen()).clamp(0, 1)
    p = psnr_batch(out, x0).mean().item()
    base_psnr = p if base_psnr is None else base_psnr
    print(f"{arm_name(*arm):>16} | {p:9.3f} | {p - base_psnr:+9.3f} | {out.mean().item():8.4f}")

In [ ]:
# ============================================================
# MEMBER 4 — STEP 14
# SEED NOISE FLOOR + t0 SENSITIVITY (VALIDATION ONLY)
# ============================================================
import torch

vb = dataset_batch("Kvasir-SEG", n=32, split="val", seed=0)
ll_v, x0_v = vb["low_light"], vb["original"]
sal_v = compute_extended_saliency(ll_v, saliency_model)

SEEDS = [42, 43, 44]
T0S = [99, 75, 50, 25, 10]
BASE, FULL = (0, 0, 0), (1, 1, 1)

def run_arm(arm, t0, seed):
    g = torch.Generator(device=device).manual_seed(seed)
    out = ddim_sample(reverse_model_check, ll_v, sal_v["M"], "sanc",
                      ArmNoise(sal_v, t0, arm), sched_m3, t0, min(t0 + 1, 25), g)
    return psnr_batch(out, x0_v)          # per-image PSNR, shape [n]

print(f"n = {ll_v.shape[0]} Kvasir-SEG validation images | "
      f"low-light PSNR {psnr_batch(ll_v, x0_v).mean().item():.3f} dB\n")
print(f"{'t0':>3} | {'base PSNR':>9} | {'seed SD':>7} | {'A+B+C - base':>12} | {'per-image SD':>12} | {'max |diff|':>10}")
for t0 in T0S:
    pb = torch.stack([run_arm(BASE, t0, s) for s in SEEDS])   # [seeds, n]
    pf = torch.stack([run_arm(FULL, t0, s) for s in SEEDS])
    seed_means = pb.mean(dim=1)                               # batch mean per seed
    diff = pf - pb                                            # paired, same seed + image
    print(f"{t0:>3} | {seed_means.mean().item():9.3f} | {seed_means.std().item():7.4f} | "
          f"{diff.mean().item():+12.4f} | {diff.std().item():12.4f} | {diff.abs().max().item():10.4f}")

In [ ]:
# ============================================================
# MEMBER 4 — STEP 15
# TEST-SPLIT ABLATION RUNNER + QUICK TIMING RUN (no files saved)
# Sampler frozen: Member 3 DDIM, t0 = 99, 25 steps (STEP 14)
# ============================================================
import os, time
import torch
import torch.nn.functional as F
import pandas as pd
from torch.utils.data import DataLoader

ABL_T0, ABL_STEPS = 99, 25

# ---- SSIM (Gaussian 11x11, sigma 1.5, valid region; for arm-vs-arm comparison) ----
_g = torch.exp(-((torch.arange(11, device=device) - 5).float() ** 2) / (2 * 1.5 ** 2))
_g = _g / _g.sum()
_W = (_g[:, None] @ _g[None, :]).expand(3, 1, 11, 11).contiguous()

def ssim_batch(x, y, C1=0.01 ** 2, C2=0.03 ** 2):
    x = x.clamp(0, 1)
    f = lambda z: F.conv2d(z, _W, groups=3)
    mx, my = f(x), f(y)
    sxx, syy, sxy = f(x * x) - mx ** 2, f(y * y) - my ** 2, f(x * y) - mx * my
    m = ((2 * mx * my + C1) * (2 * sxy + C2)) / ((mx ** 2 + my ** 2 + C1) * (sxx + syy + C2))
    return m.mean(dim=(1, 2, 3))

# ---- Test split (exclude unreadable empty mask file) ----
test_df = combined_df[combined_df["split"] == "test"].reset_index(drop=True)
ok = test_df["mask"].apply(lambda p: os.path.isfile(p) and os.path.getsize(p) > 0)
print("Test images:", len(test_df), "| excluded (empty mask file):", int((~ok).sum()))
test_df = test_df[ok].reset_index(drop=True)
print("Used:", len(test_df), "|", test_df["dataset"].value_counts().to_dict())

@torch.no_grad()
def run_ablation(df, seeds, arms=ARMS, batch_size=32):
    loader = DataLoader(SANCMedicalDataset(df), batch_size=batch_size, shuffle=False)
    rows = []
    for bi, b in enumerate(loader):
        ll, ref = b["low_light"].to(device), b["original"].to(device)
        n = ll.shape[0]
        ds_names = b.get("dataset", ["?"] * n)
        fnames = b.get("filename", [f"b{bi}_{k}" for k in range(n)])
        sal = compute_extended_saliency(ll, saliency_model)

        p, s = psnr_batch(ll, ref), ssim_batch(ll, ref)
        for k in range(n):
            rows.append(dict(arm="Low-light input", seed=-1, dataset=ds_names[k],
                             filename=fnames[k], PSNR=p[k].item(), SSIM=s[k].item()))
        for seed in seeds:
            for arm in arms:
                g = torch.Generator(device=device).manual_seed(seed * 1000 + bi)  # same z for every arm
                out = ddim_sample(reverse_model_check, ll, sal["M"], "sanc",
                                  ArmNoise(sal, ABL_T0, arm), sched_m3,
                                  ABL_T0, ABL_STEPS, g).clamp(0, 1)
                p, s = psnr_batch(out, ref), ssim_batch(out, ref)
                for k in range(n):
                    rows.append(dict(arm=arm_name(*arm), seed=seed, dataset=ds_names[k],
                                     filename=fnames[k], PSNR=p[k].item(), SSIM=s[k].item()))
    return pd.DataFrame(rows)

# ---- Quick timing run: 8 images per dataset, 1 seed ----
quick_df = test_df.groupby("dataset").head(8).reset_index(drop=True)
t_start = time.time()
quick = run_ablation(quick_df, seeds=[42])
elapsed = time.time() - t_start

print(f"\nQuick run: {len(quick_df)} images x 8 arms x 1 seed in {elapsed:.0f} s")
print(f"Estimated full run ({len(test_df)} images x 8 arms x 3 seeds): "
      f"~{elapsed * len(test_df) / len(quick_df) * 3 / 60:.0f} min\n")
order = ["Low-light input"] + [arm_name(*a) for a in ARMS]
print(quick.groupby("arm")[["PSNR", "SSIM"]].mean().reindex(order).round(4))

In [ ]:
# ============================================================
# MEMBER 4 — STEP 16
# FULL TEST-SPLIT ABLATION (1 seed) — per-image CSV saved to Drive
# ============================================================
import os, time

OUT_DIR = "/content/drive/MyDrive/SANC/results/member4"
os.makedirs(OUT_DIR, exist_ok=True)
OUT_CSV = os.path.join(OUT_DIR, "ablation_test_seed42_per_image.csv")
assert not os.path.exists(OUT_CSV), "File already exists - change OUT_CSV, do not overwrite"

t_start = time.time()
full = run_ablation(test_df, seeds=[42])
full.to_csv(OUT_CSV, index=False)
print(f"Done in {(time.time() - t_start) / 60:.1f} min | rows: {len(full)} (expected 3681)")
print("Saved:", OUT_CSV, "\n")

order = ["Low-light input"] + [arm_name(*a) for a in ARMS]
print(full.groupby("arm")[["PSNR", "SSIM"]].mean().reindex(order).round(4), "\n")
print("PSNR by dataset:")
print(full.pivot_table(index="arm", columns="dataset", values="PSNR").reindex(order).round(3))

In [ ]:
# ============================================================
# MEMBER 4 — STEP 16b
# RESCUE STEP 16 RESULTS (already in memory as `full`)
# 1) save locally first  2) print summary  3) reconnect Drive and copy
# ============================================================
import os, shutil

# 1. Save to the Colab disk immediately (does not depend on Drive)
assert "full" in globals(), "`full` is missing - stop and tell me"
LOCAL_CSV = "/content/ablation_test_seed42_per_image.csv"
full.to_csv(LOCAL_CSV, index=False)
print("Saved locally:", LOCAL_CSV, "| rows:", len(full), "(expected 3681)\n")

# 2. Summary
order = ["Low-light input"] + [arm_name(*a) for a in ARMS]
print(full.groupby("arm")[["PSNR", "SSIM"]].mean().reindex(order).round(4), "\n")
print("PSNR by dataset:")
print(full.pivot_table(index="arm", columns="dataset", values="PSNR").reindex(order).round(3), "\n")

# 3. Reconnect Drive and copy
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
OUT_DIR = "/content/drive/MyDrive/SANC/results/member4"
os.makedirs(OUT_DIR, exist_ok=True)
OUT_CSV = os.path.join(OUT_DIR, "ablation_test_seed42_per_image.csv")
if os.path.exists(OUT_CSV):
    print("Drive file already exists - NOT overwritten:", OUT_CSV)
else:
    shutil.copy(LOCAL_CSV, OUT_CSV)
    print("Copied to Drive:", OUT_CSV)

In [ ]:
import inspect

# Search currently available notebook history-independent functions
print("Forward-related source previously inspected:")
print(forward_source[:12000])

In [ ]:
print("Diffusion schedule:")
print("T:", T)
print("alpha_bars shape:", alpha_bars.shape)
print("alpha_bar at t=0:", alpha_bars[0].item())
print("alpha_bar at t=50:", alpha_bars[50].item())
print("alpha_bar at t=99:", alpha_bars[99].item())

print("\nExisting noise tensors:")

for name in ["noise", "colored_noise", "x_t", "predicted_noise"]:
    value = globals().get(name)

    if isinstance(value, torch.Tensor):
        print(
            name,
            "| Shape:", tuple(value.shape),
            "| Mean:", value.mean().item(),
            "| Std:", value.std().item()
        )

In [ ]:
import inspect

print("Reverse model training-related objects:\n")

for name, obj in globals().copy().items():

    if callable(obj):
        try:
            source = inspect.getsource(obj)

            if (
                "predicted_noise" in source
                or "colored_noise" in source
                or "sqrt_alpha_bars" in source
            ):
                print("\n" + "=" * 60)
                print("OBJECT:", name)
                print(source[:7000])

        except (OSError, TypeError):
            pass

In [ ]:
training_keywords = [
    "optimizer.zero_grad",
    "predicted_noise",
    "colored_noise",
    "noise_pred",
    "loss.backward",
    "reverse_model"
]

for i, source in enumerate(In):

    if not isinstance(source, str):
        continue

    matches = [
        keyword for keyword in training_keywords
        if keyword in source
    ]

    if matches:
        print("\n" + "=" * 70)
        print("EXECUTED CELL:", i)
        print("MATCHES:", matches)
        print(source[:8000])

In [ ]:
import re

patterns = [
    r"optimizer\.zero_grad",
    r"loss\.backward",
    r"optimizer\.step",
    r"predicted_noise\s*=",
    r"noise_pred\s*=",
    r"colored_noise\s*="
]

for i, source in enumerate(In):

    if not isinstance(source, str) or i == 1:
        continue

    found = [
        p for p in patterns
        if re.search(p, source)
    ]

    if found:
        print("\n" + "=" * 70)
        print("CELL:", i)
        print("MATCHES:", found)
        print(source[:10000])

In [ ]:
import torch
import os

checkpoint_path = (
    "/content/drive/MyDrive/SANC/models/"
    "reverse_diffusion_model.pth"
)

print("Checkpoint exists:", os.path.exists(checkpoint_path))

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("\nCheckpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("\nCheckpoint keys:")

    for key, value in checkpoint.items():

        if isinstance(value, torch.Tensor):
            print(key, "| Tensor", tuple(value.shape))

        elif isinstance(value, dict):
            print(key, "| Dictionary with", len(value), "entries")

        else:
            print(key, "|", type(value).__name__, "|", value if isinstance(value, (int, float, str, bool)) else "")

In [ ]:
import torch

required_objects = [
    "device",
    "T",
    "sanc",
    "saliency_model",
    "TimeEmbedding",
    "TimeResBlock",
    "AttentionFusion",
    "DualBranchReverseUNet",
]

print("Environment restoration check:\n")

for name in required_objects:
    exists = name in globals()
    print(f"{name}: {'Available' if exists else 'Missing'}")

print("\nCUDA available:", torch.cuda.is_available())
print("Current device:", device if "device" in globals() else "Not initialized")